In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 4 — BATCH DOWNLOAD ALL 84 CLIPS
#  Uses get_clip_url() and download_clip() from Cell 3.
#  ~5MB per clip x 84 = ~420MB total. Fine to keep all on disk.
#  This will take a few minutes — it's making 84 sequential
#  requests to Savant plus 84 downloads.
# ══════════════════════════════════════════════════════════

import time

clip_paths = {}   # play_id -> local file path
failures   = []

print(f"Downloading {len(df)} clips...\n")

for i, row in df.iterrows():
    play_id = row["play_id"]
    out_path = os.path.join(CLIPS_DIR, f"{play_id}.mp4")

    # Skip if already downloaded (e.g. re-running after a partial failure)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        clip_paths[play_id] = out_path
        continue

    clip_url = get_clip_url(play_id)
    if not clip_url:
        print(f"  [{i+1}/{len(df)}] FAILED to resolve URL for {play_id} (inning {row['inning']}, pitch {row['pitch_number']})")
        failures.append(play_id)
        continue

    ok = download_clip(clip_url, out_path)
    if ok:
        clip_paths[play_id] = out_path
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  [{i+1}/{len(df)}] downloaded (inning {row['inning']}, pitch {row['pitch_number']})")
    else:
        print(f"  [{i+1}/{len(df)}] FAILED to download {play_id}")
        failures.append(play_id)

    time.sleep(0.3)  # be polite to Savant's servers

print(f"\n{'='*50}")
print(f"Done. Downloaded: {len(clip_paths)} / {len(df)}")
if failures:
    print(f"Failed play_ids ({len(failures)}): {failures}")
else:
    print("No failures.")

# Attach local clip path to the dataframe and save
df["clip_path"] = df["play_id"].map(clip_paths)
missing_clips = df["clip_path"].isna().sum()
print(f"\nRows missing a clip: {missing_clips} / {len(df)}")

out_path = os.path.join(OUTPUT_DIR, "misiorowski_822744_pitches_with_clips.csv")
df.to_csv(out_path, index=False)
print(f"Saved updated dataframe to: {out_path}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 5 (REVISED) — PHC-BASED TRIMMING + RELEASE FRAME SEARCH
#
#  Uses BaseballCV's phc_detector model (pitcher/hitter/catcher
#  role classifier) to find the contiguous window in each clip
#  where all three are actually visible together — this is the
#  real "live pitch" window, filtering out replays, graphics,
#  entrance walk-up, crowd cuts, etc. that were confusing our
#  earlier wrist-height release search.
#
#  IMPORTANT: the old shared-pool notebook hardcoded
#  RELEASE_FRAME = 179 — that was calibrated for a DIFFERENT
#  video source (the shared server's pre-built corpus). Our
#  clips come fresh from sporty-clips.mlb.com and may have a
#  different length/frame rate, so we do NOT reuse 179 directly.
#  Instead we find release empirically WITHIN the PHC-trimmed
#  window for our own clips.
# ══════════════════════════════════════════════════════════

# ── First, make sure baseballcv is installed ──
try:
    from baseballcv.functions import LoadTools
    print("[OK] baseballcv already installed")
except ImportError:
    print("[MISSING] baseballcv not installed.")
    print("Run this in a new cell, then re-run this cell:")
    print("    pip install baseballcv")
    raise

import numpy as np
import cv2
import matplotlib.pyplot as plt

load_tools = LoadTools()
phc_model  = YOLO(load_tools.load_model("phc_detector"))
print("phc_detector classes:", phc_model.names)

CONF_THRESH = 0.4
EXPECTED    = {"pitcher", "hitter", "catcher"}


def get_phc_roles(frame):
    """Return dict of {role: confidence} for roles detected above threshold."""
    res  = phc_model(frame, verbose=False)[0]
    best = {}
    if res.boxes is not None:
        for conf, cls in zip(res.boxes.conf.cpu().numpy(), res.boxes.cls.cpu().numpy()):
            if conf < CONF_THRESH:
                continue
            role = phc_model.names[int(cls)].lower()
            if role not in best or conf > best[role]:
                best[role] = float(conf)
    return best


def find_phc_window(frames, sample_every=2):
    """
    Scan through frames (every Nth frame for speed) and find the
    longest contiguous run where pitcher, hitter, AND catcher are
    all simultaneously detected. Returns (start_idx, end_idx) in
    ORIGINAL frame indices, or (None, None) if never all three.
    """
    visible = []  # (frame_idx, bool_all_three)
    for i in range(0, len(frames), sample_every):
        roles = get_phc_roles(frames[i])
        all_three = EXPECTED.issubset(set(roles.keys()))
        visible.append((i, all_three))

    # Find longest contiguous True run
    best_start, best_end = None, None
    best_len = 0
    cur_start = None
    for idx, ok in visible:
        if ok and cur_start is None:
            cur_start = idx
        elif not ok and cur_start is not None:
            run_len = idx - cur_start
            if run_len > best_len:
                best_len = run_len
                best_start, best_end = cur_start, idx - sample_every
            cur_start = None
    if cur_start is not None:
        run_len = visible[-1][0] - cur_start
        if run_len > best_len:
            best_start, best_end = cur_start, visible[-1][0]

    return best_start, best_end, visible


def find_release_in_window(frames, tracked, start_idx, end_idx, wrist_side="r_wrist"):
    """
    Within the PHC-visible window, find release frame via peak wrist
    height (min y value) — same heuristic as before, but now scoped
    to only the real pitch-delivery window instead of the whole clip.
    """
    wrist_idx = KP[wrist_side]
    ys = []
    idxs = list(range(start_idx, end_idx + 1))
    for i in idxs:
        box, kp = tracked[i]
        if kp is None or kp[wrist_idx, 2] < 0.3:
            ys.append(np.nan)
        else:
            ys.append(kp[wrist_idx, 1])
    ys = np.array(ys)
    if np.all(np.isnan(ys)):
        return None
    best_local = int(np.nanargmin(ys))
    return idxs[best_local]


# ── TEST ON THE SAME CLIP AS BEFORE ─────────────────────────────────
test_path = df.iloc[0]["clip_path"]
print(f"\nTesting PHC trim on: {test_path}")

frames = load_frames(test_path)
print(f"Loaded {len(frames)} frames")

start_idx, end_idx, visibility_trace = find_phc_window(frames, sample_every=2)

if start_idx is None:
    print("\n[PROBLEM] Never detected all three (pitcher/hitter/catcher) simultaneously.")
    print("Confidence threshold may be too strict, or this clip's camera angle doesn't")
    print("show all three at once. Print visibility_trace to debug frame by frame.")
else:
    print(f"\nPHC-visible window: frames {start_idx} to {end_idx} ({end_idx - start_idx} frames)")

    # Re-use the pitcher tracking from Cell 5 original, but only need it
    # for frames inside the window — cheaper to just re-track full clip
    # since we already have that function.
    tracked = track_pitcher_through_clip(frames)

    release_frame = find_release_in_window(frames, tracked, start_idx, end_idx)
    print(f"Candidate release frame (within PHC window): {release_frame}")

    if release_frame is not None:
        window = range(max(start_idx, release_frame - 4), min(end_idx, release_frame + 5) + 1)
        fig, axes = plt.subplots(1, len(list(window)), figsize=(20, 4))
        window = range(max(start_idx, release_frame - 4), min(end_idx, release_frame + 5) + 1)
        for ax, fi in zip(axes, window):
            frame_rgb = cv2.cvtColor(frames[fi], cv2.COLOR_BGR2RGB)
            box, kp = tracked[fi]
            ax.imshow(frame_rgb)
            if box is not None:
                x1, y1, x2, y2 = box
                ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
            ax.set_title(f"f{fi}" + (" <-- candidate" if fi == release_frame else ""), fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.savefig(r"C:\Users\noahl\OneDrive\fatigue_cv\release_frame_check_v2.png", dpi=100)
        plt.show()
        print("\nSaved to release_frame_check_v2.png")
        print(">>> Does the candidate frame show actual ball release now?")

In [ ]:
# ══════════════════════════════════════════════════════════
#  FIX: remove BOTH conflicting pandas dist-info folders,
#  then cleanly reinstall pandas 3.0.5 (matches the actual
#  package files already on disk — no need to change versions,
#  just fix the metadata bookkeeping).
# ══════════════════════════════════════════════════════════

import shutil
import subprocess
import sys
import os
import importlib.metadata

site_packages = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages")

bad_dirs = [
    os.path.join(site_packages, "pandas-3.0.2.dist-info"),
    os.path.join(site_packages, "pandas-3.0.5.dist-info"),
]

for d in bad_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed: {d}")
    else:
        print(f"Not found (skipping): {d}")

print("\nReinstalling pandas cleanly (this will rewrite a single correct dist-info)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", "pandas==3.0.5"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "(no stdout)")
if result.stderr.strip():
    print(f"[stderr]:\n{result.stderr[-1000:]}")

print("\n" + "="*50)
print("Verifying fix...")
importlib.invalidate_caches()

# Force importlib.metadata to re-read from disk (it can cache)
importlib.reload(importlib.metadata) if hasattr(importlib.metadata, "__loader__") else None

try:
    version = importlib.metadata.version("pandas")
    print(f"[OK] importlib.metadata.version('pandas') = {version}")
except Exception as e:
    print(f"[STILL BROKEN] {type(e).__name__}: {e}")

import pandas
print(f"[OK] pandas.__version__ = {pandas.__version__}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  WORKAROUND: manually recreate minimal pandas dist-info
#  metadata WITHOUT reinstalling/touching the actual pandas
#  package files (which are fine and already working — only
#  the metadata folder was lost). This avoids the Windows
#  file-lock issue entirely since we're not overwriting any
#  binaries, just adding back a small metadata folder.
# ══════════════════════════════════════════════════════════

import sys
import os
import importlib.metadata
import pandas

site_packages = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages")
version = pandas.__version__  # "3.0.5" — confirmed working
dist_info_dir = os.path.join(site_packages, f"pandas-{version}.dist-info")

os.makedirs(dist_info_dir, exist_ok=True)

metadata_content = f"""Metadata-Version: 2.1
Name: pandas
Version: {version}
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: https://pandas.pydata.org
License: BSD-3-Clause
Requires-Python: >=3.9
"""

with open(os.path.join(dist_info_dir, "METADATA"), "w") as f:
    f.write(metadata_content)

with open(os.path.join(dist_info_dir, "INSTALLER"), "w") as f:
    f.write("manual-repair\n")

# top_level.txt helps some tools identify the importable module name
with open(os.path.join(dist_info_dir, "top_level.txt"), "w") as f:
    f.write("pandas\n")

print(f"Created minimal dist-info at: {dist_info_dir}")

# ── Verify ──
print("\nVerifying...")
try:
    v = importlib.metadata.version("pandas")
    print(f"[OK] importlib.metadata.version('pandas') = {v}")
except Exception as e:
    print(f"[STILL BROKEN] {type(e).__name__}: {e}")
    raise

print("\nTesting baseballcv import chain...")
try:
    from baseballcv.functions import LoadTools
    print("[OK] baseballcv.functions.LoadTools imported successfully")
except Exception as e:
    print(f"[FAIL] {type(e).__name__}: {e}")
    raise

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 1 — ENVIRONMENT CHECK
#  Run this first. It just checks what's installed and tells you
#  what (if anything) needs to be pip installed. Doesn't touch
#  any data yet.
# ══════════════════════════════════════════════════════════

import importlib
import sys

print(f"Python: {sys.version}\n")

packages = [
    "cv2",            # opencv-python (comes in via ultralytics too)
    "ultralytics",    # YOLOv8 pose
    "numpy",
    "pandas",
    "scipy",          # for regression / stats (linregress)
    "matplotlib",
    "requests",
    "bs4",            # beautifulsoup4, for scraping sporty-clips URLs
]

missing = []

for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown version")
        print(f"[OK]      {pkg:<12} -> {version}")
    except ImportError:
        print(f"[MISSING] {pkg}")
        missing.append(pkg)

print()
if missing:
    pip_names = {
        "cv2": "opencv-python",
        "bs4": "beautifulsoup4",
    }
    to_install = [pip_names.get(p, p) for p in missing]
    print("Run this in a terminal (or a new cell with a leading !) to install what's missing:")
    print(f"    pip install {' '.join(to_install)}")
else:
    print("Everything needed is already installed.")

# Separately check that a YOLOv8m-pose weights file loads correctly.
# This will auto-download yolov8m-pose.pt on first run if it's not
# already cached locally (~50MB, one-time).
print("\nChecking YOLOv8m-pose model load...")
try:
    from ultralytics import YOLO
    model = YOLO("yolov8m-pose.pt")
    print("[OK] yolov8m-pose.pt loaded successfully.")
except Exception as e:
    print(f"[ERROR] Could not load YOLOv8m-pose: {e}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 2 — PULL GAME DATA (MLB Stats API live feed)
#  game_pk 822744: Brewers vs Nationals, Misiorowski start
#  Pulls the FULL play-by-play + pitch-by-pitch feed and builds
#  a clean pandas DataFrame of just Misiorowski's pitches.
# ══════════════════════════════════════════════════════════

import requests
import pandas as pd
import numpy as np
import os

GAME_PK      = 822744
PITCHER_ID   = 694819   # Misiorowski MLBAM ID
OUTPUT_DIR   = r"C:\Users\noahl\OneDrive\fatigue_cv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

url = f"https://statsapi.mlb.com/api/v1.1/game/{GAME_PK}/feed/live"
resp = requests.get(url, timeout=30)
resp.raise_for_status()
data = resp.json()

all_plays = data["liveData"]["plays"]["allPlays"]

rows = []
for play in all_plays:
    matchup = play.get("matchup", {})
    pitcher = matchup.get("pitcher", {})
    if pitcher.get("id") != PITCHER_ID:
        continue

    about = play.get("about", {})
    inning = about.get("inning")
    half   = about.get("halfInning")  # "top" or "bottom"

    for event in play.get("playEvents", []):
        if not event.get("isPitch", False):
            continue

        details = event.get("details", {})
        pitch_data = event.get("pitchData", {})

        row = {
            "game_pk":        GAME_PK,
            "inning":         inning,
            "half_inning":    half,
            "at_bat_index":   play.get("atBatIndex"),
            "pitch_number":   event.get("pitchNumber"),
            "play_id":        event.get("playId"),           # needed for video lookup later
            "pitch_type":     details.get("type", {}).get("code"),
            "pitch_type_desc":details.get("type", {}).get("description"),
            "call":           details.get("call", {}).get("description"),
            "start_speed":    pitch_data.get("startSpeed"),
            "end_speed":      pitch_data.get("endSpeed"),
            "spin_rate":      pitch_data.get("breaks", {}).get("spinRate"),
            "ax":             pitch_data.get("coordinates", {}).get("aX"),
            "az":             pitch_data.get("coordinates", {}).get("aZ"),
            "px":             pitch_data.get("coordinates", {}).get("pX"),
            "pz":             pitch_data.get("coordinates", {}).get("pZ"),
            "batter_id":      matchup.get("batter", {}).get("id"),
            "batter_name":    matchup.get("batter", {}).get("fullName"),
        }
        rows.append(row)

df = pd.DataFrame(rows)

# Sanity checks
print(f"Total pitches pulled: {len(df)}")
print(f"Innings covered: {sorted(df['inning'].unique())}")
print(f"Half-inning values: {df['half_inning'].unique()}")
print(f"Pitch types: {df['pitch_type_desc'].value_counts().to_dict()}")
print(f"\nMissing play_id count: {df['play_id'].isna().sum()} / {len(df)}")
print(f"\nFirst few rows:")
print(df[["inning", "half_inning", "pitch_number", "pitch_type_desc", "start_speed", "play_id"]].head(10))

# Save raw pull immediately so we don't have to re-hit the API
out_path = os.path.join(OUTPUT_DIR, "misiorowski_822744_pitches_raw.csv")
df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 3 — RESOLVE PLAY_ID -> VIDEO URL (test on ONE clip first)
#  Baseball Savant's /sporty-videos page returns HTML, not video
#  directly. The actual MP4 lives at sporty-clips.mlb.com and is
#  embedded in a <video><source> tag we scrape out with BeautifulSoup.
#
#  This cell just tests the approach on ONE pitch before we commit
#  to downloading all 84 clips in Cell 4.
# ══════════════════════════════════════════════════════════

import requests
from bs4 import BeautifulSoup
import os
import time

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv"
CLIPS_DIR  = os.path.join(OUTPUT_DIR, "clips")
os.makedirs(CLIPS_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def get_clip_url(play_id, max_retries=3, sleep_between=1.0):
    """
    Given a Baseball Savant playId (uuid string), scrape the
    /sporty-videos page and return the direct mp4 URL, or None.
    """
    page_url = f"https://baseballsavant.mlb.com/sporty-videos?playId={play_id}"

    for attempt in range(max_retries):
        try:
            resp = requests.get(page_url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")

            video_tag = soup.find("video")
            if video_tag:
                source_tag = video_tag.find("source")
                if source_tag and source_tag.get("src"):
                    return source_tag["src"]

            # Fallback: sometimes the src is directly on <video src=...>
            if video_tag and video_tag.get("src"):
                return video_tag["src"]

            return None

        except requests.RequestException as e:
            print(f"  attempt {attempt+1} failed for {play_id}: {e}")
            time.sleep(sleep_between)

    return None


def download_clip(url, out_path, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30, stream=True)
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        except requests.RequestException as e:
            print(f"  download attempt {attempt+1} failed: {e}")
            time.sleep(1)
    return False


# ── TEST ON ONE PLAY ─────────────────────────────────────────
test_play_id = df.iloc[0]["play_id"]
print(f"Testing with play_id: {test_play_id}")

clip_url = get_clip_url(test_play_id)
print(f"Resolved clip URL: {clip_url}")

if clip_url:
    test_out = os.path.join(CLIPS_DIR, f"test_{test_play_id}.mp4")
    ok = download_clip(clip_url, test_out)
    if ok and os.path.exists(test_out):
        size_kb = os.path.getsize(test_out) / 1024
        print(f"\n[OK] Downloaded test clip: {test_out} ({size_kb:.1f} KB)")
    else:
        print("\n[ERROR] Download failed.")
else:
    print("\n[ERROR] Could not resolve a clip URL for this play_id.")
    print("Possible causes: Savant page structure changed, or this specific")
    print("play doesn't have video (rare). We'll debug the HTML directly if needed.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 4 — BATCH DOWNLOAD ALL 84 CLIPS
#  Uses get_clip_url() and download_clip() from Cell 3.
#  ~5MB per clip x 84 = ~420MB total. Fine to keep all on disk.
#  This will take a few minutes — it's making 84 sequential
#  requests to Savant plus 84 downloads.
# ══════════════════════════════════════════════════════════

import time

clip_paths = {}   # play_id -> local file path
failures   = []

print(f"Downloading {len(df)} clips...\n")

for i, row in df.iterrows():
    play_id = row["play_id"]
    out_path = os.path.join(CLIPS_DIR, f"{play_id}.mp4")

    # Skip if already downloaded (e.g. re-running after a partial failure)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        clip_paths[play_id] = out_path
        continue

    clip_url = get_clip_url(play_id)
    if not clip_url:
        print(f"  [{i+1}/{len(df)}] FAILED to resolve URL for {play_id} (inning {row['inning']}, pitch {row['pitch_number']})")
        failures.append(play_id)
        continue

    ok = download_clip(clip_url, out_path)
    if ok:
        clip_paths[play_id] = out_path
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  [{i+1}/{len(df)}] downloaded (inning {row['inning']}, pitch {row['pitch_number']})")
    else:
        print(f"  [{i+1}/{len(df)}] FAILED to download {play_id}")
        failures.append(play_id)

    time.sleep(0.3)  # be polite to Savant's servers

print(f"\n{'='*50}")
print(f"Done. Downloaded: {len(clip_paths)} / {len(df)}")
if failures:
    print(f"Failed play_ids ({len(failures)}): {failures}")
else:
    print("No failures.")

# Attach local clip path to the dataframe and save
df["clip_path"] = df["play_id"].map(clip_paths)
missing_clips = df["clip_path"].isna().sum()
print(f"\nRows missing a clip: {missing_clips} / {len(df)}")

out_path = os.path.join(OUTPUT_DIR, "misiorowski_822744_pitches_with_clips.csv")
df.to_csv(out_path, index=False)
print(f"Saved updated dataframe to: {out_path}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 5 — POSE PIPELINE CORE: mound heuristic, pitcher tracking,
#  release frame detection. TEST ON ONE CLIP FIRST.
#
#  YOLOv8-pose keypoint indices (COCO-17):
#   0 nose, 1 l_eye, 2 r_eye, 3 l_ear, 4 r_ear,
#   5 l_shoulder, 6 r_shoulder, 7 l_elbow, 8 r_elbow,
#   9 l_wrist, 10 r_wrist, 11 l_hip, 12 r_hip,
#   13 l_knee, 14 r_knee, 15 l_ankle, 16 r_ankle
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

pose_model = YOLO("yolov8m-pose.pt")

KP = {
    "nose": 0, "l_shoulder": 5, "r_shoulder": 6,
    "l_elbow": 7, "r_elbow": 8, "l_wrist": 9, "r_wrist": 10,
    "l_hip": 11, "r_hip": 12, "l_knee": 13, "r_knee": 14,
    "l_ankle": 15, "r_ankle": 16,
}

# Mound-position heuristic: pitcher is near this normalized (x, y)
# position in a standard Savant center-field broadcast angle.
# Validated empirically in prior sessions at ~(0.35, 0.65).
MOUND_X_FRAC = 0.35
MOUND_Y_FRAC = 0.65


def load_frames(video_path, max_frames=400):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


def get_all_people(frame, conf=0.3):
    """Run YOLOv8-pose on a frame, return list of (box, keypoints) for all detected people."""
    results = pose_model(frame, verbose=False, conf=conf)[0]
    people = []
    if results.keypoints is None or results.boxes is None:
        return people
    boxes = results.boxes.xyxy.cpu().numpy()
    kps   = results.keypoints.data.cpu().numpy()  # (N, 17, 3) -> x, y, conf
    for box, kp in zip(boxes, kps):
        people.append((box, kp))
    return people


def pick_pitcher(people, frame_w, frame_h, prev_box=None):
    """
    Select the pitcher from all detected people using the mound heuristic,
    with frame-to-frame IoU locking to prevent identity jumps once we have
    a previous box to anchor to.
    """
    if not people:
        return None, None

    target_x = frame_w * MOUND_X_FRAC
    target_y = frame_h * MOUND_Y_FRAC

    def box_center(b):
        return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)

    def iou(a, b):
        xA, yA = max(a[0], b[0]), max(a[1], b[1])
        xB, yB = min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        areaA = (a[2]-a[0]) * (a[3]-a[1])
        areaB = (b[2]-b[0]) * (b[3]-b[1])
        return inter / max(areaA + areaB - inter, 1e-8)

    if prev_box is not None:
        # Lock onto whichever detection best overlaps the previous frame's pitcher box
        best, best_score = None, 0.0
        for box, kp in people:
            score = iou(prev_box, box)
            if score > best_score:
                best, best_score = (box, kp), score
        if best is not None and best_score > 0.15:
            return best

    # Fall back to mound-distance heuristic
    best, best_dist = None, float("inf")
    for box, kp in people:
        cx, cy = box_center(box)
        dist = (cx - target_x) ** 2 + (cy - target_y) ** 2
        if dist < best_dist:
            best, best_dist = (box, kp), dist
    return best


def track_pitcher_through_clip(frames, conf=0.3):
    """
    Run pose estimation across all frames, tracking the pitcher via
    mound heuristic + IoU locking. Returns list of (box, keypoints) or
    (None, None) per frame.
    """
    tracked = []
    prev_box = None
    h, w = frames[0].shape[:2]
    for frame in frames:
        people = get_all_people(frame, conf=conf)
        box, kp = pick_pitcher(people, w, h, prev_box=prev_box) if people else (None, None)
        tracked.append((box, kp))
        if box is not None:
            prev_box = box
    return tracked


def find_release_frame_candidate(tracked, wrist_side="r_wrist"):
    """
    Heuristic release-frame finder: release is approximately when the
    throwing wrist reaches its highest velocity / most extended forward
    position. We approximate with peak wrist height (min y = highest
    in frame) as a starting candidate — THIS MUST BE VISUALLY VALIDATED.
    """
    wrist_idx = KP[wrist_side]
    ys = []
    for box, kp in tracked:
        if kp is None or kp[wrist_idx, 2] < 0.3:
            ys.append(np.nan)
        else:
            ys.append(kp[wrist_idx, 1])
    ys = np.array(ys)
    if np.all(np.isnan(ys)):
        return None
    # Candidate = frame of minimum wrist y (highest point) within the
    # back half of the clip (release happens after the leg lift, not windup)
    start = len(ys) // 3
    candidate = start + int(np.nanargmin(ys[start:]))
    return candidate


# ── TEST ON ONE CLIP ─────────────────────────────────────────
test_path = df.iloc[0]["clip_path"]
print(f"Testing on: {test_path}")

frames = load_frames(test_path)
print(f"Loaded {len(frames)} frames")

tracked = track_pitcher_through_clip(frames)
detected_count = sum(1 for b, k in tracked if b is not None)
print(f"Pitcher detected in {detected_count}/{len(frames)} frames")

release_candidate = find_release_frame_candidate(tracked)
print(f"Candidate release frame: {release_candidate} (out of {len(frames)})")

# Save a visual strip of frames around the candidate for manual validation
if release_candidate is not None:
    window = range(max(0, release_candidate - 4), min(len(frames), release_candidate + 5))
    fig, axes = plt.subplots(1, len(list(window)), figsize=(20, 4))
    window = range(max(0, release_candidate - 4), min(len(frames), release_candidate + 5))
    for ax, fi in zip(axes, window):
        frame_rgb = cv2.cvtColor(frames[fi], cv2.COLOR_BGR2RGB)
        box, kp = tracked[fi]
        ax.imshow(frame_rgb)
        if box is not None:
            x1, y1, x2, y2 = box
            ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
        ax.set_title(f"f{fi}" + (" <-- candidate" if fi == release_candidate else ""), fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(r"C:\Users\noahl\OneDrive\fatigue_cv\release_frame_check.png", dpi=100)
    plt.show()
    print("\nSaved validation strip to release_frame_check.png")
    print(">>> LOOK AT THE IMAGES: does the highlighted frame actually show ball release")
    print(">>> (arm extended forward, about to let go of the ball)? Tell me which frame")
    print(">>> index in the strip actually looks like release if the candidate is off.")
else:
    print("Could not find a release candidate — wrist tracking may be too unreliable on this clip.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CELL 5 (REVISED) — PHC-BASED TRIMMING + RELEASE FRAME SEARCH
#
#  Uses BaseballCV's phc_detector model (pitcher/hitter/catcher
#  role classifier) to find the contiguous window in each clip
#  where all three are actually visible together — this is the
#  real "live pitch" window, filtering out replays, graphics,
#  entrance walk-up, crowd cuts, etc. that were confusing our
#  earlier wrist-height release search.
#
#  IMPORTANT: the old shared-pool notebook hardcoded
#  RELEASE_FRAME = 179 — that was calibrated for a DIFFERENT
#  video source (the shared server's pre-built corpus). Our
#  clips come fresh from sporty-clips.mlb.com and may have a
#  different length/frame rate, so we do NOT reuse 179 directly.
#  Instead we find release empirically WITHIN the PHC-trimmed
#  window for our own clips.
# ══════════════════════════════════════════════════════════

# ── First, make sure baseballcv is installed ──
try:
    from baseballcv.functions import LoadTools
    print("[OK] baseballcv already installed")
except ImportError:
    print("[MISSING] baseballcv not installed.")
    print("Run this in a new cell, then re-run this cell:")
    print("    pip install baseballcv")
    raise

import numpy as np
import cv2
import matplotlib.pyplot as plt

load_tools = LoadTools()
phc_model  = YOLO(load_tools.load_model("phc_detector"))
print("phc_detector classes:", phc_model.names)

CONF_THRESH = 0.4
EXPECTED    = {"pitcher", "hitter", "catcher"}


def get_phc_roles(frame):
    """Return dict of {role: confidence} for roles detected above threshold."""
    res  = phc_model(frame, verbose=False)[0]
    best = {}
    if res.boxes is not None:
        for conf, cls in zip(res.boxes.conf.cpu().numpy(), res.boxes.cls.cpu().numpy()):
            if conf < CONF_THRESH:
                continue
            role = phc_model.names[int(cls)].lower()
            if role not in best or conf > best[role]:
                best[role] = float(conf)
    return best


def find_phc_window(frames, sample_every=2):
    """
    Scan through frames (every Nth frame for speed) and find the
    longest contiguous run where pitcher, hitter, AND catcher are
    all simultaneously detected. Returns (start_idx, end_idx) in
    ORIGINAL frame indices, or (None, None) if never all three.
    """
    visible = []  # (frame_idx, bool_all_three)
    for i in range(0, len(frames), sample_every):
        roles = get_phc_roles(frames[i])
        all_three = EXPECTED.issubset(set(roles.keys()))
        visible.append((i, all_three))

    # Find longest contiguous True run
    best_start, best_end = None, None
    best_len = 0
    cur_start = None
    for idx, ok in visible:
        if ok and cur_start is None:
            cur_start = idx
        elif not ok and cur_start is not None:
            run_len = idx - cur_start
            if run_len > best_len:
                best_len = run_len
                best_start, best_end = cur_start, idx - sample_every
            cur_start = None
    if cur_start is not None:
        run_len = visible[-1][0] - cur_start
        if run_len > best_len:
            best_start, best_end = cur_start, visible[-1][0]

    return best_start, best_end, visible


def find_release_in_window(frames, tracked, start_idx, end_idx, wrist_side="r_wrist"):
    """
    Within the PHC-visible window, find release frame via peak wrist
    height (min y value) — same heuristic as before, but now scoped
    to only the real pitch-delivery window instead of the whole clip.
    """
    wrist_idx = KP[wrist_side]
    ys = []
    idxs = list(range(start_idx, end_idx + 1))
    for i in idxs:
        box, kp = tracked[i]
        if kp is None or kp[wrist_idx, 2] < 0.3:
            ys.append(np.nan)
        else:
            ys.append(kp[wrist_idx, 1])
    ys = np.array(ys)
    if np.all(np.isnan(ys)):
        return None
    best_local = int(np.nanargmin(ys))
    return idxs[best_local]


# ── TEST ON THE SAME CLIP AS BEFORE ─────────────────────────────────
test_path = df.iloc[0]["clip_path"]
print(f"\nTesting PHC trim on: {test_path}")

frames = load_frames(test_path)
print(f"Loaded {len(frames)} frames")

start_idx, end_idx, visibility_trace = find_phc_window(frames, sample_every=2)

if start_idx is None:
    print("\n[PROBLEM] Never detected all three (pitcher/hitter/catcher) simultaneously.")
    print("Confidence threshold may be too strict, or this clip's camera angle doesn't")
    print("show all three at once. Print visibility_trace to debug frame by frame.")
else:
    print(f"\nPHC-visible window: frames {start_idx} to {end_idx} ({end_idx - start_idx} frames)")

    # Re-use the pitcher tracking from Cell 5 original, but only need it
    # for frames inside the window — cheaper to just re-track full clip
    # since we already have that function.
    tracked = track_pitcher_through_clip(frames)

    release_frame = find_release_in_window(frames, tracked, start_idx, end_idx)
    print(f"Candidate release frame (within PHC window): {release_frame}")

    if release_frame is not None:
        window = range(max(start_idx, release_frame - 4), min(end_idx, release_frame + 5) + 1)
        fig, axes = plt.subplots(1, len(list(window)), figsize=(20, 4))
        window = range(max(start_idx, release_frame - 4), min(end_idx, release_frame + 5) + 1)
        for ax, fi in zip(axes, window):
            frame_rgb = cv2.cvtColor(frames[fi], cv2.COLOR_BGR2RGB)
            box, kp = tracked[fi]
            ax.imshow(frame_rgb)
            if box is not None:
                x1, y1, x2, y2 = box
                ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
            ax.set_title(f"f{fi}" + (" <-- candidate" if fi == release_frame else ""), fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.savefig(r"C:\Users\noahl\OneDrive\fatigue_cv\release_frame_check_v2.png", dpi=100)
        plt.show()
        print("\nSaved to release_frame_check_v2.png")
        print(">>> Does the candidate frame show actual ball release now?")

In [ ]:
# ══════════════════════════════════════════════════════════
#  CHECK: actual FPS of these clips, to validate whether ±4
#  frames around release is the right window size in real time.
# ══════════════════════════════════════════════════════════

import cv2

test_path = df.iloc[0]["clip_path"]
cap = cv2.VideoCapture(test_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
duration_sec = frame_count / fps if fps > 0 else float("nan")

cap.release()

print(f"Clip: {test_path}")
print(f"Reported FPS:      {fps}")
print(f"Frame count:       {frame_count}")
print(f"Resolution:        {int(width)}x{int(height)}")
print(f"Implied duration:  {duration_sec:.2f} sec")

print(f"\nAt this FPS, one frame = {1000/fps:.1f} ms")
for n in [3, 4, 5, 6]:
    print(f"  ±{n} frames = ±{n * 1000/fps:.1f} ms window (total {(2*n+1)} frames)")

# Cross-check against a couple more clips in case FPS varies by clip
print("\nCross-checking FPS across a few more clips (in case it varies)...")
sample_paths = df["clip_path"].dropna().iloc[[10, 30, 50, 80]].tolist()
for p in sample_paths:
    c = cv2.VideoCapture(p)
    f = c.get(cv2.CAP_PROP_FPS)
    n = c.get(cv2.CAP_PROP_FRAME_COUNT)
    c.release()
    print(f"  {p.split(chr(92))[-1]}: fps={f}, frames={n}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  MATPLOTLIB FuncAnimation SCRUBBER — same tool used in the
#  soccer CV penalty-kick project. Gives a real play/pause/
#  frame-step player embedded directly in the notebook output.
#
#  Uses `frames`, `tracked`, `KP` already in memory.
# ══════════════════════════════════════════════════════════

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rcParams
from IPython.display import HTML
import cv2
import numpy as np

# Bump the embed size limit up front — hit this exact error before
rcParams['animation.embed_limit'] = 100  # MB

FRAME_START = 145
FRAME_END   = 215
frame_indices = list(range(FRAME_START, FRAME_END + 1))

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

sample = cv2.cvtColor(frames[FRAME_START], cv2.COLOR_BGR2RGB)
h, w = sample.shape[:2]

fig, ax = plt.subplots(figsize=(9, 6))
img_display = ax.imshow(sample)
ax.axis("off")

# Persistent artists we'll update each frame
box_rect = plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor="lime", linewidth=2, visible=False)
ax.add_patch(box_rect)

skel_lines = [ax.plot([], [], color="cyan", linewidth=2, alpha=0.8)[0] for _ in SKELETON_EDGES]
joint_dots = ax.scatter([], [], c="red", s=25, zorder=5)

frame_text = ax.text(10, 30, "", color="white", fontsize=14, weight="bold",
                      bbox=dict(facecolor="black", alpha=0.6, pad=4))

def update(i):
    frame_idx = frame_indices[i]
    frame_rgb = cv2.cvtColor(frames[frame_idx], cv2.COLOR_BGR2RGB)
    img_display.set_data(frame_rgb)

    box, kp = tracked[frame_idx]

    if box is not None:
        x1, y1, x2, y2 = box
        box_rect.set_xy((x1, y1))
        box_rect.set_width(x2 - x1)
        box_rect.set_height(y2 - y1)
        box_rect.set_visible(True)
    else:
        box_rect.set_visible(False)

    if kp is not None:
        for line, (a, b) in zip(skel_lines, SKELETON_EDGES):
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > 0.3 and kp[ib, 2] > 0.3:
                line.set_data([kp[ia, 0], kp[ib, 0]], [kp[ia, 1], kp[ib, 1]])
            else:
                line.set_data([], [])
        visible_pts = np.array([kp[idx, :2] for idx in KP.values() if kp[idx, 2] > 0.3])
        if len(visible_pts) > 0:
            joint_dots.set_offsets(visible_pts)
        else:
            joint_dots.set_offsets(np.empty((0, 2)))
    else:
        for line in skel_lines:
            line.set_data([], [])
        joint_dots.set_offsets(np.empty((0, 2)))

    label = f"frame {frame_idx}"
    if frame_idx == 179:
        label += "  <-- RELEASE (179)"
        frame_text.set_color("red")
    else:
        frame_text.set_color("white")
    frame_text.set_text(label)

    return [img_display, box_rect, frame_text, joint_dots] + skel_lines

plt.tight_layout()
anim = animation.FuncAnimation(
    fig, update,
    frames=len(frame_indices),
    interval=150,   # ms between frames when playing — slow enough to actually see
    blit=True,
)

plt.close()  # prevent a duplicate static plot from also rendering
print("Rendering interactive scrubber (may take ~15-30s)...")
HTML(anim.to_jshtml())

In [ ]:
# ══════════════════════════════════════════════════════════
#  FULL DIAGNOSTIC PIPELINE — run across all 84 pitches
#
#  For each pitch:
#    1. Load clip (frames up to 235 — enough to cover window 130-230)
#    2. Check PHC coverage across the FULL 130-230 window (not just
#       release) — gates out replays/cuts/bad camera angles
#    3. Track pitcher through the window
#    4. Extract ALL biomechanical metrics per frame in the release
#       band (175-183), both left AND right side, no assumptions
#       about which side "should" matter
#    5. Compute mean + std per metric across the release band
#       (std = stability/confidence diagnostic, not just noise)
#    6. Log per-joint mean confidence across the release band
#    7. Checkpoint to disk every 10 pitches
#
#  Requires (already in memory from earlier cells):
#    df, pose_model, phc_model, KP, load_frames,
#    track_pitcher_through_clip, get_phc_roles, EXPECTED
# ══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import os
from tqdm import tqdm

WINDOW_START   = 130
WINDOW_END     = 230
RELEASE_BAND   = range(175, 184)  # 175-183 inclusive, validated release window
PHC_SAMPLE_EVERY = 3              # sample every 3rd frame for PHC coverage check
PHC_COVERAGE_MIN = 0.85           # require 85%+ of sampled frames to show all 3 roles

CHECKPOINT_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
CHECKPOINT_EVERY = 10

# Right-handed pitcher (Misiorowski) — semantic side mapping
STRIDE_SIDE   = "l"   # lead/stride leg = left for RHP
THROW_SIDE    = "r"   # throwing arm = right for RHP


def angle_between(a, b, c):
    """Angle at point b, formed by points a-b-c, in degrees."""
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def angle_from_vertical(vec):
    """Angle of a 2D vector from the vertical axis (0, -1), in degrees."""
    vertical = np.array([0, -1])
    cos_a = np.dot(vec, vertical) / (np.linalg.norm(vec) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def compute_all_metrics(kp, conf_thresh=0.3):
    """
    Given a single frame's (17,3) keypoint array, compute every
    biomechanical metric we can derive in 2D. Returns dict of
    {metric_name: value or np.nan}, and a separate dict of
    {joint_name: confidence}.
    """
    def pt(name):
        return kp[KP[name], :2]

    def vis(name):
        return kp[KP[name], 2] > conf_thresh

    def conf(name):
        return float(kp[KP[name], 2])

    m = {}
    confidences = {name: conf(name) for name in KP.keys()}

    # ── Knee flexion (both sides) ──
    for side in ["l", "r"]:
        hip, knee, ankle = f"{side}_hip", f"{side}_knee", f"{side}_ankle"
        if vis(hip) and vis(knee) and vis(ankle):
            m[f"{side}_knee_flexion"] = angle_between(pt(hip), pt(knee), pt(ankle))
        else:
            m[f"{side}_knee_flexion"] = np.nan

    # ── Elbow flexion (both sides) ──
    for side in ["l", "r"]:
        shoulder, elbow, wrist = f"{side}_shoulder", f"{side}_elbow", f"{side}_wrist"
        if vis(shoulder) and vis(elbow) and vis(wrist):
            m[f"{side}_elbow_flexion"] = angle_between(pt(shoulder), pt(elbow), pt(wrist))
        else:
            m[f"{side}_elbow_flexion"] = np.nan

    # ── Shoulder abduction proxy (upper arm angle from vertical) ──
    for side in ["l", "r"]:
        shoulder, elbow = f"{side}_shoulder", f"{side}_elbow"
        if vis(shoulder) and vis(elbow):
            vec = pt(elbow) - pt(shoulder)
            m[f"{side}_shoulder_abduction"] = angle_from_vertical(vec)
        else:
            m[f"{side}_shoulder_abduction"] = np.nan

    # ── Trunk forward tilt (shoulder midpoint -> hip midpoint, vs vertical) ──
    if vis("l_shoulder") and vis("r_shoulder") and vis("l_hip") and vis("r_hip"):
        mid_sh  = (pt("l_shoulder") + pt("r_shoulder")) / 2
        mid_hip = (pt("l_hip") + pt("r_hip")) / 2
        trunk_vec = mid_sh - mid_hip
        m["trunk_forward_tilt"] = angle_from_vertical(trunk_vec)
        # lateral lean: horizontal component of trunk vector, normalized by trunk length
        trunk_len = np.linalg.norm(trunk_vec) + 1e-8
        m["trunk_lateral_lean"] = float(trunk_vec[0] / trunk_len)
    else:
        m["trunk_forward_tilt"] = np.nan
        m["trunk_lateral_lean"] = np.nan

    # ── Hip-shoulder separation (X-factor proxy) ──
    if vis("l_shoulder") and vis("r_shoulder") and vis("l_hip") and vis("r_hip"):
        sh_angle  = np.degrees(np.arctan2(pt("r_shoulder")[1] - pt("l_shoulder")[1],
                                            pt("r_shoulder")[0] - pt("l_shoulder")[0]))
        hip_angle = np.degrees(np.arctan2(pt("r_hip")[1] - pt("l_hip")[1],
                                            pt("r_hip")[0] - pt("l_hip")[0]))
        m["hip_shoulder_separation"] = float(abs(sh_angle - hip_angle))
        m["torso_rotation_proxy"]    = float(sh_angle)
        m["pelvis_rotation_proxy"]   = float(hip_angle)
    else:
        m["hip_shoulder_separation"] = np.nan
        m["torso_rotation_proxy"]    = np.nan
        m["pelvis_rotation_proxy"]   = np.nan

    # ── Wrist height (normalized-ish, raw pixel y — lower value = higher in frame) ──
    for side in ["l", "r"]:
        wrist = f"{side}_wrist"
        m[f"{side}_wrist_height_px"] = float(pt(wrist)[1]) if vis(wrist) else np.nan
        m[f"{side}_wrist_x_px"]      = float(pt(wrist)[0]) if vis(wrist) else np.nan

    # ── Semantic aliases based on handedness (RHP: stride=left, throw=right) ──
    m["stride_knee_flexion"]        = m[f"{STRIDE_SIDE}_knee_flexion"]
    m["throwing_knee_flexion"]      = m[f"{THROW_SIDE}_knee_flexion"]
    m["throwing_elbow_flexion"]     = m[f"{THROW_SIDE}_elbow_flexion"]
    m["glove_elbow_flexion"]        = m[f"{STRIDE_SIDE}_elbow_flexion"]  # glove side = same as stride side for RHP body opp arm; note below
    m["throwing_shoulder_abduction"]= m[f"{THROW_SIDE}_shoulder_abduction"]
    m["glove_shoulder_abduction"]   = m[f"{STRIDE_SIDE}_shoulder_abduction"]
    m["release_wrist_height_px"]    = m[f"{THROW_SIDE}_wrist_height_px"]
    m["release_wrist_x_px"]         = m[f"{THROW_SIDE}_wrist_x_px"]

    return m, confidences


def check_phc_coverage(frames, start, end, sample_every=PHC_SAMPLE_EVERY):
    """Return (coverage_fraction, n_sampled) for all-three-visible across a window."""
    checked = 0
    hits = 0
    for i in range(start, min(end + 1, len(frames)), sample_every):
        roles = get_phc_roles(frames[i])
        checked += 1
        if EXPECTED.issubset(set(roles.keys())):
            hits += 1
    coverage = hits / checked if checked > 0 else 0.0
    return coverage, checked


# ── Resume support: load existing checkpoint if present ──
if os.path.exists(CHECKPOINT_PATH):
    existing = pd.read_csv(CHECKPOINT_PATH)
    done_play_ids = set(existing["play_id"])
    print(f"Found existing checkpoint with {len(existing)} pitches already processed. Resuming.")
else:
    existing = pd.DataFrame()
    done_play_ids = set()

results = existing.to_dict("records")

pitches_to_process = df[~df["play_id"].isin(done_play_ids)].reset_index(drop=True)
print(f"Processing {len(pitches_to_process)} remaining pitches ({len(done_play_ids)} already done)\n")

for i, row in enumerate(tqdm(pitches_to_process.itertuples(), total=len(pitches_to_process), desc="Pitches")):
    clip_path = row.clip_path
    if not isinstance(clip_path, str) or not os.path.exists(clip_path):
        continue

    result_row = {
        "play_id": row.play_id,
        "inning": row.inning,
        "pitch_number": row.pitch_number,
        "pitch_type": row.pitch_type_desc,
        "start_speed": row.start_speed,
        "end_speed": row.end_speed,
        "spin_rate": row.spin_rate,
    }

    frames = load_frames(clip_path, max_frames=WINDOW_END + 10)

    if len(frames) < WINDOW_END:
        result_row["status"] = "clip_too_short"
        results.append(result_row)
        continue

    # PHC coverage check across the full window
    phc_coverage, n_checked = check_phc_coverage(frames, WINDOW_START, WINDOW_END)
    result_row["phc_coverage"] = phc_coverage
    result_row["phc_n_sampled"] = n_checked

    if phc_coverage < PHC_COVERAGE_MIN:
        result_row["status"] = "phc_coverage_too_low"
        results.append(result_row)
        continue

    # Track pitcher through the window
    window_frames = frames[:WINDOW_END + 1]
    tracked = track_pitcher_through_clip(window_frames)

    pitcher_detect_rate = sum(1 for b, k in tracked if b is not None) / len(tracked)
    result_row["pitcher_detect_rate"] = pitcher_detect_rate

    # Extract metrics across the release band
    band_metrics = []
    band_confidences = []
    for fi in RELEASE_BAND:
        if fi >= len(tracked):
            continue
        box, kp = tracked[fi]
        if kp is None:
            continue
        m, c = compute_all_metrics(kp)
        band_metrics.append(m)
        band_confidences.append(c)

    if not band_metrics:
        result_row["status"] = "no_valid_release_frames"
        results.append(result_row)
        continue

    band_df = pd.DataFrame(band_metrics)
    conf_df = pd.DataFrame(band_confidences)

    # Mean and std per metric across the release band
    for col in band_df.columns:
        result_row[f"{col}_mean"] = band_df[col].mean()
        result_row[f"{col}_std"]  = band_df[col].std()

    # Mean confidence per joint across the release band
    for col in conf_df.columns:
        result_row[f"conf_{col}"] = conf_df[col].mean()

    result_row["n_release_frames_used"] = len(band_metrics)
    result_row["status"] = "ok"

    results.append(result_row)

    # Checkpoint periodically
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

# Final save
final_df = pd.DataFrame(results)
final_df.to_csv(CHECKPOINT_PATH, index=False)

print(f"\nDone. {len(final_df)} pitches in output.")
print(f"\nStatus breakdown:")
print(final_df["status"].value_counts())
print(f"\nSaved to: {CHECKPOINT_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  DIAGNOSTIC: is YOLO running on CPU or GPU? How long does
#  one inference actually take? This tells us how much the
#  narrowed-window fix will help, and whether it's worth also
#  switching model size or enabling GPU.
# ══════════════════════════════════════════════════════════

import torch
import time

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU — this is almost certainly the bottleneck.")

print(f"\nWhat device is pose_model actually using?")
print(f"pose_model.device: {pose_model.device}")

# Time a single pose inference on a real frame
test_frame = frames[179] if 'frames' in dir() else None
if test_frame is None:
    print("\n(no `frames` in memory — skipping timing test, just showing device info)")
else:
    t0 = time.time()
    _ = pose_model(test_frame, verbose=False, conf=0.3)
    t1 = time.time()
    print(f"\nSingle pose inference time: {t1 - t0:.3f}s")

    t0 = time.time()
    _ = phc_model(test_frame, verbose=False)
    t1 = time.time()
    print(f"Single PHC inference time: {t1 - t0:.3f}s")

In [ ]:
# ══════════════════════════════════════════════════════════
#  FAST DIAGNOSTIC PIPELINE — rewritten for speed on CPU-only.
#  Only ever touches frames in [130, 230]. Nothing outside that
#  range is loaded, decoded, or inferred on.
#
#  Estimated: ~35s/pitch x 84 pitches =~ 49 minutes total.
#
#  Requires (already in memory): df, pose_model, phc_model, KP,
#  track_pitcher_through_clip, get_phc_roles, EXPECTED
# ══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import cv2
import os
from tqdm import tqdm

WINDOW_START      = 130
WINDOW_END        = 230
RELEASE_BAND      = range(175, 184)   # 175-183 inclusive
POSE_TRACK_START  = 165               # buffer before release for IoU-lock to stabilize
POSE_TRACK_END    = 195               # buffer after release
N_PHC_CHECKPOINTS = 6                 # sparse — just enough to catch a real problem

CHECKPOINT_PATH   = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
CHECKPOINT_EVERY  = 10

STRIDE_SIDE = "l"   # RHP: lead/stride leg = left
THROW_SIDE  = "r"   # RHP: throwing arm = right


# ── Fast frame access via seeking (no full-clip decode) ──

def read_frame_at(video_path, frame_idx):
    """Seek directly to one frame and read it. No decoding of anything else."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def read_frame_range(video_path, start, end):
    """Seek to `start` once, then read sequentially through `end` (inclusive)."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    out = []
    for _ in range(end - start + 1):
        ret, frame = cap.read()
        if not ret:
            break
        out.append(frame)
    cap.release()
    return out  # out[0] corresponds to frame index `start`


# ── Metric computation (unchanged from before) ──

def angle_between(a, b, c):
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def angle_from_vertical(vec):
    vertical = np.array([0, -1])
    cos_a = np.dot(vec, vertical) / (np.linalg.norm(vec) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def compute_all_metrics(kp, conf_thresh=0.3):
    def pt(name): return kp[KP[name], :2]
    def vis(name): return kp[KP[name], 2] > conf_thresh
    def conf(name): return float(kp[KP[name], 2])

    m = {}
    confidences = {name: conf(name) for name in KP.keys()}

    for side in ["l", "r"]:
        hip, knee, ankle = f"{side}_hip", f"{side}_knee", f"{side}_ankle"
        m[f"{side}_knee_flexion"] = angle_between(pt(hip), pt(knee), pt(ankle)) if (vis(hip) and vis(knee) and vis(ankle)) else np.nan

    for side in ["l", "r"]:
        shoulder, elbow, wrist = f"{side}_shoulder", f"{side}_elbow", f"{side}_wrist"
        m[f"{side}_elbow_flexion"] = angle_between(pt(shoulder), pt(elbow), pt(wrist)) if (vis(shoulder) and vis(elbow) and vis(wrist)) else np.nan

    for side in ["l", "r"]:
        shoulder, elbow = f"{side}_shoulder", f"{side}_elbow"
        m[f"{side}_shoulder_abduction"] = angle_from_vertical(pt(elbow) - pt(shoulder)) if (vis(shoulder) and vis(elbow)) else np.nan

    if vis("l_shoulder") and vis("r_shoulder") and vis("l_hip") and vis("r_hip"):
        mid_sh  = (pt("l_shoulder") + pt("r_shoulder")) / 2
        mid_hip = (pt("l_hip") + pt("r_hip")) / 2
        trunk_vec = mid_sh - mid_hip
        m["trunk_forward_tilt"] = angle_from_vertical(trunk_vec)
        trunk_len = np.linalg.norm(trunk_vec) + 1e-8
        m["trunk_lateral_lean"] = float(trunk_vec[0] / trunk_len)

        sh_angle  = np.degrees(np.arctan2(pt("r_shoulder")[1]-pt("l_shoulder")[1], pt("r_shoulder")[0]-pt("l_shoulder")[0]))
        hip_angle = np.degrees(np.arctan2(pt("r_hip")[1]-pt("l_hip")[1], pt("r_hip")[0]-pt("l_hip")[0]))
        m["hip_shoulder_separation"] = float(abs(sh_angle - hip_angle))
        m["torso_rotation_proxy"]    = float(sh_angle)
        m["pelvis_rotation_proxy"]   = float(hip_angle)
    else:
        m["trunk_forward_tilt"] = np.nan
        m["trunk_lateral_lean"] = np.nan
        m["hip_shoulder_separation"] = np.nan
        m["torso_rotation_proxy"] = np.nan
        m["pelvis_rotation_proxy"] = np.nan

    for side in ["l", "r"]:
        wrist = f"{side}_wrist"
        m[f"{side}_wrist_height_px"] = float(pt(wrist)[1]) if vis(wrist) else np.nan
        m[f"{side}_wrist_x_px"]      = float(pt(wrist)[0]) if vis(wrist) else np.nan

    m["stride_knee_flexion"]         = m[f"{STRIDE_SIDE}_knee_flexion"]
    m["throwing_knee_flexion"]       = m[f"{THROW_SIDE}_knee_flexion"]
    m["throwing_elbow_flexion"]      = m[f"{THROW_SIDE}_elbow_flexion"]
    m["glove_elbow_flexion"]         = m[f"{STRIDE_SIDE}_elbow_flexion"]
    m["throwing_shoulder_abduction"] = m[f"{THROW_SIDE}_shoulder_abduction"]
    m["glove_shoulder_abduction"]    = m[f"{STRIDE_SIDE}_shoulder_abduction"]
    m["release_wrist_height_px"]     = m[f"{THROW_SIDE}_wrist_height_px"]
    m["release_wrist_x_px"]          = m[f"{THROW_SIDE}_wrist_x_px"]

    return m, confidences


def check_phc_sparse(video_path, start, end, n_checkpoints=N_PHC_CHECKPOINTS):
    """Check PHC visibility at a small number of seeked checkpoint frames."""
    checkpoints = np.linspace(start, end, n_checkpoints).astype(int)
    hits = 0
    checked = 0
    for fi in checkpoints:
        frame = read_frame_at(video_path, int(fi))
        if frame is None:
            continue
        roles = get_phc_roles(frame)
        checked += 1
        if EXPECTED.issubset(set(roles.keys())):
            hits += 1
    coverage = hits / checked if checked > 0 else 0.0
    return coverage, checked


# ── Resume support ──
if os.path.exists(CHECKPOINT_PATH):
    existing = pd.read_csv(CHECKPOINT_PATH)
    done_play_ids = set(existing["play_id"])
    print(f"Resuming: {len(existing)} pitches already done.")
else:
    existing = pd.DataFrame()
    done_play_ids = set()

results = existing.to_dict("records")
pitches_to_process = df[~df["play_id"].isin(done_play_ids)].reset_index(drop=True)
print(f"Processing {len(pitches_to_process)} remaining pitches\n")

for i, row in enumerate(tqdm(pitches_to_process.itertuples(), total=len(pitches_to_process), desc="Pitches")):
    clip_path = row.clip_path
    if not isinstance(clip_path, str) or not os.path.exists(clip_path):
        continue

    result_row = {
        "play_id": row.play_id, "inning": row.inning, "pitch_number": row.pitch_number,
        "pitch_type": row.pitch_type_desc, "start_speed": row.start_speed,
        "end_speed": row.end_speed, "spin_rate": row.spin_rate,
    }

    # Sparse PHC check — only 6 seeked frames, nothing else decoded
    phc_coverage, n_checked = check_phc_sparse(clip_path, WINDOW_START, WINDOW_END)
    result_row["phc_coverage"] = phc_coverage
    result_row["phc_n_sampled"] = n_checked

    if phc_coverage < 0.7:
        result_row["status"] = "phc_coverage_too_low"
        results.append(result_row)
        continue

    # Only read + track the narrow pose window (165-195), nothing outside it
    window_frames = read_frame_range(clip_path, POSE_TRACK_START, POSE_TRACK_END)
    if len(window_frames) < (RELEASE_BAND.stop - POSE_TRACK_START):
        result_row["status"] = "pose_window_too_short"
        results.append(result_row)
        continue

    tracked = track_pitcher_through_clip(window_frames)
    pitcher_detect_rate = sum(1 for b, k in tracked if b is not None) / len(tracked)
    result_row["pitcher_detect_rate"] = pitcher_detect_rate

    band_metrics = []
    band_confidences = []
    for fi in RELEASE_BAND:
        local_idx = fi - POSE_TRACK_START   # convert absolute frame -> index in window_frames
        if local_idx < 0 or local_idx >= len(tracked):
            continue
        box, kp = tracked[local_idx]
        if kp is None:
            continue
        m, c = compute_all_metrics(kp)
        band_metrics.append(m)
        band_confidences.append(c)

    if not band_metrics:
        result_row["status"] = "no_valid_release_frames"
        results.append(result_row)
        continue

    band_df = pd.DataFrame(band_metrics)
    conf_df = pd.DataFrame(band_confidences)

    for col in band_df.columns:
        result_row[f"{col}_mean"] = band_df[col].mean()
        result_row[f"{col}_std"]  = band_df[col].std()
    for col in conf_df.columns:
        result_row[f"conf_{col}"] = conf_df[col].mean()

    result_row["n_release_frames_used"] = len(band_metrics)
    result_row["status"] = "ok"
    results.append(result_row)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

final_df = pd.DataFrame(results)
final_df.to_csv(CHECKPOINT_PATH, index=False)

print(f"\nDone. {len(final_df)} pitches in output.")
print(f"\nStatus breakdown:")
print(final_df["status"].value_counts())
print(f"\nSaved to: {CHECKPOINT_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  FULL EXPLORATORY ANALYSIS
#  No assumptions about what "should" be significant — run the
#  same inning-level regression across every metric we computed
#  (both mean AND std/variability), rank by significance, and
#  see what the data actually shows.
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

CSV_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
df_diag = pd.read_csv(CSV_PATH)

df_ok = df_diag[df_diag["status"] == "ok"].copy()
print(f"Valid pitches: {len(df_ok)} / {len(df_diag)}")
print(f"Innings covered: {sorted(df_ok['inning'].unique())}")
print(f"Pitches per inning:\n{df_ok['inning'].value_counts().sort_index()}\n")

# ── Per-joint confidence audit ──
conf_cols = [c for c in df_ok.columns if c.startswith("conf_")]
print("="*60)
print("PER-JOINT CONFIDENCE AUDIT (mean across all valid pitches)")
print("="*60)
conf_summary = df_ok[conf_cols].mean().sort_values()
for joint, mean_conf in conf_summary.items():
    flag = "  <-- LOW, treat with caution" if mean_conf < 0.5 else ("  <-- borderline" if mean_conf < 0.65 else "")
    print(f"  {joint:<25} {mean_conf:.3f}{flag}")

reliable_joints = set(conf_summary[conf_summary >= 0.65].index.str.replace("conf_", ""))
print(f"\nJoints with mean confidence >= 0.65 (considered reliable): {sorted(reliable_joints)}")

# ── Outlier filtering (2.5x IQR, per pitch type) + inning-level regression ──
def iqr_filter(group, col, k=2.5):
    q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return group[(group[col] >= lo) & (group[col] <= hi)]

def regress_metric(df_in, col):
    """Filter by pitch type IQR, aggregate to per-inning mean, regress vs inning."""
    filtered_parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            filtered_parts.append(group)  # too few to filter meaningfully
            continue
        filtered_parts.append(iqr_filter(group, col))
    filtered = pd.concat(filtered_parts)

    inning_means = filtered.groupby("inning")[col].mean().dropna()
    if len(inning_means) < 3:
        return None  # not enough innings with data to regress

    slope, intercept, r, p, stderr = stats.linregress(inning_means.index, inning_means.values)
    return {
        "metric": col,
        "n_pitches_used": filtered[col].notna().sum(),
        "n_innings": len(inning_means),
        "slope": slope,
        "r_squared": r**2,
        "p_value": p,
        "start_val": inning_means.values[0],
        "end_val": inning_means.values[-1],
        "total_change": inning_means.values[-1] - inning_means.values[0],
    }

# Every "_mean" metric column
metric_cols = [c for c in df_ok.columns if c.endswith("_mean")]
# Also test start_speed (velocity) the same way for direct comparison
metric_cols_plus_velocity = metric_cols + ["start_speed"]

print("\n" + "="*60)
print("REGRESSION ACROSS ALL MEAN METRICS (ranked by p-value)")
print("="*60)
results = []
for col in metric_cols_plus_velocity:
    r = regress_metric(df_ok, col)
    if r is not None:
        results.append(r)

results_df = pd.DataFrame(results).sort_values("p_value")
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 150)
print(results_df.to_string(index=False))

# ── Same regression on STD (variability) columns — often more predictive per literature ──
std_cols = [c for c in df_ok.columns if c.endswith("_std")]
print("\n" + "="*60)
print("REGRESSION ACROSS ALL VARIABILITY (STD) METRICS (ranked by p-value)")
print("="*60)
std_results = []
for col in std_cols:
    r = regress_metric(df_ok, col)
    if r is not None:
        std_results.append(r)

std_results_df = pd.DataFrame(std_results).sort_values("p_value")
print(std_results_df.to_string(index=False))

# ── Save both result tables ──
results_df.to_csv(r"C:\Users\noahl\OneDrive\fatigue_cv\regression_results_means.csv", index=False)
std_results_df.to_csv(r"C:\Users\noahl\OneDrive\fatigue_cv\regression_results_std.csv", index=False)
print("\nSaved regression_results_means.csv and regression_results_std.csv")

# ── Direct stride vs throwing knee comparison (the open question) ──
print("\n" + "="*60)
print("STRIDE KNEE vs THROWING-SIDE KNEE — direct comparison")
print("="*60)
for col in ["stride_knee_flexion_mean", "throwing_knee_flexion_mean",
            "stride_knee_flexion_std", "throwing_knee_flexion_std"]:
    r = regress_metric(df_ok, col)
    if r:
        print(f"\n{col}:")
        for k, v in r.items():
            print(f"  {k}: {v}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  ROBUSTNESS / SANITY CHECKS ON THE REGRESSION RESULTS
#  Standalone — reads the CSV fresh.
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

CSV_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
df_diag = pd.read_csv(CSV_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

def iqr_filter(group, col, k=2.5):
    q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return group[(group[col] >= lo) & (group[col] <= hi)]

def inning_agg(df_in, col, agg="mean"):
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        parts.append(iqr_filter(group, col))
    filtered = pd.concat(parts)
    return filtered.groupby("inning")[col].agg(agg).dropna()

def regress(x, y):
    slope, intercept, r, p, stderr = stats.linregress(x, y)
    return slope, r**2, p


# ══════════════════════════════════════════════════
#  1. LEVERAGE CHECK — does dropping inning 6 (n=4, noisiest)
#     change the headline results?
# ══════════════════════════════════════════════════
print("="*65)
print("1. LEVERAGE CHECK: does inning 6 (only 4 pitches) drive the result?")
print("="*65)

for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    full = inning_agg(df_ok, col)
    slope_full, r2_full, p_full = regress(full.index, full.values)

    dropped = full[full.index != 6]
    slope_drop, r2_drop, p_drop = regress(dropped.index, dropped.values)

    print(f"\n{col}:")
    print(f"  Full (innings 1-6):     slope={slope_full:+.3f}  r²={r2_full:.3f}  p={p_full:.4f}")
    print(f"  Dropping inning 6:      slope={slope_drop:+.3f}  r²={r2_drop:.3f}  p={p_drop:.4f}")
    direction_flip = np.sign(slope_full) != np.sign(slope_drop)
    print(f"  Direction flips?        {direction_flip}")
    print(f"  Still significant (p<.05) without inning 6? {p_drop < 0.05}")


# ══════════════════════════════════════════════════
#  2. MULTIPLE COMPARISONS CORRECTION
#     45 regressions were run (24 means + 21 stds). At p<.05 alone,
#     ~2 false positives are expected by chance. Apply Benjamini-
#     Hochberg (FDR) and Bonferroni to see what survives.
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("2. MULTIPLE COMPARISONS CORRECTION (45 tests total)")
print("="*65)

metric_cols = [c for c in df_ok.columns if c.endswith("_mean") or c.endswith("_std")]
metric_cols = [c for c in metric_cols if c not in ["phc_coverage"]]  # exclude non-biomech cols
all_results = []
for col in metric_cols + ["start_speed"]:
    agg_series = inning_agg(df_ok, col)
    if len(agg_series) < 3:
        continue
    slope, r2, p = regress(agg_series.index, agg_series.values)
    all_results.append({"metric": col, "slope": slope, "r_squared": r2, "p_value": p})

results_df = pd.DataFrame(all_results).sort_values("p_value").reset_index(drop=True)
n_tests = len(results_df)

# Bonferroni
results_df["bonferroni_thresh"] = 0.05 / n_tests
results_df["survives_bonferroni"] = results_df["p_value"] < results_df["bonferroni_thresh"]

# Benjamini-Hochberg (FDR)
results_df["bh_rank"] = np.arange(1, n_tests + 1)
results_df["bh_thresh"] = (results_df["bh_rank"] / n_tests) * 0.05
results_df["survives_bh"] = results_df["p_value"] < results_df["bh_thresh"]

print(f"\nTotal tests run: {n_tests}")
print(f"Bonferroni threshold (0.05/{n_tests}): {0.05/n_tests:.5f}")
print(f"\nResults surviving Bonferroni (very strict):")
print(results_df[results_df["survives_bonferroni"]][["metric", "p_value", "r_squared"]].to_string(index=False))
print(f"\nResults surviving Benjamini-Hochberg FDR (less strict, still principled):")
print(results_df[results_df["survives_bh"]][["metric", "p_value", "r_squared"]].to_string(index=False))
print(f"\nExpected false positives by chance alone at p<.05 with {n_tests} tests: ~{0.05*n_tests:.1f}")
print(f"Actual number with raw p<.05: {(results_df['p_value'] < 0.05).sum()}")


# ══════════════════════════════════════════════════
#  3. PITCH-TYPE BREAKDOWN — does the stride knee effect hold
#     within fastballs alone (his primary pitch), or is it partly
#     an artifact of pitch-mix shifting across innings?
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("3. PITCH-TYPE BREAKDOWN: fastballs only vs all pitches")
print("="*65)

print(f"\nPitch mix by inning:")
print(pd.crosstab(df_ok["inning"], df_ok["pitch_type"]))

ff_only = df_ok[df_ok["pitch_type"] == "Four-Seam Fastball"]
for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    ff_by_inning = ff_only.groupby("inning")[col].mean().dropna()
    if len(ff_by_inning) >= 3:
        slope, r2, p = regress(ff_by_inning.index, ff_by_inning.values)
        print(f"\n{col} (FASTBALLS ONLY, n={len(ff_only)}):")
        print(f"  slope={slope:+.3f}  r²={r2:.3f}  p={p:.4f}  n_innings={len(ff_by_inning)}")
    else:
        print(f"\n{col} (FASTBALLS ONLY): not enough innings with data to regress")


# ══════════════════════════════════════════════════
#  4. ARE MEAN AND STD CAPTURING THE SAME THING, OR
#     INDEPENDENT EVIDENCE? Correlate them directly.
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("4. STRIDE KNEE MEAN vs STD — same signal or independent?")
print("="*65)

mean_series = inning_agg(df_ok, "stride_knee_flexion_mean")
std_series  = inning_agg(df_ok, "stride_knee_flexion_std")
common_innings = mean_series.index.intersection(std_series.index)

corr, corr_p = stats.pearsonr(mean_series[common_innings], std_series[common_innings])
print(f"\nPearson correlation between inning-level mean and std: r={corr:.3f}, p={corr_p:.4f}")
if abs(corr) > 0.6:
    print("-> Fairly correlated: the two results may be partly capturing the same underlying drift,")
    print("   not fully independent lines of evidence.")
else:
    print("-> Not strongly correlated: these look like two genuinely separate signals,")
    print("   which strengthens the overall story rather than double-counting one effect.")


# ══════════════════════════════════════════════════
#  5. CONFIDENCE-WEIGHTED SANITY CHECK
#     Knee joints have the highest confidence of anything measured
#     (0.995+). Confirm this explicitly, and check if results hold
#     using only the highest-confidence pitches.
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("5. CONFIDENCE CHECK: does the result depend on shaky detections?")
print("="*65)

print(f"\nMean l_knee confidence across dataset: {df_ok['conf_l_knee'].mean():.4f}")
print(f"Min l_knee confidence across dataset:  {df_ok['conf_l_knee'].min():.4f}")

high_conf = df_ok[df_ok["conf_l_knee"] >= df_ok["conf_l_knee"].quantile(0.25)]
print(f"\nRestricting to top 75% confidence pitches (n={len(high_conf)} of {len(df_ok)}):")
for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    hc_series = inning_agg(high_conf, col)
    if len(hc_series) >= 3:
        slope, r2, p = regress(hc_series.index, hc_series.values)
        print(f"  {col}: slope={slope:+.3f}  r²={r2:.3f}  p={p:.4f}")

print("\n" + "="*65)
print("DONE — full robustness sweep complete")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════
#  PITCH-LEVEL ANALYSIS — uses ALL ~83 data points directly,
#  not collapsed into 6 inning-means. Predictor is cumulative
#  pitch count in the game, which is both more statistically
#  powerful and more biomechanically honest (fatigue accumulates
#  continuously, not in six discrete steps).
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)

df_ok = df_diag[df_diag["status"] == "ok"].copy()

# ── Reconstruct TRUE chronological order via at_bat_index + pitch_number ──
df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number", "inning"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

print(f"Reconstructed chronological order: {len(df_ok)} pitches, "
      f"game_pitch_number 1 to {df_ok['game_pitch_number'].max()}")
print(df_ok[["game_pitch_number", "inning", "pitch_number", "pitch_type"]].head(10).to_string(index=False))


def iqr_filter_all(df_in, col, k=2.5):
    """IQR filter per pitch type, but return the FULL filtered pitch-level
    dataframe (not aggregated) — preserves n for pitch-level regression."""
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k*iqr, q3 + k*iqr
        parts.append(group[(group[col] >= lo) & (group[col] <= hi)])
    return pd.concat(parts).sort_values("game_pitch_number")


def bootstrap_slope_ci(x, y, n_boot=5000, seed=42):
    """Bootstrap 95% CI for the regression slope by resampling pitches."""
    rng = np.random.default_rng(seed)
    n = len(x)
    slopes = []
    x, y = np.array(x), np.array(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        xb, yb = x[idx], y[idx]
        if np.std(xb) == 0:
            continue
        s, _, _, _, _ = stats.linregress(xb, yb)
        slopes.append(s)
    slopes = np.array(slopes)
    return np.percentile(slopes, 2.5), np.percentile(slopes, 97.5), slopes


# ══════════════════════════════════════════════════
#  1. PITCH-LEVEL REGRESSION on stride knee mean + std,
#     using cumulative pitch count, ALL ~83 points
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("1. PITCH-LEVEL REGRESSION (full n, cumulative pitch count)")
print("="*65)

for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std",
            "throwing_knee_flexion_mean", "throwing_knee_flexion_std",
            "start_speed"]:
    filt = iqr_filter_all(df_ok, col)
    x = filt["game_pitch_number"].values
    y = filt[col].values
    slope, intercept, r, p, stderr = stats.linregress(x, y)
    n = len(filt)
    print(f"\n{col} (n={n} pitches, filtered from {len(df_ok)}):")
    print(f"  slope = {slope:+.4f} per pitch  ({slope*83:+.2f} over full 83-pitch outing)")
    print(f"  r² = {r**2:.4f}   p = {p:.5f}   stderr = {stderr:.4f}")

    lo, hi, boot_slopes = bootstrap_slope_ci(x, y)
    print(f"  Bootstrap 95% CI on slope: [{lo:+.4f}, {hi:+.4f}]")
    ci_excludes_zero = (lo > 0) or (hi < 0)
    print(f"  95% CI excludes zero? {ci_excludes_zero}")


# ══════════════════════════════════════════════════
#  2. PITCH-TYPE-CONTROLLED REGRESSION (ANCOVA-style)
#     Does the trend hold after explicitly controlling for pitch
#     type as a covariate, at full pitch-level resolution?
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("2. PITCH-TYPE-CONTROLLED REGRESSION (full n, dummy-coded)")
print("="*65)

import itertools

def ancova_style(df_in, col):
    filt = iqr_filter_all(df_in, col).dropna(subset=[col])
    dummies = pd.get_dummies(filt["pitch_type"], drop_first=True, dtype=float)
    X = np.column_stack([filt["game_pitch_number"].values] + [dummies[c].values for c in dummies.columns])
    X = np.column_stack([np.ones(len(X)), X])  # intercept
    y = filt[col].values

    # OLS via lstsq
    coefs, residuals, rank, sv = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coefs
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan

    n, k = X.shape
    dof = n - k
    mse = ss_res / dof if dof > 0 else np.nan
    try:
        cov = mse * np.linalg.inv(X.T @ X)
        se_slope = np.sqrt(cov[1, 1])
        t_stat = coefs[1] / se_slope
        p_slope = 2 * (1 - stats.t.cdf(abs(t_stat), dof))
    except np.linalg.LinAlgError:
        se_slope, p_slope = np.nan, np.nan

    return coefs[1], r2, p_slope, n, list(dummies.columns)

for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    slope, r2, p, n, pitch_types_dummied = ancova_style(df_ok, col)
    print(f"\n{col} (n={n}, controlling for pitch type: {pitch_types_dummied}):")
    print(f"  pitch-count slope = {slope:+.4f}   r² (model) = {r2:.4f}   p (slope) = {p:.5f}")


# ══════════════════════════════════════════════════
#  3. SPEARMAN RANK CORRELATION — nonparametric, robust to
#     outliers and doesn't assume linearity
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("3. SPEARMAN RANK CORRELATION (nonparametric, full n)")
print("="*65)

for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    filt = iqr_filter_all(df_ok, col).dropna(subset=[col])
    rho, p_spear = stats.spearmanr(filt["game_pitch_number"], filt[col])
    print(f"\n{col} (n={len(filt)}):")
    print(f"  Spearman rho = {rho:+.4f}   p = {p_spear:.5f}")


# ══════════════════════════════════════════════════
#  4. COMPARE: pitch-level (full n) vs inning-aggregated (n=6)
#     side by side, to see how much power we recovered
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("4. SUMMARY: pitch-level (full n) vs inning-aggregated (n=6)")
print("="*65)
print("\n(Inning-aggregated numbers from the earlier analysis, for reference:")
print(" stride_knee_flexion_mean: r²=0.830, p=0.0115 (n=6 inning-means)")
print(" stride_knee_flexion_std:  r²=0.905, p=0.0035 (n=6 inning-means))")
print("\nSee section 1 above for the pitch-level (full n) equivalents.")

print("\n" + "="*65)
print("DONE")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════
#  ADDITIONAL CHECKS: non-linearity, early-vs-late split,
#  release point drift, spin rate — all at full pitch-level (n~83)
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

def iqr_filter_all(df_in, col, k=2.5):
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k*iqr, q3 + k*iqr
        parts.append(group[(group[col] >= lo) & (group[col] <= hi)])
    return pd.concat(parts).sort_values("game_pitch_number")


# ══════════════════════════════════════════════════
#  1. NON-LINEARITY CHECK: linear vs quadratic fit
#     Is this smooth drift, or a late-game inflection?
# ══════════════════════════════════════════════════
print("="*65)
print("1. LINEAR vs QUADRATIC FIT — smooth drift or late-game inflection?")
print("="*65)

def compare_linear_quadratic(x, y):
    # Linear
    lin_coefs = np.polyfit(x, y, 1)
    lin_pred = np.polyval(lin_coefs, x)
    ss_res_lin = np.sum((y - lin_pred)**2)

    # Quadratic
    quad_coefs = np.polyfit(x, y, 2)
    quad_pred = np.polyval(quad_coefs, x)
    ss_res_quad = np.sum((y - quad_pred)**2)

    ss_tot = np.sum((y - y.mean())**2)
    r2_lin = 1 - ss_res_lin/ss_tot
    r2_quad = 1 - ss_res_quad/ss_tot

    # F-test for whether quadratic term significantly improves fit
    n = len(x)
    df1 = 1  # one extra parameter
    df2 = n - 3
    f_stat = ((ss_res_lin - ss_res_quad) / df1) / (ss_res_quad / df2)
    p_f = 1 - stats.f.cdf(f_stat, df1, df2)

    return r2_lin, r2_quad, f_stat, p_f, quad_coefs

for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    filt = iqr_filter_all(df_ok, col).dropna(subset=[col])
    x, y = filt["game_pitch_number"].values, filt[col].values
    r2_lin, r2_quad, f_stat, p_f, quad_coefs = compare_linear_quadratic(x, y)
    print(f"\n{col}:")
    print(f"  Linear r²:    {r2_lin:.4f}")
    print(f"  Quadratic r²: {r2_quad:.4f}")
    print(f"  F-test for quadratic improvement: F={f_stat:.3f}, p={p_f:.4f}")
    print(f"  Quadratic term significant? {'YES — non-linear pattern' if p_f < 0.05 else 'No — linear is adequate'}")
    if p_f < 0.05:
        vertex_x = -quad_coefs[1] / (2*quad_coefs[0])
        print(f"  Curve vertex (inflection point) near pitch #{vertex_x:.0f}")


# ══════════════════════════════════════════════════
#  2. EARLY vs LATE HALF SPLIT — simple before/after story
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("2. EARLY HALF vs LATE HALF (Welch's t-test, unequal variance)")
print("="*65)

midpoint = df_ok["game_pitch_number"].max() // 2
for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std",
            "start_speed", "spin_rate"]:
    filt = iqr_filter_all(df_ok, col).dropna(subset=[col])
    early = filt[filt["game_pitch_number"] <= midpoint][col]
    late  = filt[filt["game_pitch_number"] > midpoint][col]

    t_stat, p_val = stats.ttest_ind(early, late, equal_var=False)
    cohens_d = (late.mean() - early.mean()) / np.sqrt((early.std()**2 + late.std()**2) / 2)

    print(f"\n{col}:")
    print(f"  Early half (n={len(early)}): mean={early.mean():.2f}, std={early.std():.2f}")
    print(f"  Late half  (n={len(late)}):  mean={late.mean():.2f}, std={late.std():.2f}")
    print(f"  Welch's t={t_stat:.3f}, p={p_val:.5f}, Cohen's d={cohens_d:+.3f}")
    effect_size_label = "small" if abs(cohens_d) < 0.5 else ("medium" if abs(cohens_d) < 0.8 else "large")
    print(f"  Effect size: {effect_size_label}")


# ══════════════════════════════════════════════════
#  3. RELEASE POINT DRIFT — direct literature connection
#     (horizontal release position variability flagged as
#     injury-risk marker in prior research)
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("3. RELEASE POINT DRIFT (horizontal release position)")
print("="*65)

for col in ["release_wrist_x_px_mean", "release_wrist_x_px_std",
            "release_wrist_height_px_mean", "release_wrist_height_px_std"]:
    if col not in df_ok.columns:
        continue
    filt = iqr_filter_all(df_ok, col).dropna(subset=[col])
    x, y = filt["game_pitch_number"].values, filt[col].values
    slope, intercept, r, p, stderr = stats.linregress(x, y)
    print(f"\n{col} (n={len(filt)}):")
    print(f"  slope={slope:+.4f} per pitch   r²={r**2:.4f}   p={p:.5f}")


# ══════════════════════════════════════════════════
#  4. SPIN RATE TREND — classic fatigue marker from the literature
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("4. SPIN RATE TREND")
print("="*65)

filt = iqr_filter_all(df_ok, "spin_rate").dropna(subset=["spin_rate"])
x, y = filt["game_pitch_number"].values, filt["spin_rate"].values
slope, intercept, r, p, stderr = stats.linregress(x, y)
print(f"\nspin_rate (n={len(filt)}):")
print(f"  slope={slope:+.4f} per pitch  ({slope*83:+.1f} over full outing)")
print(f"  r²={r**2:.4f}   p={p:.5f}")

print("\n" + "="*65)
print("DONE")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════
#  CORRECTING THE VARIABILITY INTERPRETATION + CONFOUND CHECK
#
#  stride_knee_flexion_std (as computed) = std across the 9
#  release-band frames WITHIN one pitch. This checks:
#    1. Whether that within-pitch instability is confounded by
#       detection quality degrading over the game (not mechanical)
#    2. What the CORRECT across-pitch consistency metric shows
#       (variance of the per-pitch release angle across DIFFERENT
#       pitches, early-half vs late-half)
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

def iqr_filter_all(df_in, col, k=2.5):
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k*iqr, q3 + k*iqr
        parts.append(group[(group[col] >= lo) & (group[col] <= hi)])
    return pd.concat(parts).sort_values("game_pitch_number")


# ══════════════════════════════════════════════════
#  1. CONFOUND CHECK: does detection quality degrade over
#     the game? If so, the within-pitch std trend could be
#     partly/wholly a measurement artifact, not mechanical.
# ══════════════════════════════════════════════════
print("="*65)
print("1. CONFOUND CHECK: does detection quality trend downward?")
print("="*65)

quality_cols = ["phc_coverage", "pitcher_detect_rate", "conf_l_knee", "conf_l_hip", "conf_l_ankle"]
for col in quality_cols:
    if col not in df_ok.columns:
        continue
    sub = df_ok.dropna(subset=[col])
    slope, intercept, r, p, stderr = stats.linregress(sub["game_pitch_number"], sub[col])
    print(f"\n{col} (n={len(sub)}):")
    print(f"  slope={slope:+.6f} per pitch   r²={r**2:.4f}   p={p:.4f}")
    if p < 0.05:
        direction = "DEGRADING" if slope < 0 else "IMPROVING"
        print(f"  >>> SIGNIFICANT TREND ({direction}) — this is a real confound to disclose")
    else:
        print(f"  No significant trend — not a confound for the knee-instability finding")

# Direct correlation: does within-pitch std track detection confidence?
print("\n--- Direct correlation: within-pitch std vs release-band confidence ---")
sub = df_ok.dropna(subset=["stride_knee_flexion_std", "conf_l_knee"])
r, p = stats.pearsonr(sub["conf_l_knee"], sub["stride_knee_flexion_std"])
print(f"conf_l_knee vs stride_knee_flexion_std: r={r:+.3f}, p={p:.4f}")
print("(A strong NEGATIVE correlation here would mean lower confidence -> higher std,")
print(" i.e., the 'instability' finding might just be tracking detection noise.)")


# ══════════════════════════════════════════════════
#  2. THE CORRECT ACROSS-PITCH CONSISTENCY METRIC
#     Does the per-pitch release angle become LESS repeatable
#     across different pitches as the game goes on?
#     (This is what the literature on "loss of repeatability"
#     actually refers to — not within-pitch frame noise.)
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("2. ACROSS-PITCH CONSISTENCY (the metric that matches the literature)")
print("="*65)

filt = iqr_filter_all(df_ok, "stride_knee_flexion_mean").dropna(subset=["stride_knee_flexion_mean"])
midpoint = filt["game_pitch_number"].max() // 2
early_vals = filt[filt["game_pitch_number"] <= midpoint]["stride_knee_flexion_mean"]
late_vals  = filt[filt["game_pitch_number"] > midpoint]["stride_knee_flexion_mean"]

print(f"\nEarly half (n={len(early_vals)}): std across DIFFERENT pitches = {early_vals.std():.3f}°")
print(f"Late half  (n={len(late_vals)}): std across DIFFERENT pitches = {late_vals.std():.3f}°")

# Levene's test for equality of variances (more robust than F-test to non-normality)
levene_stat, levene_p = stats.levene(early_vals, late_vals)
print(f"\nLevene's test for equal variance: stat={levene_stat:.3f}, p={levene_p:.4f}")
if levene_p < 0.05:
    direction = "INCREASES" if late_vals.std() > early_vals.std() else "DECREASES"
    print(f">>> Across-pitch consistency {direction} significantly in the second half")
    print(">>> This IS the correct evidence for a 'loss of repeatability' claim")
else:
    print(">>> No significant change in across-pitch consistency")
    print(">>> The 'loss of repeatability' framing would NOT be supported by this metric —")
    print(">>> only the within-pitch (frame-level) instability finding would stand")

# Also check with a rolling window for a smoother view
print("\n--- Rolling 15-pitch window std of stride_knee_flexion_mean, across game ---")
filt_sorted = filt.sort_values("game_pitch_number")
rolling_std = filt_sorted.set_index("game_pitch_number")["stride_knee_flexion_mean"].rolling(15, min_periods=8).std()
rolling_std = rolling_std.dropna()
slope, intercept, r, p, stderr = stats.linregress(rolling_std.index, rolling_std.values)
print(f"Rolling-window std trend: slope={slope:+.4f}/pitch, r²={r**2:.4f}, p={p:.4f}")

print("\n" + "="*65)
print("DONE")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════
#  IS THE FASTBALL VELOCITY DECLINE A CONSTANT RATE, OR LUMPY?
#  Checked at both pitch-level and inning-level resolution.
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

ff = df_ok[df_ok["pitch_type"] == "Four-Seam Fastball"].dropna(subset=["start_speed"]).copy()
ff = ff.sort_values("game_pitch_number").reset_index(drop=True)


# ══════════════════════════════════════════════════
#  1. PITCH-LEVEL: linear vs quadratic fit + residual pattern check
# ══════════════════════════════════════════════════
print("="*65)
print("1. PITCH-LEVEL: is the decline linear (constant rate) or curved?")
print("="*65)

x = ff["game_pitch_number"].values
y = ff["start_speed"].values

lin_coefs = np.polyfit(x, y, 1)
lin_pred = np.polyval(lin_coefs, x)
resid = y - lin_pred

quad_coefs = np.polyfit(x, y, 2)
quad_pred = np.polyval(quad_coefs, x)

ss_res_lin = np.sum((y - lin_pred)**2)
ss_res_quad = np.sum((y - quad_pred)**2)
ss_tot = np.sum((y - y.mean())**2)
r2_lin = 1 - ss_res_lin/ss_tot
r2_quad = 1 - ss_res_quad/ss_tot

n = len(x)
f_stat = ((ss_res_lin - ss_res_quad) / 1) / (ss_res_quad / (n - 3))
p_f = 1 - stats.f.cdf(f_stat, 1, n - 3)

print(f"Linear r²:    {r2_lin:.4f}")
print(f"Quadratic r²: {r2_quad:.4f}")
print(f"F-test for quadratic improvement: F={f_stat:.3f}, p={p_f:.4f}")
print(f"{'Non-linear pattern detected' if p_f < 0.05 else 'Linear (constant rate) is adequate — no evidence of curvature'}")

# Runs test on residual signs — tests whether residuals are randomly
# scattered (consistent linear rate) or show streaks (lumpy/uneven decline)
signs = np.sign(resid)
signs = signs[signs != 0]
n_pos = np.sum(signs > 0)
n_neg = np.sum(signs < 0)
runs = 1 + np.sum(signs[1:] != signs[:-1])
n_total = n_pos + n_neg
expected_runs = (2 * n_pos * n_neg) / n_total + 1
var_runs = (2*n_pos*n_neg*(2*n_pos*n_neg - n_total)) / (n_total**2 * (n_total - 1))
z_runs = (runs - expected_runs) / np.sqrt(var_runs) if var_runs > 0 else np.nan
p_runs = 2 * (1 - stats.norm.cdf(abs(z_runs)))

print(f"\nRuns test on residuals: observed runs={runs}, expected={expected_runs:.1f}, z={z_runs:.3f}, p={p_runs:.4f}")
print(f"{'Residuals show non-random streaking (LUMPY decline)' if p_runs < 0.05 else 'Residuals look randomly scattered — decline IS consistent/steady'}")


# ══════════════════════════════════════════════════
#  2. INNING-LEVEL: how much does the per-inning drop vary?
# ══════════════════════════════════════════════════
print("\n" + "="*65)
print("2. INNING-LEVEL: consistency of the inning-to-inning drop")
print("="*65)

inning_means = ff.groupby("inning")["start_speed"].mean()
print(f"\nPer-inning fastball velocity means:")
print(inning_means.to_string())

deltas = inning_means.diff().dropna()
print(f"\nInning-to-inning changes (mph):")
print(deltas.to_string())
print(f"\nMean delta: {deltas.mean():+.3f} mph/inning")
print(f"Std of deltas: {deltas.std():.3f} mph")
cv = abs(deltas.std() / deltas.mean()) if deltas.mean() != 0 else np.nan
print(f"Coefficient of variation of deltas: {cv:.2f}")
print(f"({'Fairly consistent rate' if cv < 1 else 'Inconsistent / lumpy rate — high variability between innings'})")

slope_inn, intercept_inn, r_inn, p_inn, stderr_inn = stats.linregress(inning_means.index, inning_means.values)
print(f"\nInning-level linear fit: slope={slope_inn:+.4f} mph/inning, r²={r_inn**2:.4f}, p={p_inn:.4f}")

print("\n" + "="*65)
print("DONE")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════
#  IS VELOCITY FLAT AFTER THE NORMAL FIRST-INNING SETTLING
#  PERIOD, WHILE MECHANICS KEEP DRIFTING UNDERNEATH?
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

def iqr_filter_all(df_in, col, k=2.5):
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k*iqr, q3 + k*iqr
        parts.append(group[(group[col] >= lo) & (group[col] <= hi)])
    return pd.concat(parts).sort_values("game_pitch_number")

post_settle = df_ok[df_ok["inning"] >= 2].copy()
ff_post = post_settle[post_settle["pitch_type"] == "Four-Seam Fastball"].dropna(subset=["start_speed"])

print("="*65)
print("FASTBALL VELOCITY: innings 2-6 only (excluding first-inning settling)")
print("="*65)
x = ff_post["game_pitch_number"].values
y = ff_post["start_speed"].values
slope, intercept, r, p, stderr = stats.linregress(x, y)
print(f"n={len(ff_post)} pitches")
print(f"slope={slope:+.4f} mph/pitch   r²={r**2:.4f}   p={p:.4f}")
print(f"{'STILL significant after removing inning 1' if p < 0.05 else 'NOT significant once inning 1 settling is excluded'}")

print(f"\n--- Comparison ---")
ff_full = df_ok[df_ok["pitch_type"] == "Four-Seam Fastball"].dropna(subset=["start_speed"])
slope_f, _, r_f, p_f, _ = stats.linregress(ff_full["game_pitch_number"], ff_full["start_speed"])
print(f"Full game (n={len(ff_full)}):        slope={slope_f:+.4f}  r²={r_f**2:.4f}  p={p_f:.4f}")
print(f"Post-settling only (n={len(ff_post)}): slope={slope:+.4f}  r²={r**2:.4f}  p={p:.4f}")

print("\n" + "="*65)
print("STRIDE KNEE MECHANICS: innings 2-6 only (same restriction)")
print("="*65)

for col in ["stride_knee_flexion_mean", "stride_knee_flexion_std"]:
    filt_full = iqr_filter_all(df_ok, col).dropna(subset=[col])
    filt_post = iqr_filter_all(post_settle, col).dropna(subset=[col])

    slope_full, _, r_full, p_full, _ = stats.linregress(filt_full["game_pitch_number"], filt_full[col])
    slope_post, _, r_post, p_post, _ = stats.linregress(filt_post["game_pitch_number"], filt_post[col])

    print(f"\n{col}:")
    print(f"  Full game (n={len(filt_full)}):        slope={slope_full:+.4f}  r²={r_full**2:.4f}  p={p_full:.5f}")
    print(f"  Post-settling only (n={len(filt_post)}): slope={slope_post:+.4f}  r²={r_post**2:.4f}  p={p_post:.5f}")
    print(f"  {'Signal HOLDS UP without inning 1' if p_post < 0.05 else 'Signal DISAPPEARS without inning 1'}")

print("\n" + "="*65)
print("DONE")
print("="*65)

In [ ]:
# ══════════════════════════════════════════════════════════
#  REAL-VIDEO GIF BUILDER — 5 formats, all using your actual
#  broadcast clips and pose tracking, not synthetic data.
#
#  Reuses functions already in your session: read_frame_range,
#  track_pitcher_through_clip, KP. Run in the same kernel.
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Syracuse palette (BGR order for cv2, since cv2 uses BGR not RGB)
ORANGE_BGR = (0, 105, 247)   # #F76900
NAVY_BGR   = (84, 14, 0)     # #000E54
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

def angle_between(a, b, c):
    ba = a - b; bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    """Save a list of BGR (cv2-style) frames as a looping GIF."""
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")


def draw_skeleton(frame_bgr, kp, box=None, line_color=ORANGE_BGR, pt_color=NAVY_BGR,
                   line_thickness=3, pt_radius=6, conf_thresh=0.3):
    frame = frame_bgr.copy()
    if box is not None:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(frame, pa, pb, line_color, line_thickness, cv2.LINE_AA)
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(frame, pt, pt_radius, pt_color, -1, cv2.LINE_AA)
    return frame


# ══════════════════════════════════════════════════
#  FORMAT A — Real skeleton overlaid on real broadcast video,
#  single delivery, looping.
# ══════════════════════════════════════════════════
def build_format_A(clip_path, start=165, end=195, out_name="A_skeleton_overlay.gif", fps=15):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    annotated = [draw_skeleton(f, kp, box) for f, (box, kp) in zip(frames, tracked)]
    frames_to_gif(annotated, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  FORMAT B — Two real clips side by side (e.g. early vs late
#  inning), each with skeleton overlay, synced frame-by-frame.
# ══════════════════════════════════════════════════
def build_format_B(clip_path_left, clip_path_right, start=165, end=195,
                    label_left="INNING 1", label_right="INNING 6",
                    out_name="B_side_by_side.gif", fps=15):
    frames_l = read_frame_range(clip_path_left, start, end)
    frames_r = read_frame_range(clip_path_right, start, end)
    tracked_l = track_pitcher_through_clip(frames_l)
    tracked_r = track_pitcher_through_clip(frames_r)

    n = min(len(frames_l), len(frames_r))
    combined = []
    for i in range(n):
        al = draw_skeleton(frames_l[i], tracked_l[i][1], tracked_l[i][0])
        ar = draw_skeleton(frames_r[i], tracked_r[i][1], tracked_r[i][0])

        h = min(al.shape[0], ar.shape[0])
        al = cv2.resize(al, (int(al.shape[1] * h / al.shape[0]), h))
        ar = cv2.resize(ar, (int(ar.shape[1] * h / ar.shape[0]), h))

        cv2.putText(al, label_left, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, WHITE_BGR, 3, cv2.LINE_AA)
        cv2.putText(al, label_left, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, NAVY_BGR, 2, cv2.LINE_AA)
        cv2.putText(ar, label_right, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, WHITE_BGR, 3, cv2.LINE_AA)
        cv2.putText(ar, label_right, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, ORANGE_BGR, 2, cv2.LINE_AA)

        divider = np.full((h, 4, 3), 255, dtype=np.uint8)
        combined.append(np.hstack([al, divider, ar]))

    frames_to_gif(combined, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  FORMAT C — Real video + skeleton on top, PLUS a live-updating
#  line chart underneath tracking a metric (e.g. knee angle)
#  frame by frame, with a moving marker.
# ══════════════════════════════════════════════════
def build_format_C(clip_path, start=165, end=195, out_name="C_video_plus_graph.gif",
                    fps=15, metric_side="l"):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)

    metric_vals = []
    for box, kp in tracked:
        if kp is None:
            metric_vals.append(np.nan)
            continue
        hip = kp[KP[f"{metric_side}_hip"], :2]
        knee = kp[KP[f"{metric_side}_knee"], :2]
        ankle = kp[KP[f"{metric_side}_ankle"], :2]
        metric_vals.append(angle_between(hip, knee, ankle))
    metric_vals = np.array(metric_vals)

    n = len(frames)
    combined = []
    for i in range(n):
        annotated = draw_skeleton(frames[i], tracked[i][1], tracked[i][0])

        fig, ax = plt.subplots(figsize=(annotated.shape[1] / 100, 2.2), dpi=100)
        fig.patch.set_facecolor("white")
        ax.plot(range(n), metric_vals, color="#ADB3B8", linewidth=1.5)
        ax.plot(range(i + 1), metric_vals[:i + 1], color="#F76900", linewidth=3)
        if not np.isnan(metric_vals[i]):
            ax.scatter([i], [metric_vals[i]], color="#000E54", s=80, zorder=5)
        ax.set_ylabel("Knee °", fontsize=9)
        ax.set_xticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()
        fig.canvas.draw()
        chart_img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(
            fig.canvas.get_width_height()[::-1] + (4,))
        chart_img = cv2.cvtColor(chart_img, cv2.COLOR_RGBA2BGR)
        plt.close(fig)

        chart_resized = cv2.resize(chart_img, (annotated.shape[1], chart_img.shape[0]))
        combined.append(np.vstack([annotated, chart_resized]))

    frames_to_gif(combined, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  FORMAT D — "HUD" telemetry style: glowing joints, live
#  angle readout burned into the real video.
# ══════════════════════════════════════════════════
def build_format_D(clip_path, start=165, end=195, out_name="D_hud_telemetry.gif",
                    fps=15, metric_side="l"):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)

    combined = []
    for i, (box, kp) in enumerate(tracked):
        base = frames[i]
        overlay = base.copy()

        if kp is not None:
            for a, b in SKELETON_EDGES:
                ia, ib = KP[a], KP[b]
                if kp[ia, 2] > 0.3 and kp[ib, 2] > 0.3:
                    pa = tuple(int(v) for v in kp[ia, :2])
                    pb = tuple(int(v) for v in kp[ib, :2])
                    cv2.line(overlay, pa, pb, ORANGE_BGR, 5, cv2.LINE_AA)
            for name, idx in KP.items():
                if kp[idx, 2] > 0.3:
                    pt = tuple(int(v) for v in kp[idx, :2])
                    cv2.circle(overlay, pt, 12, ORANGE_BGR, 2, cv2.LINE_AA)
                    cv2.circle(overlay, pt, 4, WHITE_BGR, -1, cv2.LINE_AA)

            hip = kp[KP[f"{metric_side}_hip"], :2]
            knee = kp[KP[f"{metric_side}_knee"], :2]
            ankle = kp[KP[f"{metric_side}_ankle"], :2]
            angle = angle_between(hip, knee, ankle)
            label = f"STRIDE KNEE: {angle:.0f} deg"
            cv2.putText(overlay, label, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 5, cv2.LINE_AA)
            cv2.putText(overlay, label, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, WHITE_BGR, 2, cv2.LINE_AA)

        blended = cv2.addWeighted(overlay, 0.8, base, 0.2, 0)
        combined.append(blended)

    frames_to_gif(combined, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  FORMAT E — Ghost-trail / stroboscopic multi-exposure:
#  overlays several past frames' skeletons with fading opacity
#  on top of the current video frame, showing the whole motion path.
# ══════════════════════════════════════════════════
def build_format_E(clip_path, start=165, end=195, out_name="E_ghost_trail.gif",
                    fps=12, trail_length=6):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)

    combined = []
    for i in range(len(frames)):
        canvas = frames[i].copy().astype(np.float32)

        for j in range(max(0, i - trail_length), i + 1):
            box, kp = tracked[j]
            if kp is None:
                continue
            alpha = 1.0 - (i - j) / (trail_length + 1)
            skel_layer = np.zeros_like(canvas)
            for a, b in SKELETON_EDGES:
                ia, ib = KP[a], KP[b]
                if kp[ia, 2] > 0.3 and kp[ib, 2] > 0.3:
                    pa = tuple(int(v) for v in kp[ia, :2])
                    pb = tuple(int(v) for v in kp[ib, :2])
                    cv2.line(skel_layer, pa, pb, ORANGE_BGR, 3, cv2.LINE_AA)
            canvas = canvas + skel_layer.astype(np.float32) * alpha * 0.6

        canvas = np.clip(canvas, 0, 255).astype(np.uint8)
        combined.append(canvas)

    frames_to_gif(combined, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  RUN — pick an early-inning and a late-inning fastball
#  automatically from your data, and build all 5 formats.
# ══════════════════════════════════════════════════
early_pitch = df[(df["inning"] == 1) & (df["pitch_type_desc"] == "Four-Seam Fastball")].iloc[0]
late_pitch  = df[(df["inning"] == 6) & (df["pitch_type_desc"] == "Four-Seam Fastball")].iloc[-1]

print(f"Early pitch: inning {early_pitch['inning']}, clip {early_pitch['clip_path']}")
print(f"Late pitch:  inning {late_pitch['inning']}, clip {late_pitch['clip_path']}")

build_format_A(early_pitch["clip_path"], out_name="A_skeleton_overlay_early.gif")
build_format_B(early_pitch["clip_path"], late_pitch["clip_path"],
               label_left="INNING 1", label_right="INNING 6")
build_format_C(late_pitch["clip_path"], out_name="C_video_plus_graph_late.gif")
build_format_D(late_pitch["clip_path"], out_name="D_hud_telemetry_late.gif")
build_format_E(late_pitch["clip_path"], out_name="E_ghost_trail_late.gif")

print("\nAll 5 GIFs built. Check:", OUTPUT_DIR)

In [ ]:
# ══════════════════════════════════════════════════════════
#  MITOSIS-TRANSITION GIF
#
#  Loop 1: real broadcast video + skeleton overlaid, full delivery
#  Transition: frame 0 splits apart — video peels left, skeleton
#              (on white bg) peels right
#  Loop 2: same delivery plays again, side by side (video | skeleton)
#  Then cuts back to Loop 1 and repeats
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
#  (already in your session)
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)
NAVY_BGR   = (84, 14, 0)
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

def draw_skeleton_on(canvas, kp, box=None, line_color=ORANGE_BGR, pt_color=NAVY_BGR,
                      line_thickness=3, pt_radius=6, conf_thresh=0.3):
    canvas = canvas.copy()
    if box is not None:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 200, 0), 2)
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, line_color, line_thickness, cv2.LINE_AA)
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, pt_radius, pt_color, -1, cv2.LINE_AA)
    return canvas

def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")


def build_mitosis_gif(clip_path, start=165, end=195, out_name="mitosis_transition.gif",
                       fps=15, split_frames=14):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    n = len(frames)
    h, w = frames[0].shape[:2]

    all_output_frames = []

    # ── LOOP 1: combined skeleton-over-video, full delivery ──
    for i in range(n):
        combined = draw_skeleton_on(frames[i], tracked[i][1], tracked[i][0])
        all_output_frames.append(combined)

    # ── TRANSITION: frame 0 splits apart ──
    video_only = frames[0].copy()
    skeleton_only = np.full((h, w, 3), 255, dtype=np.uint8)  # white background
    skeleton_only = draw_skeleton_on(skeleton_only, tracked[0][1], box=None)

    half_w = w // 2
    for step in range(split_frames):
        t = step / (split_frames - 1)
        ease_t = t * t * (3 - 2 * t)  # smootherstep easing

        canvas = np.full((h, w, 3), 255, dtype=np.uint8)

        # Video panel shrinks from full width down to half width (left side)
        target_w = int(w * (1 - ease_t) + half_w * ease_t)
        vid_resized = cv2.resize(video_only, (target_w, h))
        canvas[:, 0:target_w] = vid_resized

        # Skeleton panel grows in from the right, 0 -> half width
        skel_target_w = int(half_w * ease_t)
        if skel_target_w > 0:
            skel_resized = cv2.resize(skeleton_only, (skel_target_w, h))
            canvas[:, w - skel_target_w:w] = skel_resized

        all_output_frames.append(canvas)

    # ── LOOP 2: side by side, same delivery replayed in sync ──
    for i in range(n):
        vid_panel = cv2.resize(frames[i], (half_w, h))
        skel_canvas = np.full((h, w - half_w, 3), 255, dtype=np.uint8)
        skel_panel = draw_skeleton_on(skel_canvas, tracked[i][1], box=None)
        skel_panel = cv2.resize(skel_panel, (w - half_w, h))

        divider = np.full((h, 3, 3), 200, dtype=np.uint8)
        side_by_side = np.hstack([vid_panel[:, :vid_panel.shape[1]-2], divider,
                                    skel_panel[:, 2:]])
        side_by_side = cv2.resize(side_by_side, (w, h))
        all_output_frames.append(side_by_side)

    frames_to_gif(all_output_frames, os.path.join(OUTPUT_DIR, out_name), fps=fps)
    print(f"Total frames in full loop: {len(all_output_frames)} "
          f"({n} combined + {split_frames} transition + {n} side-by-side)")


# ══════════════════════════════════════════════════
#  RUN — pick a pitch and build it
# ══════════════════════════════════════════════════
sample_pitch = df[df["pitch_type_desc"] == "Four-Seam Fastball"].iloc[0]
print(f"Building mitosis GIF from: {sample_pitch['clip_path']}")

build_mitosis_gif(sample_pitch["clip_path"])

In [ ]:
# ══════════════════════════════════════════════════════════
#  THREE ALTERNATIVE OVERLAY TREATMENTS (not skeleton lines)
#  Same pitch, same frame range, for direct comparison.
#  Each treatment gets: (a) overlaid on real video, (b) isolated
#  on white background — 6 GIFs total.
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)
NAVY_BGR   = (84, 14, 0)
WHITE_BGR  = (255, 255, 255)
GRAY_BGR   = (184, 179, 173)

def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")

def angle_between(a, b, c):
    ba = a - b; bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


# ══════════════════════════════════════════════════
#  TREATMENT A — Glow-dot joints, no connecting lines.
#  "Motion capture dot cloud" look.
# ══════════════════════════════════════════════════
def draw_glow_dots(canvas, kp, conf_thresh=0.3):
    canvas = canvas.copy()
    overlay = np.zeros_like(canvas)
    if kp is not None:
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(overlay, pt, 18, ORANGE_BGR, -1, cv2.LINE_AA)
        overlay = cv2.GaussianBlur(overlay, (0, 0), sigmaX=8)
        canvas = cv2.addWeighted(canvas, 1.0, overlay, 0.85, 0)
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, 4, WHITE_BGR, -1, cv2.LINE_AA)
                cv2.circle(canvas, pt, 4, NAVY_BGR, 1, cv2.LINE_AA)
    return canvas


# ══════════════════════════════════════════════════
#  TREATMENT B — Angle-arc focus at the stride knee only.
#  Faint context dots elsewhere, real attention on one joint.
# ══════════════════════════════════════════════════
def draw_angle_arc(canvas, kp, joint_side="l", conf_thresh=0.3):
    canvas = canvas.copy()
    if kp is None:
        return canvas

    # Faint context dots at all joints
    for name, idx in KP.items():
        if kp[idx, 2] > conf_thresh:
            pt = tuple(int(v) for v in kp[idx, :2])
            cv2.circle(canvas, pt, 3, GRAY_BGR, -1, cv2.LINE_AA)

    hip_i, knee_i, ankle_i = KP[f"{joint_side}_hip"], KP[f"{joint_side}_knee"], KP[f"{joint_side}_ankle"]
    if kp[hip_i,2] < conf_thresh or kp[knee_i,2] < conf_thresh or kp[ankle_i,2] < conf_thresh:
        return canvas

    hip = kp[hip_i, :2]; knee = kp[knee_i, :2]; ankle = kp[ankle_i, :2]
    angle = angle_between(hip, knee, ankle)

    v1 = hip - knee
    v2 = ankle - knee
    ang1 = np.degrees(np.arctan2(v1[1], v1[0]))
    ang2 = np.degrees(np.arctan2(v2[1], v2[0]))

    radius = 45
    center = tuple(int(v) for v in knee)
    cv2.ellipse(canvas, center, (radius, radius), 0, min(ang1,ang2), max(ang1,ang2),
                ORANGE_BGR, 4, cv2.LINE_AA)

    # Thicker lines just for the two segments forming this angle
    cv2.line(canvas, tuple(int(v) for v in hip), center, ORANGE_BGR, 4, cv2.LINE_AA)
    cv2.line(canvas, center, tuple(int(v) for v in ankle), ORANGE_BGR, 4, cv2.LINE_AA)
    cv2.circle(canvas, center, 7, NAVY_BGR, -1, cv2.LINE_AA)

    label = f"{angle:.0f} deg"
    label_pos = (center[0] + 20, center[1])
    cv2.putText(canvas, label, label_pos, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 4, cv2.LINE_AA)
    cv2.putText(canvas, label, label_pos, cv2.FONT_HERSHEY_SIMPLEX, 0.8, WHITE_BGR, 2, cv2.LINE_AA)

    return canvas


# ══════════════════════════════════════════════════
#  TREATMENT C — Filled segment highlight (thigh + shin as
#  translucent colored bands, biomechanical-diagram feel).
# ══════════════════════════════════════════════════
def draw_segment_fill(canvas, kp, joint_side="l", conf_thresh=0.3, band_width=14):
    canvas = canvas.copy()
    if kp is None:
        return canvas

    def band_polygon(p1, p2, width):
        p1, p2 = np.array(p1), np.array(p2)
        direction = p2 - p1
        length = np.linalg.norm(direction) + 1e-8
        perp = np.array([-direction[1], direction[0]]) / length * (width / 2)
        return np.array([p1 + perp, p2 + perp, p2 - perp, p1 - perp], dtype=np.int32)

    hip_i, knee_i, ankle_i = KP[f"{joint_side}_hip"], KP[f"{joint_side}_knee"], KP[f"{joint_side}_ankle"]
    overlay = canvas.copy()

    if kp[hip_i,2] > conf_thresh and kp[knee_i,2] > conf_thresh:
        poly = band_polygon(kp[hip_i,:2], kp[knee_i,:2], band_width)
        cv2.fillPoly(overlay, [poly], NAVY_BGR)
    if kp[knee_i,2] > conf_thresh and kp[ankle_i,2] > conf_thresh:
        poly = band_polygon(kp[knee_i,:2], kp[ankle_i,:2], band_width)
        cv2.fillPoly(overlay, [poly], ORANGE_BGR)

    canvas = cv2.addWeighted(overlay, 0.55, canvas, 0.45, 0)

    if kp[knee_i,2] > conf_thresh:
        pt = tuple(int(v) for v in kp[knee_i,:2])
        cv2.circle(canvas, pt, 6, WHITE_BGR, -1, cv2.LINE_AA)
        cv2.circle(canvas, pt, 6, (0,0,0), 1, cv2.LINE_AA)

    return canvas


# ══════════════════════════════════════════════════
#  BUILD: overlay-on-video AND isolated-on-white for each treatment
# ══════════════════════════════════════════════════
def build_treatment_pair(clip_path, draw_fn, name, start=165, end=195, fps=15, **draw_kwargs):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    h, w = frames[0].shape[:2]

    overlay_frames = [draw_fn(frames[i], tracked[i][1], **draw_kwargs) for i in range(len(frames))]
    frames_to_gif(overlay_frames, os.path.join(OUTPUT_DIR, f"{name}_over_video.gif"), fps=fps)

    isolated_frames = []
    for i in range(len(frames)):
        white_canvas = np.full((h, w, 3), 255, dtype=np.uint8)
        isolated_frames.append(draw_fn(white_canvas, tracked[i][1], **draw_kwargs))
    frames_to_gif(isolated_frames, os.path.join(OUTPUT_DIR, f"{name}_isolated.gif"), fps=fps)


# ══════════════════════════════════════════════════
#  RUN — same pitch as the mitosis GIF, all three treatments
# ══════════════════════════════════════════════════
sample_pitch = df[df["pitch_type_desc"] == "Four-Seam Fastball"].iloc[0]
print(f"Using clip: {sample_pitch['clip_path']}\n")

print("--- Treatment A: Glow dots ---")
build_treatment_pair(sample_pitch["clip_path"], draw_glow_dots, "A_glow_dots")

print("\n--- Treatment B: Angle arc focus (stride knee) ---")
build_treatment_pair(sample_pitch["clip_path"], draw_angle_arc, "B_angle_arc", joint_side="l")

print("\n--- Treatment C: Segment fill bands ---")
build_treatment_pair(sample_pitch["clip_path"], draw_segment_fill, "C_segment_fill", joint_side="l")

print("\nAll 6 GIFs built. Check:", OUTPUT_DIR)

In [ ]:
# ══════════════════════════════════════════════════════════
#  SYNCED DUAL-PANEL GIF — Style 1 (white/orange/navy)
#  Adapted from your prior dark-theme version (cell 10 in the
#  old notebook). Same technique: skeleton panel + live-updating
#  graph panel, stacked, for two pitches side by side, synced
#  frame by frame, with release frame marked.
#
#  Changes from the old version:
#    - White background instead of dark theme, matches Style 1
#    - Title framing is neutral (no "broke down before his X did"
#      causal injury claim)
#    - Uses real corrected pitch data (fastballs)
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)
NAVY_BGR   = (84, 14, 0)
GRAY_BGR   = (184, 179, 173)
WHITE_BGR  = (255, 255, 255)
ORANGE_MPL = "#F76900"
NAVY_MPL   = "#000E54"
GRAY_MPL   = "#ADB3B8"

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

def angle_between(a, b, c):
    ba = a - b; bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def draw_skeleton(frame_bgr, kp, box=None, conf_thresh=0.3):
    frame = frame_bgr.copy()
    if box is not None:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(frame, pa, pb, ORANGE_BGR, 3, cv2.LINE_AA)
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(frame, pt, 6, NAVY_BGR, -1, cv2.LINE_AA)
    return frame

def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")


def render_graph_panel(knee_vals, current_idx, color_mpl, panel_w, graph_h, release_idx):
    """Live-updating line graph: gray for the full trace, colored for
    everything up through the current frame, dot at the current point."""
    n = len(knee_vals)
    y_min, y_max = np.nanmin(knee_vals) - 3, np.nanmax(knee_vals) + 3

    fig, ax = plt.subplots(figsize=(panel_w / 100, graph_h / 100), facecolor="white")
    ax.set_facecolor("white")

    ax.plot(range(n), knee_vals, color=GRAY_MPL, linewidth=1.2, zorder=1)

    past_x = list(range(current_idx + 1))
    past_y = knee_vals[:current_idx + 1]
    ax.plot(past_x, past_y, color=color_mpl, linewidth=2.5, zorder=2)
    if len(past_y) > 0 and not np.isnan(past_y[-1]):
        ax.scatter(past_x[-1], past_y[-1], color=color_mpl, s=55, zorder=3)

    ax.axvline(release_idx, color=NAVY_MPL, linewidth=1, linestyle="--", alpha=0.5)

    ax.set_xlim(0, n - 1)
    ax.set_ylim(y_min, y_max)
    ax.set_ylabel("Knee °", color="#888888", fontsize=9)
    ax.tick_params(colors="#aaaaaa", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("#dddddd")
    ax.tick_params(axis="x", labelbottom=False)

    fig.tight_layout(pad=0.4)
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(
        fig.canvas.get_width_height()[::-1] + (4,))
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    plt.close(fig)
    return cv2.resize(img, (panel_w, graph_h))


def build_synced_dual_panel(clip_path_left, clip_path_right, left_label, right_label,
                              left_velo, right_velo, start=165, end=195,
                              out_name="synced_dual_panel.gif", fps=15, release_frame=179):
    frames_l = read_frame_range(clip_path_left, start, end)
    frames_r = read_frame_range(clip_path_right, start, end)
    tracked_l = track_pitcher_through_clip(frames_l)
    tracked_r = track_pitcher_through_clip(frames_r)

    def knee_series(tracked):
        vals = []
        for box, kp in tracked:
            if kp is None:
                vals.append(np.nan)
                continue
            hip, knee, ankle = kp[KP["l_hip"], :2], kp[KP["l_knee"], :2], kp[KP["l_ankle"], :2]
            vals.append(angle_between(hip, knee, ankle))
        return np.array(vals)

    knee_l = knee_series(tracked_l)
    knee_r = knee_series(tracked_r)
    release_idx = release_frame - start

    left_avg = np.nanmean(knee_l)
    right_avg = np.nanmean(knee_r)

    PANEL_W, SKEL_H, GRAPH_H = 640, 400, 180
    TITLE_H, LABEL_H = 55, 34
    TOTAL_W = PANEL_W * 2

    title_bar = np.full((TITLE_H, TOTAL_W, 3), 255, dtype=np.uint8)
    cv2.putText(title_bar, "STRIDE KNEE FLEXION: EARLY VS. LATE IN THE OUTING",
                (20, 35), cv2.FONT_HERSHEY_DUPLEX, 0.85, (60, 60, 60), 1, cv2.LINE_AA)

    label_bar = np.full((LABEL_H, TOTAL_W, 3), 255, dtype=np.uint8)
    cv2.putText(label_bar, f"{left_label}  |  {left_velo:.1f} mph  |  avg knee: {left_avg:.1f} deg",
                (20, 24), cv2.FONT_HERSHEY_DUPLEX, 0.55, NAVY_BGR, 1, cv2.LINE_AA)
    cv2.putText(label_bar, f"{right_label}  |  {right_velo:.1f} mph  |  avg knee: {right_avg:.1f} deg",
                (PANEL_W + 20, 24), cv2.FONT_HERSHEY_DUPLEX, 0.55, ORANGE_BGR, 1, cv2.LINE_AA)

    n = min(len(frames_l), len(frames_r))
    output_frames = []
    for i in range(n):
        left_graph = render_graph_panel(knee_l, i, NAVY_MPL, PANEL_W, GRAPH_H, release_idx)
        right_graph = render_graph_panel(knee_r, i, ORANGE_MPL, PANEL_W, GRAPH_H, release_idx)
        left_skel = cv2.resize(draw_skeleton(frames_l[i], tracked_l[i][1], tracked_l[i][0]), (PANEL_W, SKEL_H))
        right_skel = cv2.resize(draw_skeleton(frames_r[i], tracked_r[i][1], tracked_r[i][0]), (PANEL_W, SKEL_H))

        graph_row = np.hstack([left_graph, right_graph])
        skel_row = np.hstack([left_skel, right_skel])
        cv2.line(skel_row, (PANEL_W, 0), (PANEL_W, SKEL_H), (220, 220, 220), 2)
        cv2.line(graph_row, (PANEL_W, 0), (PANEL_W, GRAPH_H), (220, 220, 220), 2)

        full = np.vstack([title_bar, label_bar, graph_row, skel_row])
        output_frames.append(full)

    frames_to_gif(output_frames, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  RUN — early inning-1 fastball vs late inning-6 fastball
# ══════════════════════════════════════════════════
early_pitch = df[(df["inning"] == 1) & (df["pitch_type_desc"] == "Four-Seam Fastball")].iloc[0]
late_pitch  = df[(df["inning"] == 6) & (df["pitch_type_desc"] == "Four-Seam Fastball")].iloc[-1]

build_synced_dual_panel(
    early_pitch["clip_path"], late_pitch["clip_path"],
    left_label="INNING 1", right_label="INNING 6",
    left_velo=early_pitch["start_speed"], right_velo=late_pitch["start_speed"],
)

In [ ]:
# ══════════════════════════════════════════════════════════
#  SKELETON-ONLY ON BLANK CANVAS — Style 1 palette
#  No video underneath, no emphasized joint (all joints equal
#  size/weight), played back at 0.5x speed, looped.
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)   # #F76900
NAVY_BGR   = (84, 14, 0)     # #000E54
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

def draw_skeleton_blank(canvas, kp, conf_thresh=0.3, line_thickness=4, joint_radius=8):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, ORANGE_BGR, line_thickness, cv2.LINE_AA)
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, joint_radius, WHITE_BGR, -1, cv2.LINE_AA)
                cv2.circle(canvas, pt, joint_radius, NAVY_BGR, -1, cv2.LINE_AA)
    return canvas

def frames_to_gif_slow(frame_list_bgr, out_path, base_fps=15, speed=0.5, loop=0):
    """Save frames as a looping GIF, played back at `speed`x the base fps
    (0.5 = half speed) by lengthening each frame's display duration."""
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    effective_fps = base_fps * speed
    duration_ms = int(1000 / effective_fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames, "
          f"{duration_ms}ms/frame = {speed}x speed, looping)")


def build_skeleton_only_slow(clip_path, start=165, end=195,
                               out_name="skeleton_only_0.5x.gif", base_fps=15, speed=0.5):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    h, w = frames[0].shape[:2]

    canvases = []
    for box, kp in tracked:
        blank = np.full((h, w, 3), 255, dtype=np.uint8)  # white canvas
        canvases.append(draw_skeleton_blank(blank, kp))

    frames_to_gif_slow(canvases, os.path.join(OUTPUT_DIR, out_name),
                        base_fps=base_fps, speed=speed)


# ══════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════
sample_pitch = df[df["pitch_type_desc"] == "Four-Seam Fastball"].iloc[0]
print(f"Using clip: {sample_pitch['clip_path']}")

build_skeleton_only_slow(sample_pitch["clip_path"])

In [ ]:
# ══════════════════════════════════════════════════════════
#  FIND A CLIP WITH ZERO DROPPED JOINTS ACROSS THE WHOLE
#  165-195 WINDOW (every one of the 13 tracked joints stays
#  above confidence threshold on every single frame).
#
#  Stops at the first clean one it finds rather than scanning
#  everything (each clip takes ~25s to track, so this could
#  otherwise take a long time).
# ══════════════════════════════════════════════════════════

import numpy as np

DRAWN_JOINTS = {"l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist",
                 "l_hip", "r_hip", "l_knee", "r_knee", "l_ankle", "r_ankle"}  # excludes nose — never drawn

def check_clip_completeness(clip_path, start=165, end=195, conf_thresh=0.3):
    """
    Returns (is_fully_clean, worst_joint, min_confidence, n_frames_with_any_drop)
    Only checks joints actually used in the skeleton drawing (excludes nose).
    """
    frames = read_frame_range(clip_path, start, end)
    if len(frames) < (end - start + 1) * 0.9:
        return False, "clip_too_short", 0.0, -1

    tracked = track_pitcher_through_clip(frames)

    worst_conf = 1.0
    worst_joint = None
    frames_with_drop = 0

    for box, kp in tracked:
        if kp is None:
            frames_with_drop += 1
            worst_conf = 0.0
            worst_joint = "PITCHER NOT DETECTED"
            continue
        frame_min = 1.0
        for name, idx in KP.items():
            if name not in DRAWN_JOINTS:
                continue
            c = kp[idx, 2]
            if c < frame_min:
                frame_min = c
            if c < worst_conf:
                worst_conf = c
                worst_joint = name
        if frame_min < conf_thresh:
            frames_with_drop += 1

    is_clean = (frames_with_drop == 0)
    return is_clean, worst_joint, worst_conf, frames_with_drop


# ══════════════════════════════════════════════════
#  Scan candidates — fastballs first, in game order, stop at
#  the first fully clean one. Caps at MAX_CHECK to avoid an
#  unbounded search.
# ══════════════════════════════════════════════════
MAX_CHECK = 20
CONF_THRESH = 0.3

candidates = df[df["pitch_type_desc"] == "Four-Seam Fastball"].reset_index(drop=True)
print(f"Scanning up to {MAX_CHECK} fastballs for a fully clean clip "
      f"(every joint > {CONF_THRESH} confidence, every frame, 165-195)...\n")

clean_clip = None
for i, row in candidates.head(MAX_CHECK).iterrows():
    print(f"[{i+1}/{min(MAX_CHECK, len(candidates))}] checking inning {row['inning']}, "
          f"pitch {row['pitch_number']}...", end=" ")
    is_clean, worst_joint, worst_conf, n_drops = check_clip_completeness(
        row["clip_path"], conf_thresh=CONF_THRESH)

    if is_clean:
        print(f"CLEAN — every joint stayed above {CONF_THRESH} the whole time!")
        clean_clip = row
        break
    else:
        print(f"drop on {n_drops} frame(s), worst joint '{worst_joint}' (conf={worst_conf:.2f})")

if clean_clip is not None:
    print(f"\n>>> Found clean clip: inning {clean_clip['inning']}, "
          f"pitch {clean_clip['pitch_number']}")
    print(f">>> clip_path: {clean_clip['clip_path']}")
    print(f">>> play_id: {clean_clip['play_id']}")
else:
    print(f"\nNo fully clean clip found in the first {MAX_CHECK} fastballs.")
    print("Options: raise MAX_CHECK, lower CONF_THRESH slightly, or check all pitch types.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  SKELETON-ONLY ON BLANK CANVAS — Style 1 palette
#  No video underneath, no emphasized joint (all joints equal
#  size/weight), played back at 0.5x speed, looped.
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)   # #F76900
NAVY_BGR   = (84, 14, 0)     # #000E54
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

def draw_skeleton_blank(canvas, kp, conf_thresh=0.2, line_thickness=4, joint_radius=8):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, ORANGE_BGR, line_thickness, cv2.LINE_AA)
        for name, idx in KP.items():
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, joint_radius, WHITE_BGR, -1, cv2.LINE_AA)
                cv2.circle(canvas, pt, joint_radius, NAVY_BGR, -1, cv2.LINE_AA)
    return canvas

def frames_to_gif_slow(frame_list_bgr, out_path, base_fps=15, speed=0.5, loop=0):
    """Save frames as a looping GIF, played back at `speed`x the base fps
    (0.5 = half speed) by lengthening each frame's display duration."""
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    effective_fps = base_fps * speed
    duration_ms = int(1000 / effective_fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames, "
          f"{duration_ms}ms/frame = {speed}x speed, looping)")


def build_skeleton_only_slow(clip_path, start=165, end=195,
                               out_name="skeleton_only_0.5x.gif", base_fps=15, speed=0.5):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    h, w = frames[0].shape[:2]

    canvases = []
    for box, kp in tracked:
        blank = np.full((h, w, 3), 255, dtype=np.uint8)  # white canvas
        canvases.append(draw_skeleton_blank(blank, kp))

    frames_to_gif_slow(canvases, os.path.join(OUTPUT_DIR, out_name),
                        base_fps=base_fps, speed=speed)


# ══════════════════════════════════════════════════
#  RUN — using the verified fully-clean clip found by the scanner
# ══════════════════════════════════════════════════
clean_clip_path = r"C:\Users\noahl\OneDrive\fatigue_cv\clips\a6d0cf9f-271a-341d-9be0-85cbe90235a4.mp4"
print(f"Using clean clip: {clean_clip_path}")

build_skeleton_only_slow(clean_clip_path)

In [ ]:
# ══════════════════════════════════════════════════════════
#  SHOULD-BE-VISIBLE DIAGNOSTIC
#  For each frame and each joint, checks:
#    1. In-bounds       — is the point inside the video frame?
#    2. In-bbox         — is it within/near the tracked pitcher box?
#    3. Temporal jump   — did it move an implausible distance vs
#                         the previous frame?
#    4. Bone length     — for connected joints, is the pixel
#                         distance close to this clip's own median
#                         bone length for that connection?
#
#  A joint that passes all 4 checks (clearly SHOULD be visible)
#  but still has low model confidence is the real problem case —
#  distinct from a joint that's genuinely occluded/off-frame,
#  where low confidence is expected and correct.
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]


def compute_median_bone_lengths(tracked, conf_thresh=0.3):
    """Median pixel length of each skeleton edge across all confident frames."""
    lengths = {edge: [] for edge in SKELETON_EDGES}
    for box, kp in tracked:
        if kp is None:
            continue
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                d = np.linalg.norm(kp[ia, :2] - kp[ib, :2])
                lengths[(a, b)].append(d)
    return {edge: (np.median(vals) if vals else np.nan) for edge, vals in lengths.items()}


def should_be_visible_report(clip_path, start=165, end=195, conf_thresh=0.3,
                               bbox_pad=30, max_jump_multiplier=3.0,
                               bone_tolerance=0.4):
    """
    Returns a per-frame, per-joint DataFrame with columns:
      frame, joint, model_conf, in_bounds, in_bbox, temporal_ok,
      bone_ok, should_be_visible, flagged_problem

    flagged_problem = True means: heuristics say this joint SHOULD
    be visible, but the model's confidence was low anyway — a real
    tracking gap, not expected occlusion.
    """
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    h, w = frames[0].shape[:2]
    n = len(frames)

    median_bones = compute_median_bone_lengths(tracked, conf_thresh)

    joint_displacements = {name: [] for name in KP.keys()}
    prev_kp = None
    for box, kp in tracked:
        if kp is not None and prev_kp is not None:
            for name, idx in KP.items():
                if kp[idx, 2] > conf_thresh and prev_kp[idx, 2] > conf_thresh:
                    d = np.linalg.norm(kp[idx, :2] - prev_kp[idx, :2])
                    joint_displacements[name].append(d)
        prev_kp = kp
    typical_disp = {name: (np.median(vals) if vals else 15.0)
                     for name, vals in joint_displacements.items()}

    rows = []
    prev_kp = None
    for frame_idx, (box, kp) in enumerate(tracked):
        abs_frame = start + frame_idx

        if kp is None:
            for name in KP.keys():
                rows.append({
                    "frame": abs_frame, "joint": name, "model_conf": 0.0,
                    "in_bounds": False, "in_bbox": False, "temporal_ok": False,
                    "bone_ok": np.nan, "should_be_visible": False,
                    "flagged_problem": False,
                })
            prev_kp = None
            continue

        for name, idx in KP.items():
            x, y, conf = kp[idx]

            in_bounds = (0 <= x <= w) and (0 <= y <= h)

            in_bbox = True
            if box is not None:
                x1, y1, x2, y2 = box
                in_bbox = (x1 - bbox_pad <= x <= x2 + bbox_pad and
                           y1 - bbox_pad <= y <= y2 + bbox_pad)

            temporal_ok = True
            if prev_kp is not None and prev_kp[idx, 2] > conf_thresh:
                jump = np.linalg.norm(kp[idx, :2] - prev_kp[idx, :2])
                max_allowed = typical_disp[name] * max_jump_multiplier + 10
                temporal_ok = jump <= max_allowed

            bone_ok_vals = []
            for a, b in SKELETON_EDGES:
                if name not in (a, b):
                    continue
                other = b if name == a else a
                other_idx = KP[other]
                if kp[other_idx, 2] > conf_thresh:
                    d = np.linalg.norm(kp[idx, :2] - kp[other_idx, :2])
                    median_len = median_bones.get((a, b), np.nan)
                    if not np.isnan(median_len) and median_len > 0:
                        ratio = abs(d - median_len) / median_len
                        bone_ok_vals.append(ratio <= bone_tolerance)
            bone_ok = all(bone_ok_vals) if bone_ok_vals else np.nan

            should_be_visible = in_bounds and in_bbox and temporal_ok and (bone_ok != False)
            flagged_problem = should_be_visible and (conf < conf_thresh)

            rows.append({
                "frame": abs_frame, "joint": name, "model_conf": round(float(conf), 3),
                "in_bounds": in_bounds, "in_bbox": in_bbox, "temporal_ok": temporal_ok,
                "bone_ok": bone_ok, "should_be_visible": should_be_visible,
                "flagged_problem": flagged_problem,
            })

        prev_kp = kp

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════
#  RUN on the clean clip we found earlier
# ══════════════════════════════════════════════════
clean_clip_path = r"C:\Users\noahl\OneDrive\fatigue_cv\clips\a6d0cf9f-271a-341d-9be0-85cbe90235a4.mp4"

report = should_be_visible_report(clean_clip_path)

print(f"Total joint-frame checks: {len(report)}")
print(f"Flagged as real problems (should be visible, but low confidence): "
      f"{report['flagged_problem'].sum()}")

problems = report[report["flagged_problem"]]
if len(problems) > 0:
    print("\n--- Flagged problem frames ---")
    print(problems[["frame", "joint", "model_conf"]].to_string(index=False))
else:
    print("\nNo real problems flagged — every joint that heuristically SHOULD")
    print("be visible also has adequate model confidence throughout.")

print("\n--- Per-joint summary across the whole clip ---")
summary = report.groupby("joint").agg(
    mean_conf=("model_conf", "mean"),
    min_conf=("model_conf", "min"),
    pct_should_be_visible=("should_be_visible", "mean"),
    n_flagged_problems=("flagged_problem", "sum"),
).round(3)
print(summary.to_string())

In [ ]:
# ══════════════════════════════════════════════════════════
#  TEST 5 RANDOM PITCHES — displays inline, doesn't save to disk.
#  Also drops the nose point entirely (never connected to any
#  skeleton line anyway, and its confidence flickers around the
#  threshold all clip long — that's the "head disappearing" bug).
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image as PILImage
from IPython.display import Image as IPyImage, display
from io import BytesIO
import random

ORANGE_BGR = (0, 105, 247)   # #F76900
NAVY_BGR   = (84, 14, 0)     # #000E54
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

# Only the 12 joints actually used in the skeleton — nose excluded,
# since it's never connected to anything and just flickers.
DRAWN_JOINTS = {j for edge in SKELETON_EDGES for j in edge}


def draw_skeleton_blank(canvas, kp, conf_thresh=0.3, line_thickness=4, joint_radius=8):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, ORANGE_BGR, line_thickness, cv2.LINE_AA)
        for name in DRAWN_JOINTS:            # <-- only drawn joints, nose excluded
            idx = KP[name]
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, joint_radius, WHITE_BGR, -1, cv2.LINE_AA)
                cv2.circle(canvas, pt, joint_radius, NAVY_BGR, -1, cv2.LINE_AA)
    return canvas


def build_gif_bytes(clip_path, start=165, end=195, base_fps=15, speed=0.5):
    """Builds the GIF entirely in memory, returns raw bytes (nothing saved to disk)."""
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    h, w = frames[0].shape[:2]

    canvases = []
    for box, kp in tracked:
        blank = np.full((h, w, 3), 255, dtype=np.uint8)
        canvases.append(draw_skeleton_blank(blank, kp))

    pil_frames = [PILImage.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in canvases]
    effective_fps = base_fps * speed
    duration_ms = int(1000 / effective_fps)

    buf = BytesIO()
    pil_frames[0].save(
        buf, format="GIF", save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=0, optimize=False
    )
    buf.seek(0)
    return buf.getvalue()


# ══════════════════════════════════════════════════
#  RUN — 5 random pitches, displayed inline, not saved
# ══════════════════════════════════════════════════
random.seed()  # true randomness each run; set a number here if you want repeatability
sample_rows = df.sample(5).reset_index(drop=True)

for i, row in sample_rows.iterrows():
    print(f"\n=== Pitch {i+1}/5 — Inning {row['inning']}, pitch #{row['pitch_number']}, "
          f"{row['pitch_type_desc']}, play_id {row['play_id']} ===")
    try:
        gif_bytes = build_gif_bytes(row["clip_path"])
        display(IPyImage(data=gif_bytes, format="gif"))
    except Exception as e:
        print(f"  Failed on this pitch: {e}")

In [ ]:
# ══════════════════════════════════════════════════════════
#  MITOSIS-TRANSITION GIF — v2
#  Same concept as before: combined skeleton-over-video plays,
#  then splits into video-only (left) and skeleton-only (right),
#  then plays side by side in sync, then loops.
#
#  Fixes from before:
#    - Uses the verified clean pitch (inning 1, pitch 1) — every
#      drawn joint stays confident the whole clip
#    - Nose/head point is never drawn at all (it's not part of
#      the skeleton lines anyway, and was flickering)
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)
NAVY_BGR   = (84, 14, 0)
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]

# Only joints that are actually part of the skeleton lines — no head/nose
DRAWN_JOINTS = {j for edge in SKELETON_EDGES for j in edge}


def draw_skeleton_on(canvas, kp, box=None, line_color=ORANGE_BGR, pt_color=NAVY_BGR,
                      line_thickness=3, pt_radius=6, conf_thresh=0.3):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, line_color, line_thickness, cv2.LINE_AA)
        for name in DRAWN_JOINTS:          # <-- nose excluded, only real skeleton joints
            idx = KP[name]
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, pt_radius, pt_color, -1, cv2.LINE_AA)
    return canvas


def compute_pitcher_crop_box(tracked, frame_w, frame_h, pad_frac=0.35):
    """
    Union of all valid pitcher bounding boxes across the tracked clip,
    padded out, clamped to frame bounds. This becomes a fixed 'zoom'
    region so both panels show a well-framed, centered view of the
    pitcher instead of the whole wide broadcast frame.
    """
    xs1, ys1, xs2, ys2 = [], [], [], []
    for box, kp in tracked:
        if box is None:
            continue
        x1, y1, x2, y2 = box
        xs1.append(x1); ys1.append(y1); xs2.append(x2); ys2.append(y2)

    x1, y1, x2, y2 = min(xs1), min(ys1), max(xs2), max(ys2)
    w, h = x2 - x1, y2 - y1
    pad_x, pad_y = w * pad_frac, h * pad_frac

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(frame_w, x2 + pad_x)
    y2 = min(frame_h, y2 + pad_y)
    return int(x1), int(y1), int(x2), int(y2)


def resize_with_letterbox(img, target_w, target_h, bg_color=255):
    """
    Scales img to fit within (target_w, target_h) preserving aspect
    ratio, then pads with bg_color to exactly fill the target size —
    like a YouTube TV multiview panel, never stretched.
    """
    h, w = img.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h))

    canvas = np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas


def shift_keypoints(kp, box, offset_x, offset_y):
    """Returns a copy of kp/box shifted by the crop offset, for drawing
    onto a cropped canvas instead of the full original frame."""
    if kp is None:
        shifted_kp = None
    else:
        shifted_kp = kp.copy()
        shifted_kp[:, 0] -= offset_x
        shifted_kp[:, 1] -= offset_y
    if box is None:
        shifted_box = None
    else:
        x1, y1, x2, y2 = box
        shifted_box = (x1 - offset_x, y1 - offset_y, x2 - offset_x, y2 - offset_y)
    return shifted_kp, shifted_box
def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")


def build_mitosis_gif(clip_path, start=165, end=195, out_name="mitosis_v4.gif",
                       fps=15, split_frames=20):
    frames = read_frame_range(clip_path, start, end)
    tracked = track_pitcher_through_clip(frames)
    n = len(frames)
    h, w = frames[0].shape[:2]

    cx1, cy1, cx2, cy2 = compute_pitcher_crop_box(tracked, w, h)
    half_w = w // 2

    all_output_frames = []

    # LOOP 1: combined skeleton-over-video, FULL wide broadcast frame
    for i in range(n):
        combined = draw_skeleton_on(frames[i], tracked[i][1], tracked[i][0])
        all_output_frames.append(combined)

    # ── TRANSITION ──
    # Freezes on the SAME last frame Loop 1 just ended on (no pose jump),
    # then continuously: (a) zooms from full-frame to the pitcher crop,
    # (b) splits the panel in two, (c) fades the skeleton off the video
    # panel as it appears on the new skeleton-only panel. All three
    # happen together over the same eased timeline, so it reads as one
    # smooth camera move instead of a hard cut.
    freeze_idx = n - 1
    freeze_frame = frames[freeze_idx]
    freeze_box, freeze_kp = tracked[freeze_idx]

    for step in range(split_frames):
        t = step / (split_frames - 1)
        ease_t = t * t * (3 - 2 * t)

        # Interpolate the crop window from full-frame -> target pitcher crop
        icx1 = int(0   * (1 - ease_t) + cx1 * ease_t)
        icy1 = int(0   * (1 - ease_t) + cy1 * ease_t)
        icx2 = int(w   * (1 - ease_t) + cx2 * ease_t)
        icy2 = int(h   * (1 - ease_t) + cy2 * ease_t)
        icx2 = max(icx2, icx1 + 2)
        icy2 = max(icy2, icy1 + 2)

        kp_shifted, box_shifted = shift_keypoints(freeze_kp, freeze_box, icx1, icy1)

        # Left panel: video cropped to the CURRENT zoom level, with the
        # skeleton overlay fading out (alpha 1 -> 0) as it "moves" right
        vid_crop = freeze_frame[icy1:icy2, icx1:icx2]
        vid_with_skel = draw_skeleton_on(vid_crop, kp_shifted, box_shifted)
        skel_alpha = 1.0 - ease_t
        left_content = cv2.addWeighted(vid_with_skel, skel_alpha, vid_crop, 1 - skel_alpha, 0)

        # Right panel: skeleton-only on white, SAME zoom level as left,
        # fading in as it separates
        right_canvas = np.full((icy2 - icy1, icx2 - icx1, 3), 255, dtype=np.uint8)
        right_content = draw_skeleton_on(right_canvas, kp_shifted, box=None)

        target_w_left = int(w * (1 - ease_t) + half_w * ease_t)
        target_w_right = int(half_w * ease_t)

        canvas = np.full((h, w, 3), 255, dtype=np.uint8)
        left_panel = resize_with_letterbox(left_content, target_w_left, h)
        canvas[:, 0:target_w_left] = left_panel

        if target_w_right > 0:
            right_panel = resize_with_letterbox(right_content, target_w_right, h)
            canvas[:, w - target_w_right:w] = right_panel

        all_output_frames.append(canvas)

    # Precompute cropped, letterboxed panels for loop 2 (uses the fixed
    # final crop box, not the interpolated one)
    crop_w, crop_h = cx2 - cx1, cy2 - cy1
    cropped_video_frames = []
    cropped_skeleton_frames = []
    for i in range(n):
        kp_shifted, box_shifted = shift_keypoints(tracked[i][1], tracked[i][0], cx1, cy1)
        cropped_video_frames.append(frames[i][cy1:cy2, cx1:cx2])
        skel_canvas = np.full((crop_h, crop_w, 3), 255, dtype=np.uint8)
        cropped_skeleton_frames.append(draw_skeleton_on(skel_canvas, kp_shifted, box=None))

    # LOOP 2: side by side, same delivery replayed in sync from the start
    for i in range(n):
        vid_panel = resize_with_letterbox(cropped_video_frames[i], half_w, h)
        skel_panel = resize_with_letterbox(cropped_skeleton_frames[i], w - half_w, h)

        divider = np.full((h, 3, 3), 200, dtype=np.uint8)
        side_by_side = np.hstack([vid_panel[:, :vid_panel.shape[1]-2], divider,
                                    skel_panel[:, 2:]])
        side_by_side = cv2.resize(side_by_side, (w, h))
        all_output_frames.append(side_by_side)

    frames_to_gif(all_output_frames, os.path.join(OUTPUT_DIR, out_name), fps=fps)
    print(f"Total frames in full loop: {len(all_output_frames)} "
          f"({n} combined + {split_frames} transition + {n} side-by-side)")
    print(f"Crop region used for split panels: ({cx1},{cy1}) to ({cx2},{cy2}), "
          f"size {crop_w}x{crop_h}")


# ══════════════════════════════════════════════════
#  RUN — using the verified clean clip
# ══════════════════════════════════════════════════
clean_clip_path = r"C:\Users\noahl\OneDrive\fatigue_cv\clips\a6d0cf9f-271a-341d-9be0-85cbe90235a4.mp4"
print(f"Building mitosis GIF from verified clean clip: {clean_clip_path}")

build_mitosis_gif(clean_clip_path)

In [ ]:
# ══════════════════════════════════════════════════════════
#  THREE PRIORITY CHARTS — Style 1 (white/orange/navy),
#  single y-axis per panel, bigger text, real corrected data.
#  Standalone — reads fresh from your CSVs.
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

ORANGE = "#F76900"
NAVY   = "#000E54"
GRAY   = "#3F403F"
LTGRAY = "#ADB3B8"
BAND   = "#F7F7F8"

plt.rcParams["font.family"] = "Bahnschrift"

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\charts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

def iqr_filter_all(df_in, col, k=2.5):
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k*iqr, q3 + k*iqr
        parts.append(group[(group[col] >= lo) & (group[col] <= hi)])
    return pd.concat(parts).sort_values("game_pitch_number")


# ══════════════════════════════════════════════════════════
#  CHART A — COLD OPEN: fastball velocity flat post-settling
#  vs stride knee mechanics still moving. Two single-axis panels.
# ══════════════════════════════════════════════════════════
ff_unfiltered = df_ok[df_ok["pitch_type"] == "Four-Seam Fastball"].dropna(
    subset=["start_speed", "stride_knee_flexion_mean"]).copy()
ff = iqr_filter_all(ff_unfiltered, "stride_knee_flexion_mean")
n_removed = len(ff_unfiltered) - len(ff)
if n_removed > 0:
    print(f"Chart A: removed {n_removed} outlier pitch(es) via IQR filter, "
          f"{len(ff)} fastballs remain")
ff = ff.sort_values("game_pitch_number").reset_index(drop=True)
ff["ff_seq"] = np.arange(1, len(ff) + 1)   # renumbered, no gap

settle_end = ff[ff["inning"] == 1]["ff_seq"].max()

vel_slope, vel_int, vel_r, vel_p, _ = stats.linregress(
    ff[ff["inning"] >= 2]["ff_seq"], ff[ff["inning"] >= 2]["start_speed"])
knee_slope, knee_int, knee_r, knee_p, _ = stats.linregress(
    ff["ff_seq"], ff["stride_knee_flexion_mean"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor="white")
fig.text(0.06, 0.96, "EVERY NUMBER ON THE BROADCAST SAID HE WAS FINE",
          fontsize=22, color=GRAY, family="Liberation Sans")
fig.text(0.06, 0.905, "FASTBALLS ONLY  ·  MISIOROWSKI  ·  05.01.2026",
          fontsize=13, color=LTGRAY, family="Liberation Sans")

ax1, ax2 = axes

ax1.axvspan(0.5, settle_end + 0.5, color=BAND, zorder=0)
ax1.text(settle_end / 2 + 0.5, ff["start_speed"].min() - 1.2, "SETTLING",
          fontsize=10, color=LTGRAY, ha="center", family="Liberation Sans")
ax1.scatter(ff["ff_seq"], ff["start_speed"], color=LTGRAY, s=45, zorder=2)
x_line = np.array([settle_end + 0.5, ff["ff_seq"].max()])
ax1.plot(x_line, vel_slope * x_line + vel_int, color=NAVY, linewidth=3, zorder=3)
ax1.set_title("Fastball Velocity (mph)", fontsize=17, color=GRAY, loc="left", pad=14)
ax1.set_xlabel("Fastball sequence", fontsize=13, color=GRAY)
ax1.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax1.spines[s].set_visible(False)
ax1.spines["left"].set_color(LTGRAY); ax1.spines["bottom"].set_color(LTGRAY)
ax1.text(0.97, 0.06, f"post-settling: p = {vel_p:.2f} (flat)", transform=ax1.transAxes,
          fontsize=12, color=NAVY, ha="right", family="Liberation Sans")

ax2.axvspan(0.5, settle_end + 0.5, color=BAND, zorder=0)
ax2.scatter(ff["ff_seq"], ff["stride_knee_flexion_mean"], color=LTGRAY, s=45, zorder=2)
x_line_full = np.array([1, ff["ff_seq"].max()])
ax2.plot(x_line_full, knee_slope * x_line_full + knee_int, color=ORANGE, linewidth=3, zorder=3)
ax2.set_title("Stride Knee Flexion (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax2.set_xlabel("Fastball sequence", fontsize=13, color=GRAY)
ax2.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax2.spines[s].set_visible(False)
ax2.spines["left"].set_color(LTGRAY); ax2.spines["bottom"].set_color(LTGRAY)
ax2.text(0.97, 0.06, f"full outing: p = {knee_p:.3f}", transform=ax2.transAxes,
          fontsize=12, color=ORANGE, ha="right", family="Liberation Sans")

plt.tight_layout(rect=[0, 0, 1, 0.87])
out_a = os.path.join(OUTPUT_DIR, "A_cold_open_velocity_vs_knee.png")
plt.savefig(out_a, dpi=200, facecolor="white")
plt.close()
print(f"Saved: {out_a}")


# ══════════════════════════════════════════════════════════
#  CHART B — THE RESULT: stride knee mean + variability,
#  full robust dataset (all pitch types, IQR-filtered)
# ══════════════════════════════════════════════════════════
mean_filt = iqr_filter_all(df_ok, "stride_knee_flexion_mean").dropna(subset=["stride_knee_flexion_mean"])
mean_filt = mean_filt.sort_values("game_pitch_number").reset_index(drop=True)
mean_filt["seq"] = np.arange(1, len(mean_filt) + 1)

std_filt  = iqr_filter_all(df_ok, "stride_knee_flexion_std").dropna(subset=["stride_knee_flexion_std"])
std_filt = std_filt.sort_values("game_pitch_number").reset_index(drop=True)
std_filt["seq"] = np.arange(1, len(std_filt) + 1)

m_slope, m_int, m_r, m_p, _ = stats.linregress(mean_filt["seq"], mean_filt["stride_knee_flexion_mean"])
s_slope, s_int, s_r, s_p, _ = stats.linregress(std_filt["seq"], std_filt["stride_knee_flexion_std"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor="white")
fig.text(0.06, 0.96, "STRIDE KNEE MECHANICS DEGRADED ACROSS THE OUTING",
          fontsize=22, color=GRAY, family="Liberation Sans")
fig.text(0.06, 0.905, f"n = {len(mean_filt)} PITCHES  ·  ALL PITCH TYPES, OUTLIER-FILTERED",
          fontsize=13, color=LTGRAY, family="Liberation Sans")

ax1, ax2 = axes

ax1.scatter(mean_filt["seq"], mean_filt["stride_knee_flexion_mean"],
            color=LTGRAY, s=40, zorder=2)
x_line = np.array([mean_filt["seq"].min(), mean_filt["seq"].max()])
ax1.plot(x_line, m_slope * x_line + m_int, color=ORANGE, linewidth=3, zorder=3)
ax1.set_title("Average Angle (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax1.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax1.set_ylabel("Stride Knee Flexion (°)", fontsize=13, color=GRAY)
ax1.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax1.spines[s].set_visible(False)
ax1.spines["left"].set_color(LTGRAY); ax1.spines["bottom"].set_color(LTGRAY)
ax1.text(0.05, 0.06, f"p = {m_p:.4f}", transform=ax1.transAxes,
          fontsize=13, color=ORANGE, family="Liberation Sans")

ax2.scatter(std_filt["seq"], std_filt["stride_knee_flexion_std"],
            color=LTGRAY, s=40, zorder=2)
x_line2 = np.array([std_filt["seq"].min(), std_filt["seq"].max()])
ax2.plot(x_line2, s_slope * x_line2 + s_int, color=ORANGE, linewidth=3, zorder=3)
ax2.set_title("Instability at Release (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax2.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax2.set_ylabel("Frame-to-Frame Std Dev (°)", fontsize=13, color=GRAY)
ax2.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax2.spines[s].set_visible(False)
ax2.spines["left"].set_color(LTGRAY); ax2.spines["bottom"].set_color(LTGRAY)
ax2.text(0.05, 0.06, f"p = {s_p:.5f}", transform=ax2.transAxes,
          fontsize=13, color=ORANGE, family="Liberation Sans")

plt.tight_layout(rect=[0, 0, 1, 0.87])
out_b = os.path.join(OUTPUT_DIR, "B_result_mean_and_variability.png")
plt.savefig(out_b, dpi=200, facecolor="white")
plt.close()
print(f"Saved: {out_b}")


# ══════════════════════════════════════════════════════════
#  CHART C — THE MECHANISM: stride knee (significant) vs
#  throwing-side knee (flat), same full dataset
# ══════════════════════════════════════════════════════════
stride_filt = iqr_filter_all(df_ok, "stride_knee_flexion_mean").dropna(subset=["stride_knee_flexion_mean"])
stride_filt = stride_filt.sort_values("game_pitch_number").reset_index(drop=True)
stride_filt["seq"] = np.arange(1, len(stride_filt) + 1)

throw_filt  = iqr_filter_all(df_ok, "throwing_knee_flexion_mean").dropna(subset=["throwing_knee_flexion_mean"])
throw_filt = throw_filt.sort_values("game_pitch_number").reset_index(drop=True)
throw_filt["seq"] = np.arange(1, len(throw_filt) + 1)

st_slope, st_int, st_r, st_p, _ = stats.linregress(stride_filt["seq"], stride_filt["stride_knee_flexion_mean"])
th_slope, th_int, th_r, th_p, _ = stats.linregress(throw_filt["seq"], throw_filt["throwing_knee_flexion_mean"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor="white")
fig.text(0.06, 0.96, "THE STRIDE KNEE, NOT THE THROWING SIDE",
          fontsize=22, color=GRAY, family="Liberation Sans")
fig.text(0.06, 0.905, "THE LEAD LEG BRACES THE KINETIC CHAIN — THAT'S WHY IT MOVES",
          fontsize=13, color=LTGRAY, family="Liberation Sans")

ax1, ax2 = axes

ax1.scatter(throw_filt["seq"], throw_filt["throwing_knee_flexion_mean"],
            color=LTGRAY, s=40, zorder=2)
x_line = np.array([throw_filt["seq"].min(), throw_filt["seq"].max()])
ax1.plot(x_line, th_slope * x_line + th_int, color=LTGRAY, linewidth=3, zorder=3)
ax1.set_title("Throwing-Side Knee (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax1.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax1.set_ylabel("Knee Flexion (°)", fontsize=13, color=GRAY)
ax1.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax1.spines[s].set_visible(False)
ax1.spines["left"].set_color(LTGRAY); ax1.spines["bottom"].set_color(LTGRAY)
ax1.text(0.05, 0.06, f"p = {th_p:.2f}  (not significant)", transform=ax1.transAxes,
          fontsize=13, color=LTGRAY, family="Liberation Sans")

ax2.scatter(stride_filt["seq"], stride_filt["stride_knee_flexion_mean"],
            color=LTGRAY, s=40, zorder=2)
x_line2 = np.array([stride_filt["seq"].min(), stride_filt["seq"].max()])
ax2.plot(x_line2, st_slope * x_line2 + st_int, color=ORANGE, linewidth=3, zorder=3)
ax2.set_title("Stride (Lead) Knee (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax2.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax2.set_ylabel("Knee Flexion (°)", fontsize=13, color=GRAY)
ax2.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax2.spines[s].set_visible(False)
ax2.spines["left"].set_color(LTGRAY); ax2.spines["bottom"].set_color(LTGRAY)
ax2.text(0.05, 0.06, f"p = {st_p:.4f}", transform=ax2.transAxes,
          fontsize=13, color=ORANGE, family="Liberation Sans")

plt.tight_layout(rect=[0, 0, 1, 0.87])
out_c = os.path.join(OUTPUT_DIR, "C_mechanism_stride_vs_throwing.png")
plt.savefig(out_c, dpi=200, facecolor="white")
plt.close()
print(f"Saved: {out_c}")

print("\nAll three charts built.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  THREE PRIORITY CHARTS — Style 1 (white/orange/navy),
#  single y-axis per panel, bigger text, real corrected data.
#  Standalone — reads fresh from your CSVs.
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

ORANGE = "#F76900"
NAVY   = "#000E54"
GRAY   = "#3F403F"
LTGRAY = "#ADB3B8"
BAND   = "#F7F7F8"

plt.rcParams["font.family"] = "Bahnschrift"

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\charts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DIAG_PATH = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_full_diagnostics.csv"
RAW_PATH  = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"

df_diag = pd.read_csv(DIAG_PATH)
df_raw  = pd.read_csv(RAW_PATH)
df_ok = df_diag[df_diag["status"] == "ok"].copy()

df_raw_order = df_raw[["play_id", "at_bat_index", "pitch_number"]].copy()
df_ok = df_ok.merge(df_raw_order, on="play_id", suffixes=("", "_raw"))
df_ok = df_ok.sort_values(["at_bat_index", "pitch_number"]).reset_index(drop=True)
df_ok["game_pitch_number"] = np.arange(1, len(df_ok) + 1)

def iqr_filter_all(df_in, col, k=2.5):
    parts = []
    for pt, group in df_in.groupby("pitch_type"):
        if group[col].notna().sum() < 3:
            parts.append(group)
            continue
        q1, q3 = group[col].quantile(0.25), group[col].quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - k*iqr, q3 + k*iqr
        parts.append(group[(group[col] >= lo) & (group[col] <= hi)])
    return pd.concat(parts).sort_values("game_pitch_number")


# ══════════════════════════════════════════════════════════
#  CHART A — COLD OPEN: fastball velocity flat post-settling
#  vs stride knee mechanics still moving. Two single-axis panels.
# ══════════════════════════════════════════════════════════
ff_unfiltered = df_ok[df_ok["pitch_type"] == "Four-Seam Fastball"].dropna(
    subset=["start_speed", "stride_knee_flexion_mean"]).copy()
ff = iqr_filter_all(ff_unfiltered, "stride_knee_flexion_mean")
n_removed = len(ff_unfiltered) - len(ff)
if n_removed > 0:
    print(f"Chart A: removed {n_removed} outlier pitch(es) via IQR filter, "
          f"{len(ff)} fastballs remain")
ff = ff.sort_values("game_pitch_number").reset_index(drop=True)
ff["ff_seq"] = np.arange(1, len(ff) + 1)   # renumbered, no gap

settle_end = ff[ff["inning"] == 1]["ff_seq"].max()

vel_slope, vel_int, vel_r, vel_p, _ = stats.linregress(
    ff[ff["inning"] >= 2]["ff_seq"], ff[ff["inning"] >= 2]["start_speed"])
knee_slope, knee_int, knee_r, knee_p, _ = stats.linregress(
    ff["ff_seq"], ff["stride_knee_flexion_mean"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor="white")
fig.text(0.06, 0.96, "Velocity Remained Consistent as Mechanical Signs Emerged",
          fontsize=22, color=GRAY, family="Bahnschrift")
fig.text(0.06, 0.905, "FASTBALLS ONLY  ·  JACOB MISIOROWSKI  ·  05.01.2026",
          fontsize=13, color=LTGRAY, family="Bahnschrift")

ax1, ax2 = axes

ax1.axvspan(0.5, settle_end + 0.5, color=BAND, zorder=0)
ax1.text(settle_end / 2 + 0.5, ff["start_speed"].min() - 0.65, "SETTLING",
          fontsize=10, color=LTGRAY, ha="center", family="Bahnschrift")
ax1.scatter(ff["ff_seq"], ff["start_speed"], color=LTGRAY, s=45, zorder=2)
x_line = np.array([settle_end + 0.5, ff["ff_seq"].max()])
ax1.plot(x_line, vel_slope * x_line + vel_int, color=NAVY, linewidth=3, zorder=3)
ax1.set_title("Fastball Velocity (mph)", fontsize=17, color=GRAY, loc="left", pad=14)
ax1.set_xlabel("Fastball sequence", fontsize=13, color=GRAY)
ax1.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax1.spines[s].set_visible(False)
ax1.spines["left"].set_color(LTGRAY); ax1.spines["bottom"].set_color(LTGRAY)
ax1.text(0.97, 0.06, f"post-settling: p = {vel_p:.2f} (flat)", transform=ax1.transAxes,
          fontsize=12, color=NAVY, ha="right", family="Bahnschrift")

ax2.axvspan(0.5, settle_end + 0.5, color=BAND, zorder=0)
ax2.scatter(ff["ff_seq"], ff["stride_knee_flexion_mean"], color=LTGRAY, s=45, zorder=2)
x_line_full = np.array([1, ff["ff_seq"].max()])
ax2.plot(x_line_full, knee_slope * x_line_full + knee_int, color=ORANGE, linewidth=3, zorder=3)
ax2.set_title("Stride Knee Flexion (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax2.set_xlabel("Fastball sequence", fontsize=13, color=GRAY)
ax2.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax2.spines[s].set_visible(False)
ax2.spines["left"].set_color(LTGRAY); ax2.spines["bottom"].set_color(LTGRAY)
ax2.text(0.97, 0.1, f"full outing: p = {knee_p:.3f}", transform=ax2.transAxes,
          fontsize=12, color=ORANGE, ha="right", family="Bahnschrift")

plt.tight_layout(rect=[0, 0, 1, 0.87])
out_a = os.path.join(OUTPUT_DIR, "A_cold_open_velocity_vs_knee.png")
plt.savefig(out_a, dpi=200, facecolor="white")
plt.close()
print(f"Saved: {out_a}")


# ══════════════════════════════════════════════════════════
#  CHART B — THE RESULT: stride knee mean + variability,
#  full robust dataset (all pitch types, IQR-filtered)
# ══════════════════════════════════════════════════════════
mean_filt = iqr_filter_all(df_ok, "stride_knee_flexion_mean").dropna(subset=["stride_knee_flexion_mean"])
mean_filt = mean_filt.sort_values("game_pitch_number").reset_index(drop=True)
mean_filt["seq"] = np.arange(1, len(mean_filt) + 1)

std_filt  = iqr_filter_all(df_ok, "stride_knee_flexion_std").dropna(subset=["stride_knee_flexion_std"])
std_filt = std_filt.sort_values("game_pitch_number").reset_index(drop=True)
std_filt["seq"] = np.arange(1, len(std_filt) + 1)

m_slope, m_int, m_r, m_p, _ = stats.linregress(mean_filt["seq"], mean_filt["stride_knee_flexion_mean"])
s_slope, s_int, s_r, s_p, _ = stats.linregress(std_filt["seq"], std_filt["stride_knee_flexion_std"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor="white")
fig.text(0.06, 0.96, "Stride Knee Mechanics Degraded Across the Outing",
          fontsize=22, color=GRAY, family="Bahnschrift")
fig.text(0.06, 0.905, f"n = {len(mean_filt)} Pitches  ·  All Pitch Types",
          fontsize=13, color=LTGRAY, family="Bahnschrift")

ax1, ax2 = axes

ax1.scatter(mean_filt["seq"], mean_filt["stride_knee_flexion_mean"],
            color=LTGRAY, s=40, zorder=2)
x_line = np.array([mean_filt["seq"].min(), mean_filt["seq"].max()])
ax1.plot(x_line, m_slope * x_line + m_int, color=ORANGE, linewidth=3, zorder=3)
ax1.set_title("Average Angle (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax1.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax1.set_ylabel("Stride Knee Flexion (°)", fontsize=13, color=GRAY)
ax1.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax1.spines[s].set_visible(False)
ax1.spines["left"].set_color(LTGRAY); ax1.spines["bottom"].set_color(LTGRAY)
ax1.text(0.05, 0.06, f"p = {m_p:.4f}", transform=ax1.transAxes,
          fontsize=13, color=ORANGE, family="Bahnschrift")

ax2.scatter(std_filt["seq"], std_filt["stride_knee_flexion_std"],
            color=LTGRAY, s=40, zorder=2)
x_line2 = np.array([std_filt["seq"].min(), std_filt["seq"].max()])
ax2.plot(x_line2, s_slope * x_line2 + s_int, color=ORANGE, linewidth=3, zorder=3)
ax2.set_title("Instability at Release (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax2.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax2.set_ylabel("Frame-to-Frame Std Dev (°)", fontsize=13, color=GRAY)
ax2.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax2.spines[s].set_visible(False)
ax2.spines["left"].set_color(LTGRAY); ax2.spines["bottom"].set_color(LTGRAY)
ax2.text(0.05, 0.06, f"p = {s_p:.5f}", transform=ax2.transAxes,
          fontsize=13, color=ORANGE, family="Bahnschrift")

plt.tight_layout(rect=[0, 0, 1, 0.87])
out_b = os.path.join(OUTPUT_DIR, "B_result_mean_and_variability.png")
plt.savefig(out_b, dpi=200, facecolor="white")
plt.close()
print(f"Saved: {out_b}")


# ══════════════════════════════════════════════════════════
#  CHART C — THE MECHANISM: stride knee (significant) vs
#  throwing-side knee (flat), same full dataset
# ══════════════════════════════════════════════════════════
stride_filt = iqr_filter_all(df_ok, "stride_knee_flexion_mean").dropna(subset=["stride_knee_flexion_mean"])
stride_filt = stride_filt.sort_values("game_pitch_number").reset_index(drop=True)
stride_filt["seq"] = np.arange(1, len(stride_filt) + 1)

throw_filt  = iqr_filter_all(df_ok, "throwing_knee_flexion_mean").dropna(subset=["throwing_knee_flexion_mean"])
throw_filt = throw_filt.sort_values("game_pitch_number").reset_index(drop=True)
throw_filt["seq"] = np.arange(1, len(throw_filt) + 1)

st_slope, st_int, st_r, st_p, _ = stats.linregress(stride_filt["seq"], stride_filt["stride_knee_flexion_mean"])
th_slope, th_int, th_r, th_p, _ = stats.linregress(throw_filt["seq"], throw_filt["throwing_knee_flexion_mean"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor="white")
fig.text(0.06, 0.96, "The Lead Leg Changed, the Throwing-Side Leg Did Not.",
          fontsize=22, color=GRAY, family="Bahnschrift")
fig.text(0.06, 0.905, "The lead leg braces the kinetic chain during the delivery",
          fontsize=13, color=LTGRAY, family="Bahnschrift")

ax1, ax2 = axes

ax1.scatter(throw_filt["seq"], throw_filt["throwing_knee_flexion_mean"],
            color=LTGRAY, s=40, zorder=2)
x_line = np.array([throw_filt["seq"].min(), throw_filt["seq"].max()])
ax1.plot(x_line, th_slope * x_line + th_int, color=LTGRAY, linewidth=3, zorder=3)
ax1.set_title("Throwing-Side Knee (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax1.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax1.set_ylabel("Knee Flexion (°)", fontsize=13, color=GRAY)
ax1.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax1.spines[s].set_visible(False)
ax1.spines["left"].set_color(LTGRAY); ax1.spines["bottom"].set_color(LTGRAY)
ax1.text(0.05, 0.06, f"p = {th_p:.2f}", transform=ax1.transAxes,
          fontsize=13, color=LTGRAY, family="Bahnschrift")

ax2.scatter(stride_filt["seq"], stride_filt["stride_knee_flexion_mean"],
            color=LTGRAY, s=40, zorder=2)
x_line2 = np.array([stride_filt["seq"].min(), stride_filt["seq"].max()])
ax2.plot(x_line2, st_slope * x_line2 + st_int, color=ORANGE, linewidth=3, zorder=3)
ax2.set_title("Stride (Lead) Knee (°)", fontsize=17, color=GRAY, loc="left", pad=14)
ax2.set_xlabel("Pitch number (in-game)", fontsize=13, color=GRAY)
ax2.set_ylabel("Knee Flexion (°)", fontsize=13, color=GRAY)
ax2.tick_params(labelsize=12, colors=GRAY)
for s in ["top","right"]: ax2.spines[s].set_visible(False)
ax2.spines["left"].set_color(LTGRAY); ax2.spines["bottom"].set_color(LTGRAY)
ax2.text(0.05, 0.06, f"p = {st_p:.4f}", transform=ax2.transAxes,
          fontsize=13, color=ORANGE, family="Bahnschrift")

plt.tight_layout(rect=[0, 0, 1, 0.87])
out_c = os.path.join(OUTPUT_DIR, "C_mechanism_stride_vs_throwing.png")
plt.savefig(out_c, dpi=200, facecolor="white")
plt.close()
print(f"Saved: {out_c}")

print("\nAll three charts built.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  FIND A CLEAN LATE-INNING CLIP (same scanner logic as before,
#  checking only the 12 drawn joints, not nose)
# ══════════════════════════════════════════════════════════

import numpy as np

DRAWN_JOINTS = {"l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist",
                 "l_hip", "r_hip", "l_knee", "r_knee", "l_ankle", "r_ankle"}

def check_clip_completeness(clip_path, start=165, end=195, conf_thresh=0.3):
    frames = read_frame_range(clip_path, start, end)
    if len(frames) < (end - start + 1) * 0.9:
        return False, "clip_too_short", 0.0, -1

    tracked = track_pitcher_through_clip(frames)

    worst_conf = 1.0
    worst_joint = None
    frames_with_drop = 0

    for box, kp in tracked:
        if kp is None:
            frames_with_drop += 1
            worst_conf = 0.0
            worst_joint = "PITCHER NOT DETECTED"
            continue
        frame_min = 1.0
        for name, idx in KP.items():
            if name not in DRAWN_JOINTS:
                continue
            c = kp[idx, 2]
            if c < frame_min:
                frame_min = c
            if c < worst_conf:
                worst_conf = c
                worst_joint = name
        if frame_min < conf_thresh:
            frames_with_drop += 1

    is_clean = (frames_with_drop == 0)
    return is_clean, worst_joint, worst_conf, frames_with_drop


# ══════════════════════════════════════════════════
#  Scan late-inning fastballs (inning 5-6) for a clean clip
# ══════════════════════════════════════════════════
MAX_CHECK = 20
CONF_THRESH = 0.3

late_candidates = df[(df["inning"] >= 5) &
                     (df["pitch_type_desc"] == "Four-Seam Fastball")].reset_index(drop=True)
print(f"Scanning {min(MAX_CHECK, len(late_candidates))} late-inning (5-6) fastballs "
      f"for a fully clean clip...\n")

clean_late_clip = None
for i, row in late_candidates.head(MAX_CHECK).iterrows():
    print(f"[{i+1}/{min(MAX_CHECK, len(late_candidates))}] checking inning {row['inning']}, "
          f"pitch {row['pitch_number']}...", end=" ")
    is_clean, worst_joint, worst_conf, n_drops = check_clip_completeness(row["clip_path"])

    if is_clean:
        print(f"CLEAN!")
        clean_late_clip = row
        break
    else:
        print(f"drop on {n_drops} frame(s), worst joint '{worst_joint}' (conf={worst_conf:.2f})")

if clean_late_clip is not None:
    print(f"\n>>> Found clean late-inning clip: inning {clean_late_clip['inning']}, "
          f"pitch {clean_late_clip['pitch_number']}")
    print(f">>> clip_path: {clean_late_clip['clip_path']}")
    print(f">>> play_id: {clean_late_clip['play_id']}")
else:
    print(f"\nNo fully clean late-inning clip found in the first {MAX_CHECK} checked.")
    print("Options: raise MAX_CHECK, or widen to include inning 4 too.")

In [ ]:
# ══════════════════════════════════════════════════════════
#  EARLY vs LATE INNING SIDE-BY-SIDE COMPARISON
#  Real video + skeleton overlay on both panels (no graphs).
#  Each panel independently cropped/letterboxed around its own
#  pitcher, same technique as the mitosis GIF, synced and looping.
#
#  Reuses: read_frame_range, track_pitcher_through_clip, KP
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
from PIL import Image
import os

OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ORANGE_BGR = (0, 105, 247)
NAVY_BGR   = (84, 14, 0)
WHITE_BGR  = (255, 255, 255)

SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]
DRAWN_JOINTS = {j for edge in SKELETON_EDGES for j in edge}


def draw_skeleton_on(canvas, kp, conf_thresh=0.3, line_thickness=3, pt_radius=6):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, ORANGE_BGR, line_thickness, cv2.LINE_AA)
        for name in DRAWN_JOINTS:
            idx = KP[name]
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, pt_radius, NAVY_BGR, -1, cv2.LINE_AA)
    return canvas


def compute_pitcher_crop_box(tracked, frame_w, frame_h, pad_frac=0.35):
    xs1, ys1, xs2, ys2 = [], [], [], []
    for box, kp in tracked:
        if box is None:
            continue
        x1, y1, x2, y2 = box
        xs1.append(x1); ys1.append(y1); xs2.append(x2); ys2.append(y2)
    x1, y1, x2, y2 = min(xs1), min(ys1), max(xs2), max(ys2)
    w, h = x2 - x1, y2 - y1
    pad_x, pad_y = w * pad_frac, h * pad_frac
    x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
    x2 = min(frame_w, x2 + pad_x); y2 = min(frame_h, y2 + pad_y)
    return int(x1), int(y1), int(x2), int(y2)


def resize_with_letterbox(img, target_w, target_h, bg_color=255):
    h, w = img.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h))
    canvas = np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas


def shift_keypoints(kp, offset_x, offset_y):
    if kp is None:
        return None
    shifted = kp.copy()
    shifted[:, 0] -= offset_x
    shifted[:, 1] -= offset_y
    return shifted


def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")


def build_side_by_side_comparison(clip_left, clip_right, label_left, label_right,
                                    start=165, end=195, out_name="early_vs_late.gif", fps=15):
    frames_l = read_frame_range(clip_left, start, end)
    frames_r = read_frame_range(clip_right, start, end)
    tracked_l = track_pitcher_through_clip(frames_l)
    tracked_r = track_pitcher_through_clip(frames_r)
    n = min(len(frames_l), len(frames_r))

    h, w = frames_l[0].shape[:2]
    half_w = w // 2

    lcx1, lcy1, lcx2, lcy2 = compute_pitcher_crop_box(tracked_l, w, h)
    rcx1, rcy1, rcx2, rcy2 = compute_pitcher_crop_box(tracked_r, w, h)

    output_frames = []
    for i in range(n):
        box_l, kp_l = tracked_l[i]
        box_r, kp_r = tracked_r[i]

        kp_l_shifted = shift_keypoints(kp_l, lcx1, lcy1)
        kp_r_shifted = shift_keypoints(kp_r, rcx1, rcy1)

        crop_l = frames_l[i][lcy1:lcy2, lcx1:lcx2]
        crop_r = frames_r[i][rcy1:rcy2, rcx1:rcx2]

        panel_l = draw_skeleton_on(crop_l, kp_l_shifted)
        panel_r = draw_skeleton_on(crop_r, kp_r_shifted)

        panel_l = resize_with_letterbox(panel_l, half_w, h)
        panel_r = resize_with_letterbox(panel_r, w - half_w, h)

        cv2.putText(panel_l, label_left, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                    WHITE_BGR, 4, cv2.LINE_AA)
        cv2.putText(panel_l, label_left, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                    NAVY_BGR, 2, cv2.LINE_AA)
        cv2.putText(panel_r, label_right, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                    WHITE_BGR, 4, cv2.LINE_AA)
        cv2.putText(panel_r, label_right, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.9,
                    ORANGE_BGR, 2, cv2.LINE_AA)

        divider = np.full((h, 3, 3), 220, dtype=np.uint8)
        combined = np.hstack([panel_l, divider, panel_r])
        combined = cv2.resize(combined, (w, h))
        output_frames.append(combined)

    frames_to_gif(output_frames, os.path.join(OUTPUT_DIR, out_name), fps=fps)


# ══════════════════════════════════════════════════
#  RUN — verified clean early (inning 1) and late (inning 5) clips
# ══════════════════════════════════════════════════
early_clip_path = r"C:\Users\noahl\OneDrive\fatigue_cv\clips\a6d0cf9f-271a-341d-9be0-85cbe90235a4.mp4"
late_clip_path  = r"C:\Users\noahl\OneDrive\fatigue_cv\clips\e511bc10-981d-3264-a2d7-130c558c27c2.mp4"

build_side_by_side_comparison(early_clip_path, late_clip_path,
                                label_left="INNING 1", label_right="INNING 5")

In [ ]:
# ══════════════════════════════════════════════════════════
#  FREEZE / FADE / ZOOM COMPARISON GIF
#  Two pitches (verified clean early-inning + late-inning
#  fastballs), side by side, synced:
#
#    Phase 1: real video + skeleton overlay plays normally
#    Phase 2: freezes at release, video fades out leaving just
#              the skeleton, camera zooms into the stride knee
#    Phase 3: holds on the zoomed knee, angle number displayed
#    Phase 4: zooms back out, video fades back in
#    Phase 5: finishes the rest of the delivery, then loops
#
#  FULLY SELF-CONTAINED — assumes a fresh kernel. Re-loads
#  everything from scratch: models, tracking functions, pitch
#  metadata CSV.
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import os
from ultralytics import YOLO

# ── Paths ──
RAW_CSV    = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"
OUTPUT_DIR = r"C:\Users\noahl\OneDrive\fatigue_cv\gifs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Palette (BGR for cv2) ──
ORANGE_BGR = (0, 105, 247)
NAVY_BGR   = (84, 14, 0)
WHITE_BGR  = (255, 255, 255)

# ── COCO-17 keypoint map ──
KP = {
    "nose": 0, "l_eye": 1, "r_eye": 2, "l_ear": 3, "r_ear": 4,
    "l_shoulder": 5, "r_shoulder": 6, "l_elbow": 7, "r_elbow": 8,
    "l_wrist": 9, "r_wrist": 10, "l_hip": 11, "r_hip": 12,
    "l_knee": 13, "r_knee": 14, "l_ankle": 15, "r_ankle": 16,
}
SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]
DRAWN_JOINTS = {j for edge in SKELETON_EDGES for j in edge}

MOUND_X_FRAC = 0.35
MOUND_Y_FRAC = 0.65
RELEASE_FRAME = 179   # validated release frame, absolute index in the raw clip

print("Loading YOLOv8m-pose model (may take a moment)...")
pose_model = YOLO("yolov8m-pose.pt")
print("Model loaded.")


# ══════════════════════════════════════════════════
#  Core pose / tracking (rebuilt from scratch)
# ══════════════════════════════════════════════════
def read_frame_range(video_path, start, end):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    out = []
    for _ in range(end - start + 1):
        ret, frame = cap.read()
        if not ret:
            break
        out.append(frame)
    cap.release()
    return out


def get_all_people(frame, conf=0.3):
    results = pose_model(frame, verbose=False, conf=conf)[0]
    people = []
    if results.keypoints is None or results.boxes is None:
        return people
    boxes = results.boxes.xyxy.cpu().numpy()
    kps = results.keypoints.data.cpu().numpy()
    for box, kp in zip(boxes, kps):
        people.append((box, kp))
    return people


def pick_pitcher(people, frame_w, frame_h, prev_box=None):
    if not people:
        return None, None
    target_x = frame_w * MOUND_X_FRAC
    target_y = frame_h * MOUND_Y_FRAC

    def box_center(b):
        return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)

    def iou(a, b):
        xA, yA = max(a[0], b[0]), max(a[1], b[1])
        xB, yB = min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        areaA = (a[2]-a[0]) * (a[3]-a[1])
        areaB = (b[2]-b[0]) * (b[3]-b[1])
        return inter / max(areaA + areaB - inter, 1e-8)

    if prev_box is not None:
        best, best_score = None, 0.0
        for box, kp in people:
            score = iou(prev_box, box)
            if score > best_score:
                best, best_score = (box, kp), score
        if best is not None and best_score > 0.15:
            return best

    best, best_dist = None, float("inf")
    for box, kp in people:
        cx, cy = box_center(box)
        dist = (cx - target_x) ** 2 + (cy - target_y) ** 2
        if dist < best_dist:
            best, best_dist = (box, kp), dist
    return best


def track_pitcher_through_clip(frames, conf=0.3):
    tracked = []
    prev_box = None
    h, w = frames[0].shape[:2]
    for frame in frames:
        people = get_all_people(frame, conf=conf)
        box, kp = pick_pitcher(people, w, h, prev_box=prev_box) if people else (None, None)
        tracked.append((box, kp))
        if box is not None:
            prev_box = box
    return tracked


def angle_between(a, b, c):
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def draw_skeleton(canvas, kp, conf_thresh=0.3, line_thickness=3, pt_radius=6):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, ORANGE_BGR, line_thickness, cv2.LINE_AA)
        for name in DRAWN_JOINTS:
            idx = KP[name]
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, pt_radius, NAVY_BGR, -1, cv2.LINE_AA)
    return canvas


def compute_pitcher_crop_box(tracked, frame_w, frame_h, pad_frac=0.35):
    xs1, ys1, xs2, ys2 = [], [], [], []
    for box, kp in tracked:
        if box is None:
            continue
        x1, y1, x2, y2 = box
        xs1.append(x1); ys1.append(y1); xs2.append(x2); ys2.append(y2)
    x1, y1, x2, y2 = min(xs1), min(ys1), max(xs2), max(ys2)
    w, h = x2 - x1, y2 - y1
    pad_x, pad_y = w * pad_frac, h * pad_frac
    x1 = max(0, x1 - pad_x); y1 = max(0, y1 - pad_y)
    x2 = min(frame_w, x2 + pad_x); y2 = min(frame_h, y2 + pad_y)
    return int(x1), int(y1), int(x2), int(y2)


def resize_with_letterbox(img, target_w, target_h, bg_color=255):
    h, w = img.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(img, (new_w, new_h))
    canvas = np.full((target_h, target_w, 3), bg_color, dtype=np.uint8)
    x_off = (target_w - new_w) // 2
    y_off = (target_h - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas


def shift_keypoints(kp, offset_x, offset_y):
    if kp is None:
        return None
    shifted = kp.copy()
    shifted[:, 0] -= offset_x
    shifted[:, 1] -= offset_y
    return shifted


def frames_to_gif(frame_list_bgr, out_path, fps=15, loop=0):
    pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frame_list_bgr]
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=loop, optimize=False
    )
    print(f"Saved: {out_path}  ({len(frame_list_bgr)} frames @ {fps}fps)")


# ══════════════════════════════════════════════════
#  Scanner: find a fully clean clip (every drawn joint
#  above threshold, every frame, 165-195)
# ══════════════════════════════════════════════════
def check_clip_completeness(clip_path, start=165, end=195, conf_thresh=0.3):
    frames = read_frame_range(clip_path, start, end)
    if len(frames) < (end - start + 1) * 0.9:
        return False, "clip_too_short", 0.0, -1
    tracked = track_pitcher_through_clip(frames)
    worst_conf, worst_joint, frames_with_drop = 1.0, None, 0
    for box, kp in tracked:
        if kp is None:
            frames_with_drop += 1
            continue
        frame_min = 1.0
        for name in DRAWN_JOINTS:
            idx = KP[name]
            c = kp[idx, 2]
            if c < frame_min:
                frame_min = c
            if c < worst_conf:
                worst_conf, worst_joint = c, name
        if frame_min < conf_thresh:
            frames_with_drop += 1
    return (frames_with_drop == 0), worst_joint, worst_conf, frames_with_drop


def find_clean_clip(candidates_df, max_check=20, conf_thresh=0.3):
    for i, row in candidates_df.head(max_check).iterrows():
        is_clean, worst_joint, worst_conf, n_drops = check_clip_completeness(
            row["clip_path"], conf_thresh=conf_thresh)
        print(f"  checking inning {row['inning']}, pitch {row['pitch_number']}...", end=" ")
        if is_clean:
            print("CLEAN!")
            return row
        else:
            print(f"drop on {n_drops} frame(s), worst '{worst_joint}' ({worst_conf:.2f})")
    return None


# ══════════════════════════════════════════════════
#  Build the actual freeze/fade/zoom comparison
# ══════════════════════════════════════════════════
def build_freeze_zoom_comparison(clip_left, clip_right, label_left, label_right,
                                   start=165, end=195, release_frame=RELEASE_FRAME,
                                   out_name="freeze_zoom_comparison.gif", fps=15,
                                   transition_frames=10, hold_frames=15,
                                   zoom_pad_px=90):
    frames_l = read_frame_range(clip_left, start, end)
    frames_r = read_frame_range(clip_right, start, end)
    tracked_l = track_pitcher_through_clip(frames_l)
    tracked_r = track_pitcher_through_clip(frames_r)
    n = min(len(frames_l), len(frames_r))
    release_idx = release_frame - start

    h, w = frames_l[0].shape[:2]
    half_w = w // 2

    body_crop_l = compute_pitcher_crop_box(tracked_l, w, h)
    body_crop_r = compute_pitcher_crop_box(tracked_r, w, h)

    def knee_angle_and_point(kp, side="l"):
        hip = kp[KP[f"{side}_hip"], :2]
        knee = kp[KP[f"{side}_knee"], :2]
        ankle = kp[KP[f"{side}_ankle"], :2]
        return angle_between(hip, knee, ankle), knee

    # Knee position + angle at release, for each side (stride = left side, RHP)
    angle_l, knee_pt_l = knee_angle_and_point(tracked_l[release_idx][1], "l")
    angle_r, knee_pt_r = knee_angle_and_point(tracked_r[release_idx][1], "l")

    def knee_zoom_box(knee_pt, frame_w, frame_h, pad=zoom_pad_px):
        x1 = max(0, int(knee_pt[0] - pad))
        y1 = max(0, int(knee_pt[1] - pad))
        x2 = min(frame_w, int(knee_pt[0] + pad))
        y2 = min(frame_h, int(knee_pt[1] + pad))
        return x1, y1, x2, y2

    zoom_crop_l = knee_zoom_box(knee_pt_l, w, h)
    zoom_crop_r = knee_zoom_box(knee_pt_r, w, h)

    def render_panel(frame_raw, kp, box_crop, video_alpha, panel_w, panel_h,
                      angle_label=None):
        cx1, cy1, cx2, cy2 = box_crop
        kp_shifted = shift_keypoints(kp, cx1, cy1)
        vid_crop = frame_raw[cy1:cy2, cx1:cx2]
        skel_only = np.full_like(vid_crop, 255)
        skel_only = draw_skeleton(skel_only, kp_shifted)
        vid_with_skel = draw_skeleton(vid_crop.copy(), kp_shifted)

        blended = cv2.addWeighted(vid_with_skel, video_alpha, skel_only, 1 - video_alpha, 0)
        panel = resize_with_letterbox(blended, panel_w, panel_h)

        if angle_label is not None:
            text = f"{angle_label:.0f} deg"
            cv2.putText(panel, text, (20, panel_h - 25), cv2.FONT_HERSHEY_DUPLEX,
                        1.1, (0, 0, 0), 5, cv2.LINE_AA)
            cv2.putText(panel, text, (20, panel_h - 25), cv2.FONT_HERSHEY_DUPLEX,
                        1.1, ORANGE_BGR, 2, cv2.LINE_AA)
        return panel

    def compose(frame_idx_l, frame_idx_r, crop_l, crop_r, alpha, label_angle=False):
        kp_l = tracked_l[frame_idx_l][1]
        kp_r = tracked_r[frame_idx_r][1]
        panel_l = render_panel(frames_l[frame_idx_l], kp_l, crop_l, alpha, half_w, h,
                                angle_label=angle_l if label_angle else None)
        panel_r = render_panel(frames_r[frame_idx_r], kp_r, crop_r, alpha, w - half_w, h,
                                angle_label=angle_r if label_angle else None)

        cv2.putText(panel_l, label_left, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.85,
                    WHITE_BGR, 4, cv2.LINE_AA)
        cv2.putText(panel_l, label_left, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.85,
                    NAVY_BGR, 2, cv2.LINE_AA)
        cv2.putText(panel_r, label_right, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.85,
                    WHITE_BGR, 4, cv2.LINE_AA)
        cv2.putText(panel_r, label_right, (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.85,
                    ORANGE_BGR, 2, cv2.LINE_AA)

        divider = np.full((h, 3, 3), 220, dtype=np.uint8)
        combo = np.hstack([panel_l, divider, panel_r])
        return cv2.resize(combo, (w, h))

    output_frames = []

    # PHASE 1: normal playback up through release
    for i in range(0, release_idx + 1):
        output_frames.append(compose(i, i, body_crop_l, body_crop_r, alpha=1.0))

    # PHASE 2: freeze at release, fade video->skeleton, zoom body->knee
    for step in range(transition_frames):
        t = step / (transition_frames - 1)
        ease_t = t * t * (3 - 2 * t)
        alpha = 1.0 - ease_t  # video fades out

        cl = tuple(int(body_crop_l[k] * (1 - ease_t) + zoom_crop_l[k] * ease_t) for k in range(4))
        cr = tuple(int(body_crop_r[k] * (1 - ease_t) + zoom_crop_r[k] * ease_t) for k in range(4))

        output_frames.append(compose(release_idx, release_idx, cl, cr, alpha=alpha,
                                       label_angle=(ease_t > 0.6)))

    # PHASE 3: hold on zoomed knee with angle label
    for _ in range(hold_frames):
        output_frames.append(compose(release_idx, release_idx, zoom_crop_l, zoom_crop_r,
                                       alpha=0.0, label_angle=True))

    # PHASE 4: zoom back out, fade video back in (reverse of phase 2)
    for step in range(transition_frames):
        t = step / (transition_frames - 1)
        ease_t = t * t * (3 - 2 * t)
        alpha = ease_t  # video fades back in

        cl = tuple(int(zoom_crop_l[k] * (1 - ease_t) + body_crop_l[k] * ease_t) for k in range(4))
        cr = tuple(int(zoom_crop_r[k] * (1 - ease_t) + body_crop_r[k] * ease_t) for k in range(4))

        output_frames.append(compose(release_idx, release_idx, cl, cr, alpha=alpha,
                                       label_angle=(ease_t < 0.4)))

    # PHASE 5: finish the rest of the delivery
    for i in range(release_idx + 1, n):
        output_frames.append(compose(i, i, body_crop_l, body_crop_r, alpha=1.0))

    frames_to_gif(output_frames, os.path.join(OUTPUT_DIR, out_name), fps=fps)
    print(f"Stride knee angle at release — {label_left}: {angle_l:.1f} deg, "
          f"{label_right}: {angle_r:.1f} deg")


# ══════════════════════════════════════════════════
#  RUN — reload metadata, find clean early/late fastballs, build
# ══════════════════════════════════════════════════
df = pd.read_csv(RAW_CSV)

print("Scanning for a clean EARLY-inning (inning 1) fastball...")
early_candidates = df[(df["inning"] == 1) & (df["pitch_type_desc"] == "Four-Seam Fastball")].reset_index(drop=True)
early_pitch = find_clean_clip(early_candidates)

print("\nScanning for a clean LATE-inning (5-6) fastball...")
late_candidates = df[(df["inning"] >= 5) & (df["pitch_type_desc"] == "Four-Seam Fastball")].reset_index(drop=True)
late_pitch = find_clean_clip(late_candidates)

if early_pitch is None or late_pitch is None:
    print("\nCould not find clean clips automatically — widen MAX_CHECK or supply play_ids manually.")
else:
    print(f"\nUsing early: inning {early_pitch['inning']} pitch {early_pitch['pitch_number']}")
    print(f"Using late:  inning {late_pitch['inning']} pitch {late_pitch['pitch_number']}")

    build_freeze_zoom_comparison(
        early_pitch["clip_path"], late_pitch["clip_path"],
        label_left="INNING 1", label_right=f"INNING {late_pitch['inning']}",
    )

In [ ]:
# ══════════════════════════════════════════════════════════
#  SCAN ALL EARLY (innings 1-2) AND LATE (innings 5-6) PITCHES
#  For each: check full skeleton coverage (165-195 window),
#  and if clean, compute the stride (left) knee angle at release.
#  Builds a table so you can pick the best-contrasting pair
#  yourself, instead of relying on the first clean one found.
#
#  FULLY SELF-CONTAINED — safe even after a kernel restart.
#  Checkpoints to CSV as it goes, so an interruption doesn't
#  lose progress — reruns skip already-processed pitches.
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
import pandas as pd
import os
from ultralytics import YOLO

RAW_CSV        = r"C:\Users\noahl\OneDrive\fatigue_cv\misiorowski_822744_pitches_with_clips.csv"
CHECKPOINT_CSV = r"C:\Users\noahl\OneDrive\fatigue_cv\pitch_coverage_and_knee_angle.csv"

KP = {
    "nose": 0, "l_eye": 1, "r_eye": 2, "l_ear": 3, "r_ear": 4,
    "l_shoulder": 5, "r_shoulder": 6, "l_elbow": 7, "r_elbow": 8,
    "l_wrist": 9, "r_wrist": 10, "l_hip": 11, "r_hip": 12,
    "l_knee": 13, "r_knee": 14, "l_ankle": 15, "r_ankle": 16,
}
SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]
DRAWN_JOINTS = {j for edge in SKELETON_EDGES for j in edge}

MOUND_X_FRAC = 0.35
MOUND_Y_FRAC = 0.65
WINDOW_START, WINDOW_END = 165, 195
RELEASE_FRAME = 179

print("Loading YOLOv8m-pose model (may take a moment)...")
pose_model = YOLO("yolov8m-pose.pt")
print("Model loaded.\n")


def read_frame_range(video_path, start, end):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    out = []
    for _ in range(end - start + 1):
        ret, frame = cap.read()
        if not ret:
            break
        out.append(frame)
    cap.release()
    return out


def get_all_people(frame, conf=0.3):
    results = pose_model(frame, verbose=False, conf=conf)[0]
    people = []
    if results.keypoints is None or results.boxes is None:
        return people
    boxes = results.boxes.xyxy.cpu().numpy()
    kps = results.keypoints.data.cpu().numpy()
    for box, kp in zip(boxes, kps):
        people.append((box, kp))
    return people


def pick_pitcher(people, frame_w, frame_h, prev_box=None):
    if not people:
        return None, None
    target_x = frame_w * MOUND_X_FRAC
    target_y = frame_h * MOUND_Y_FRAC

    def box_center(b):
        return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)

    def iou(a, b):
        xA, yA = max(a[0], b[0]), max(a[1], b[1])
        xB, yB = min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        areaA = (a[2]-a[0]) * (a[3]-a[1])
        areaB = (b[2]-b[0]) * (b[3]-b[1])
        return inter / max(areaA + areaB - inter, 1e-8)

    if prev_box is not None:
        best, best_score = None, 0.0
        for box, kp in people:
            score = iou(prev_box, box)
            if score > best_score:
                best, best_score = (box, kp), score
        if best is not None and best_score > 0.15:
            return best

    best, best_dist = None, float("inf")
    for box, kp in people:
        cx, cy = box_center(box)
        dist = (cx - target_x) ** 2 + (cy - target_y) ** 2
        if dist < best_dist:
            best, best_dist = (box, kp), dist
    return best


def track_pitcher_through_clip(frames, conf=0.3):
    tracked = []
    prev_box = None
    h, w = frames[0].shape[:2]
    for frame in frames:
        people = get_all_people(frame, conf=conf)
        box, kp = pick_pitcher(people, w, h, prev_box=prev_box) if people else (None, None)
        tracked.append((box, kp))
        if box is not None:
            prev_box = box
    return tracked


def angle_between(a, b, c):
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def evaluate_pitch(clip_path, conf_thresh=0.3):
    """
    Returns dict with: is_clean, worst_joint, worst_conf, n_drops,
    stride_knee_angle (at release, only if clean).
    """
    frames = read_frame_range(clip_path, WINDOW_START, WINDOW_END)
    if len(frames) < (WINDOW_END - WINDOW_START + 1) * 0.9:
        return {"is_clean": False, "worst_joint": "clip_too_short",
                "worst_conf": 0.0, "n_drops": -1, "stride_knee_angle": np.nan}

    tracked = track_pitcher_through_clip(frames)

    worst_conf, worst_joint, n_drops = 1.0, None, 0
    for box, kp in tracked:
        if kp is None:
            n_drops += 1
            continue
        frame_min = 1.0
        for name in DRAWN_JOINTS:
            idx = KP[name]
            c = kp[idx, 2]
            if c < frame_min:
                frame_min = c
            if c < worst_conf:
                worst_conf, worst_joint = c, name
        if frame_min < conf_thresh:
            n_drops += 1

    is_clean = (n_drops == 0)

    knee_angle = np.nan
    if is_clean:
        release_idx = RELEASE_FRAME - WINDOW_START
        box, kp = tracked[release_idx]
        if kp is not None:
            hip = kp[KP["l_hip"], :2]
            knee = kp[KP["l_knee"], :2]
            ankle = kp[KP["l_ankle"], :2]
            knee_angle = angle_between(hip, knee, ankle)

    return {"is_clean": is_clean, "worst_joint": worst_joint,
            "worst_conf": worst_conf, "n_drops": n_drops,
            "stride_knee_angle": knee_angle}


# ══════════════════════════════════════════════════
#  RUN — all pitches from innings 1-2 and innings 5-6
# ══════════════════════════════════════════════════
df = pd.read_csv(RAW_CSV)

candidates = df[df["inning"].isin([1, 2, 5, 6])].copy()
candidates["group"] = candidates["inning"].apply(lambda i: "early" if i in [1, 2] else "late")
candidates = candidates.sort_values(["group", "inning", "pitch_number"]).reset_index(drop=True)

print(f"Total candidate pitches to evaluate: {len(candidates)} "
      f"({(candidates['group']=='early').sum()} early, {(candidates['group']=='late').sum()} late)\n")

# Resume support
if os.path.exists(CHECKPOINT_CSV):
    existing = pd.read_csv(CHECKPOINT_CSV)
    done_play_ids = set(existing["play_id"])
    print(f"Resuming — {len(existing)} pitches already evaluated.\n")
else:
    existing = pd.DataFrame()
    done_play_ids = set()

results = existing.to_dict("records")
remaining = candidates[~candidates["play_id"].isin(done_play_ids)].reset_index(drop=True)

for i, row in remaining.iterrows():
    print(f"[{i+1}/{len(remaining)}] {row['group']} — inning {row['inning']}, "
          f"pitch {row['pitch_number']}, {row['pitch_type_desc']}...", end=" ")

    eval_result = evaluate_pitch(row["clip_path"])

    result_row = {
        "play_id": row["play_id"], "group": row["group"], "inning": row["inning"],
        "pitch_number": row["pitch_number"], "pitch_type": row["pitch_type_desc"],
        "clip_path": row["clip_path"],
        **eval_result,
    }
    results.append(result_row)

    if eval_result["is_clean"]:
        print(f"CLEAN — stride knee at release: {eval_result['stride_knee_angle']:.1f} deg")
    else:
        print(f"drop on {eval_result['n_drops']} frame(s), "
              f"worst '{eval_result['worst_joint']}' ({eval_result['worst_conf']:.2f})")

    # Checkpoint every 5 pitches
    if (i + 1) % 5 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False)

final_df = pd.DataFrame(results)
final_df.to_csv(CHECKPOINT_CSV, index=False)

print(f"\n{'='*60}")
print(f"Done. {len(final_df)} pitches evaluated, {final_df['is_clean'].sum()} clean.")
print(f"Saved to: {CHECKPOINT_CSV}")
print(f"{'='*60}\n")

clean_only = final_df[final_df["is_clean"]].copy()
print("CLEAN PITCHES — sorted by stride knee angle at release:\n")
print(clean_only[["group", "inning", "pitch_number", "pitch_type", "stride_knee_angle"]]
      .sort_values("stride_knee_angle").to_string(index=False))

print("\n--- Summary by group ---")
print(clean_only.groupby("group")["stride_knee_angle"].agg(["count", "mean", "min", "max"]))

In [ ]:
# ══════════════════════════════════════════════════════════
#  SCRUBBABLE DUAL-PANEL VIDEO — best-contrast pair
#  Real broadcast video + skeleton overlay, side by side,
#  synced, with live stride knee angle labeled on each panel.
#  Uses matplotlib FuncAnimation + jshtml for real frame-by-frame
#  scrubbing controls (not a GIF file).
#
#  FULLY SELF-CONTAINED — safe after a kernel restart.
#  Picks the best-contrast clean pair directly from the
#  checkpoint CSV (early = max knee angle, late = min knee angle).
# ══════════════════════════════════════════════════════════

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rcParams
from IPython.display import HTML
from ultralytics import YOLO

CHECKPOINT_CSV = r"C:\Users\noahl\OneDrive\fatigue_cv\pitch_coverage_and_knee_angle.csv"

rcParams['animation.embed_limit'] = 100  # MB, avoid the size-limit error from before

KP = {
    "nose": 0, "l_eye": 1, "r_eye": 2, "l_ear": 3, "r_ear": 4,
    "l_shoulder": 5, "r_shoulder": 6, "l_elbow": 7, "r_elbow": 8,
    "l_wrist": 9, "r_wrist": 10, "l_hip": 11, "r_hip": 12,
    "l_knee": 13, "r_knee": 14, "l_ankle": 15, "r_ankle": 16,
}
SKELETON_EDGES = [
    ("l_shoulder", "r_shoulder"),
    ("l_shoulder", "l_elbow"), ("l_elbow", "l_wrist"),
    ("r_shoulder", "r_elbow"), ("r_elbow", "r_wrist"),
    ("l_shoulder", "l_hip"), ("r_shoulder", "r_hip"),
    ("l_hip", "r_hip"),
    ("l_hip", "l_knee"), ("l_knee", "l_ankle"),
    ("r_hip", "r_knee"), ("r_knee", "r_ankle"),
]
DRAWN_JOINTS = {j for edge in SKELETON_EDGES for j in edge}

MOUND_X_FRAC = 0.35
MOUND_Y_FRAC = 0.65
WINDOW_START, WINDOW_END = 165, 195
RELEASE_FRAME = 179

print("Loading YOLOv8m-pose model (may take a moment)...")
pose_model = YOLO("yolov8m-pose.pt")
print("Model loaded.\n")


def read_frame_range(video_path, start, end):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    out = []
    for _ in range(end - start + 1):
        ret, frame = cap.read()
        if not ret:
            break
        out.append(frame)
    cap.release()
    return out


def get_all_people(frame, conf=0.3):
    results = pose_model(frame, verbose=False, conf=conf)[0]
    people = []
    if results.keypoints is None or results.boxes is None:
        return people
    boxes = results.boxes.xyxy.cpu().numpy()
    kps = results.keypoints.data.cpu().numpy()
    for box, kp in zip(boxes, kps):
        people.append((box, kp))
    return people


def pick_pitcher(people, frame_w, frame_h, prev_box=None):
    if not people:
        return None, None
    target_x = frame_w * MOUND_X_FRAC
    target_y = frame_h * MOUND_Y_FRAC

    def box_center(b):
        return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)

    def iou(a, b):
        xA, yA = max(a[0], b[0]), max(a[1], b[1])
        xB, yB = min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        areaA = (a[2]-a[0]) * (a[3]-a[1])
        areaB = (b[2]-b[0]) * (b[3]-b[1])
        return inter / max(areaA + areaB - inter, 1e-8)

    if prev_box is not None:
        best, best_score = None, 0.0
        for box, kp in people:
            score = iou(prev_box, box)
            if score > best_score:
                best, best_score = (box, kp), score
        if best is not None and best_score > 0.15:
            return best

    best, best_dist = None, float("inf")
    for box, kp in people:
        cx, cy = box_center(box)
        dist = (cx - target_x) ** 2 + (cy - target_y) ** 2
        if dist < best_dist:
            best, best_dist = (box, kp), dist
    return best


def track_pitcher_through_clip(frames, conf=0.3):
    tracked = []
    prev_box = None
    h, w = frames[0].shape[:2]
    for frame in frames:
        people = get_all_people(frame, conf=conf)
        box, kp = pick_pitcher(people, w, h, prev_box=prev_box) if people else (None, None)
        tracked.append((box, kp))
        if box is not None:
            prev_box = box
    return tracked


def angle_between(a, b, c):
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))


def draw_skeleton_cv(canvas, kp, conf_thresh=0.3, line_thickness=3, pt_radius=6):
    canvas = canvas.copy()
    if kp is not None:
        for a, b in SKELETON_EDGES:
            ia, ib = KP[a], KP[b]
            if kp[ia, 2] > conf_thresh and kp[ib, 2] > conf_thresh:
                pa = tuple(int(v) for v in kp[ia, :2])
                pb = tuple(int(v) for v in kp[ib, :2])
                cv2.line(canvas, pa, pb, (0, 105, 247), line_thickness, cv2.LINE_AA)
        for name in DRAWN_JOINTS:
            idx = KP[name]
            if kp[idx, 2] > conf_thresh:
                pt = tuple(int(v) for v in kp[idx, :2])
                cv2.circle(canvas, pt, pt_radius, (84, 14, 0), -1, cv2.LINE_AA)
    return canvas


def stride_knee_angle(kp, conf_thresh=0.3):
    if kp is None:
        return None
    if kp[KP["l_hip"], 2] < conf_thresh or kp[KP["l_knee"], 2] < conf_thresh or kp[KP["l_ankle"], 2] < conf_thresh:
        return None
    hip = kp[KP["l_hip"], :2]
    knee = kp[KP["l_knee"], :2]
    ankle = kp[KP["l_ankle"], :2]
    return angle_between(hip, knee, ankle)


# ══════════════════════════════════════════════════
#  Pick the best-contrast pair from the checkpoint CSV
# ══════════════════════════════════════════════════
scan_df = pd.read_csv(CHECKPOINT_CSV)
clean_df = scan_df[scan_df["is_clean"] == True].copy()

early_row = clean_df[clean_df["group"] == "early"].sort_values("stride_knee_angle", ascending=False).iloc[0]
late_row  = clean_df[clean_df["group"] == "late"].sort_values("stride_knee_angle", ascending=True).iloc[0]

print(f"EARLY pick: inning {early_row['inning']}, pitch {early_row['pitch_number']}, "
      f"{early_row['pitch_type']} — {early_row['stride_knee_angle']:.1f} deg")
print(f"LATE pick:  inning {late_row['inning']}, pitch {late_row['pitch_number']}, "
      f"{late_row['pitch_type']} — {late_row['stride_knee_angle']:.1f} deg")
print(f"Contrast: {early_row['stride_knee_angle'] - late_row['stride_knee_angle']:.1f} degrees\n")

label_left  = f"INNING {early_row['inning']} ({early_row['pitch_type']})"
label_right = f"INNING {late_row['inning']} ({late_row['pitch_type']})"

# ══════════════════════════════════════════════════
#  SYNC FIX: the early clip runs 2 frames behind the late
#  clip's delivery timing. Skip ahead 2 frames at the start
#  (old absolute frame 167 becomes the new start — "old frame
#  3" becomes "new frame 1"), and extend the end by 2 frames
#  to keep the same total window length. Late clip's window
#  is untouched.
# ══════════════════════════════════════════════════
SYNC_OFFSET = 2
EARLY_WINDOW_START = WINDOW_START + SYNC_OFFSET   # 167
EARLY_WINDOW_END   = WINDOW_END + SYNC_OFFSET      # 197
LATE_WINDOW_START  = WINDOW_START                  # 165 (unchanged)
LATE_WINDOW_END    = WINDOW_END                    # 195 (unchanged)

print(f"Early clip window shifted: {WINDOW_START}-{WINDOW_END} -> "
      f"{EARLY_WINDOW_START}-{EARLY_WINDOW_END}")
print(f"Late clip window unchanged: {LATE_WINDOW_START}-{LATE_WINDOW_END}\n")


# ══════════════════════════════════════════════════
#  Track both clips
# ══════════════════════════════════════════════════
print("Tracking early clip (shifted window)...")
frames_l = read_frame_range(early_row["clip_path"], EARLY_WINDOW_START, EARLY_WINDOW_END)
tracked_l = track_pitcher_through_clip(frames_l)

print("Tracking late clip...")
frames_r = read_frame_range(late_row["clip_path"], LATE_WINDOW_START, LATE_WINDOW_END)
tracked_r = track_pitcher_through_clip(frames_r)

n = min(len(frames_l), len(frames_r))
frame_indices = list(range(n))
# Both windows are the same length, so the release lands at the same
# relative index for both (e.g. 179-165=14 for late, 181-167=14 for early)
release_idx = RELEASE_FRAME - WINDOW_START


# ══════════════════════════════════════════════════
#  Build the FuncAnimation scrubber
# ══════════════════════════════════════════════════
sample_l = cv2.cvtColor(frames_l[0], cv2.COLOR_BGR2RGB)
sample_r = cv2.cvtColor(frames_r[0], cv2.COLOR_BGR2RGB)
h, w = sample_l.shape[:2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
img_l = ax1.imshow(sample_l)
img_r = ax2.imshow(sample_r)
ax1.axis("off")
ax2.axis("off")
ax1.set_title(label_left, fontsize=13)
ax2.set_title(label_right, fontsize=13)

text_l = ax1.text(10, 40, "", color="white", fontsize=16, weight="bold",
                   bbox=dict(facecolor="black", alpha=0.6, pad=4))
text_r = ax2.text(10, 40, "", color="white", fontsize=16, weight="bold",
                   bbox=dict(facecolor="black", alpha=0.6, pad=4))

def update(i):
    fi = frame_indices[i]
    box_l, kp_l = tracked_l[fi]
    box_r, kp_r = tracked_r[fi]

    annotated_l = draw_skeleton_cv(frames_l[fi], kp_l)
    annotated_r = draw_skeleton_cv(frames_r[fi], kp_r)

    img_l.set_data(cv2.cvtColor(annotated_l, cv2.COLOR_BGR2RGB))
    img_r.set_data(cv2.cvtColor(annotated_r, cv2.COLOR_BGR2RGB))

    angle_l = stride_knee_angle(kp_l)
    angle_r = stride_knee_angle(kp_r)
    label_l = f"frame {fi}"
    label_r = f"frame {fi}"
    if angle_l is not None:
        label_l += f"  |  knee: {angle_l:.0f} deg"
    if angle_r is not None:
        label_r += f"  |  knee: {angle_r:.0f} deg"
    if fi == release_idx:
        label_l += "  <-- RELEASE"
        label_r += "  <-- RELEASE"
    text_l.set_text(label_l)
    text_r.set_text(label_r)

    return [img_l, img_r, text_l, text_r]

plt.tight_layout()
anim = animation.FuncAnimation(fig, update, frames=n, interval=150, blit=True)
plt.close()

print("Rendering interactive scrubber (may take 20-40s)...")
HTML(anim.to_jshtml())